# CPMK 3 · Unified instance segmentation test evaluation

Enam checkpoint dievaluasi pada dua test set dalam 12 kombinasi primer. Original test Panoptic Indoor v5 memakai 359 foto. Personal test memakai 40 foto P001–P040 dan ground truth yang sudah dibekukan. Kedua test set digunakan untuk penilaian akhir.

### Protokol

Mask AP@[.50:.95], AP50, AP75, dan AR100 memakai COCOeval. Bounding-box AP merupakan metrik sekunder. Confidence minimum 0,001 dan batas 100 deteksi per foto mengikuti unified validation. YOLO memakai input 640, NMS 0,70, rect, dan retina masks. Mask R-CNN memakai panjang sisi 640–960, NMS 0,50, serta threshold 0,5 untuk mask probabilistik. Inferensi memakai FP32 dengan mask pada resolusi foto. Backbone Mask R-CNN memakai FrozenBatchNorm2d seperti pada training.

Perbandingan spesialisasi memproyeksikan prediksi All-Class yang sudah tersimpan ke tiga kelas Hard-3, lalu menilainya pada anotasi yang sama. Batas deteksi global berlaku sebelum proyeksi sehingga 74 kelas pada model All-Class dapat bersaing untuk ruang keluaran. Scope lintas domain memakai kategori dengan GT non-crowd pada kedua set. AP yang tidak terdefinisi tercatat sebagai null.

### Menjalankan di Kaggle

Aktifkan Tesla T4. Pasang Panoptic Indoor v5, input personal sebagai dataset privat, dan bobot dari input yang sudah ada. Discovery menerima ZIP, direktori hasil ekstraksi, dan folder bersarang.

1. Pada cell **02 Configuration**, gunakan PREFLIGHT dan jalankan semua cell. Pastikan muncul PREFLIGHT PASS.
2. Ubah MODE menjadi SMOKE_TEST. Jalankan semua cell dan pastikan muncul SMOKE PASS (NON-FINAL). Subset dipilih berdasarkan urutan ID serta keberadaan GT. Metrik smoke bersifat non-final.
3. Ubah MODE menjadi FULL_EVALUATION dan gunakan Save & Run All. Notebook mengulang preflight dan menjalankan atau memulihkan gate smoke. Unduh ZIP Complete jika 12 kombinasi berhasil.

Pasang ZIP atau folder hasil versi sebelumnya sebagai input untuk pemulihan. Isi RESUME_INPUTS dengan path mount aktual. Gunakan SELECT_PAIRS untuk membatasi kombinasi yang perlu dijalankan ulang. Reuse hanya berlaku ketika fingerprint dan checksum cocok. Simpan versi Kaggle atau unduh ZIP Partial agar hasil yang berhasil tersedia pada sesi berikutnya.


### 00 Install pinned evaluation packages

In [1]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

INSTALL_PINNED_DEPS = True
PINNED = {"ultralytics": "8.4.216", "pycocotools": "2.0.10"}
missing = []
for name, target in PINNED.items():
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = None
    if installed != target:
        missing.append(f"{name}=={target}")
if missing:
    if not INSTALL_PINNED_DEPS:
        raise RuntimeError(f"Dependency perlu dipasang: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.9 MB/s  0:00:00
  Attempting uninstall: pycocotools
    Found existing installation: pycocotools 2.0.11
    Uninstalling pycocotools-2.0.11:
      Successfully uninstalled pycocotools-2.0.11



### 01 Environment

In [2]:
import gc
import gzip
import hashlib
import io
import json
import math
import os
import platform
import re
import shutil
import time
import traceback
import uuid
import zipfile
from collections import Counter
from contextlib import redirect_stdout
from copy import deepcopy
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path, PurePosixPath

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import ultralytics
from PIL import Image
from pycocotools import mask as coco_mask
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.ops import FrozenBatchNorm2d
from torchvision.transforms.functional import to_tensor
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


### 02 Configuration

In [3]:
MODE = "FULL_EVALUATION"     # PREFLIGHT / STRESS_TEST / FULL_EVALUATION
APPROVE_FULL_RECOVERY = True
INPUT_ROOTS = ["/kaggle/input"]
WORK_ROOT = "/kaggle/working/CPMK3_unified_test_recovery"
CHECKPOINT_OVERRIDES = {}
RESUME_INPUTS = []       # Isi dengan path ZIP Partial Run01 atau folder mount; boleh ditambah output recovery lama.
SELECT_PAIRS = []        # Set secara eksplisit pada mode FULL_EVALUATION.
RECOVERY_PIPELINE_SHA256 = "339f60be0639cee7942fd8d9d715ea6a4a0434b70b9991a91550a9529658c75d"
RUN01_ID = "20261126T092418Z-43649903"
RUN01_PIPELINE_SHA256 = "34c9460d0815aa9a68306dc0888cf81d5a0927a7221da62291528e781c1b45eb"
RUN01_MANIFEST_SHA256 = "b0c178fc8b23f92bce64cf65db3c79d61228797467970b3fe3710b3ffcb731dd"
RUN01_INVENTORY_SHA256 = "2220f3083fe5ad492c99731e34186942eafe413fe215a2b9808b061923aaad95"
RUN01_ALLOWED = {
    "maskrcnn_all__original": "3aab98b96a59d06d1881bef1e7ac689f0d8d5f896ae9eced5cd2ff2716577d81",
    "maskrcnn_hard3__original": "7a388f159dd5ebd21968e6ce28983b27aeea6e90118b53b2d02e74b0b724e65f",
    "maskrcnn_hard3__personal": "44b61603341caeeb434c3048f5e9c7e2c893c6100abe93f5e83cddcc8d1fb62b",
    "yolo11n_hard3__original": "8bfa09ae50631f9d4735c32dddcb5a11f450072591ad88b319580e066cd6fe1d",
    "yolo11n_hard3__personal": "c025af412bafef66de03327b24a989cd2d0f0b46c3a6e9f899e065bb00e91415",
    "yolov8n_hard3__original": "b60dfb0778b39448ab4e49c835f217aee539e8b8b6915b122583d49e77ac7de1",
    "yolov8n_hard3__personal": "41f322a51490088e6cfec31a9994934567518b4c566d9873413559c7f1c58dfa"
}
RECOVERY_REQUIRED = [
    "maskrcnn_all__personal",
    "yolo11n_all__original",
    "yolo11n_all__personal",
    "yolov8n_all__original",
    "yolov8n_all__personal"
]

#### Input lock dari artefak lokal terverifikasi

In [4]:
PIPELINE_SHA256 = '34c9460d0815aa9a68306dc0888cf81d5a0927a7221da62291528e781c1b45eb'
LOCK = {
  "class_order": {
    "all": [
      "backpack",
      "bed",
      "blanket",
      "book",
      "bottle",
      "bowl",
      "box",
      "cabinet-merged",
      "cardboard",
      "ceiling-merged",
      "cell phone",
      "chair",
      "clock",
      "couch",
      "counter",
      "cup",
      "curtain",
      "dining table",
      "door-stuff",
      "fire-extinguisher",
      "floor-other-merged",
      "floor-wood",
      "flower",
      "food-other-merged",
      "fork",
      "hair drier",
      "handbag",
      "keyboard",
      "knife",
      "laptop",
      "light",
      "microwave",
      "mirror-stuff",
      "mouse",
      "oven",
      "pan",
      "paper",
      "paper-merged",
      "person",
      "pillow",
      "plate",
      "plug",
      "potted plant",
      "refrigerator",
      "remote",
      "rug-merged",
      "scissor",
      "scissors",
      "shelf",
      "sink",
      "spoon",
      "st-",
      "stairs",
      "stove-merged",
      "table-merged",
      "teddy bear",
      "toaster",
      "toilet",
      "toothbrush",
      "towel",
      "toy",
      "trash-bin",
      "tree-merged",
      "tv",
      "vase",
      "wall",
      "wall-brick",
      "wall-other-merged",
      "wall-stone",
      "wall-tile",
      "wall-wood",
      "wallet",
      "window-blind",
      "window-other"
    ],
    "hard3": [
      "bed",
      "book",
      "cabinet-merged"
    ]
  },
  "models": {
    "yolo11n_all": {
      "family": "yolo11n",
      "scope": "all",
      "local": "experiments/yolo11n_all_classes/weights/best.pt",
      "alias": "yolo11n_all_best.pt",
      "size": 6077156,
      "sha256": "bcfc5e73096871f15394987a6683531b9a9baabb3e14026c575bd2a94a14defc",
      "historical_mount": "/kaggle/input/models/wayanradityaputra/cpmk3-yolo11n-seg/pytorch/all-classes/1/best.pt"
    },
    "yolo11n_hard3": {
      "family": "yolo11n",
      "scope": "hard3",
      "local": "experiments/yolo11n_hard3/weights/best.pt",
      "alias": "yolo11n_hard3_best.pt",
      "size": 6041572,
      "sha256": "6e17a8d8b2a4291da424d8ecaddb080744e5c83f17ce375aeaa550b8cae1f775",
      "historical_mount": None
    },
    "yolov8n_all": {
      "family": "yolov8n",
      "scope": "all",
      "local": "experiments/yolov8n_all_classes/weights/best.pt",
      "alias": "yolov8n_all_best.pt",
      "size": 7078260,
      "sha256": "4e19c04c344e0849cbf1d1154db89233ab9be5f0ffb817513e7ad611e28b19f6",
      "historical_mount": "/kaggle/input/models/wayanradityaputra/cpmk3-yolov8n-seg/pytorch/all-classes/1/best.pt"
    },
    "yolov8n_hard3": {
      "family": "yolov8n",
      "scope": "hard3",
      "local": "experiments/yolov8n_hard3/weights/best.pt",
      "alias": "yolov8n_hard3_best.pt",
      "size": 7043124,
      "sha256": "87f5775efe1cc799e805f9808161c15adae8e7ab7e1ff496dacbd722a815af03",
      "historical_mount": None
    },
    "maskrcnn_all": {
      "family": "maskrcnn",
      "scope": "all",
      "local": "experiments/maskrcnn_all_classes/weights/best.pth",
      "alias": "maskrcnn_all_best.pth",
      "size": 177777774,
      "sha256": "32165bb4e2d822300237ce64f5a969423cd4dbd6f32b559bf5a4cd8ade31cafe",
      "historical_mount": "/kaggle/input/models/wayanradityaputra/cpmk3-mask-rcnn/pytorch/all-classes/1/best.pth"
    },
    "maskrcnn_hard3": {
      "family": "maskrcnn",
      "scope": "hard3",
      "local": "experiments/maskrcnn_hard3/weights/best.pth",
      "alias": "maskrcnn_hard3_best.pth",
      "size": 351112522,
      "sha256": "8337a3a93c6dcbbc68fcb0b8534d7660461109be03facff22910ecb5d87939d6",
      "historical_mount": None
    }
  },
  "references": {
    "freeze": {
      "name": "ground_truth_freeze.json",
      "size": 2084,
      "sha256": "2dffd7406da3f09f3ca265f2966be8844bcddb9210c859b0a82374524818fca7"
    },
    "mapping": {
      "name": "category_mapping.json",
      "size": 22171,
      "sha256": "dcf63b6daad2dbb5145b203e86d3e01047161537b67de8215726ca584949fa24"
    },
    "personal_provenance": {
      "name": "dataset_provenance.json",
      "size": 3041,
      "sha256": "5d1f1c574338f737318116a3009ec48ea04534f25e630a4e30ab445a1ef2d79c"
    },
    "personal_all": {
      "name": "all_class_master_corrected.coco.json",
      "size": 1575667,
      "sha256": "85b204036437434a055af3b2ef80add2a54e48e39851a2b2e355563222d6ac23"
    },
    "personal_hard3": {
      "name": "hard3_corrected.coco.json",
      "size": 308559,
      "sha256": "938bab7a90ac3ccfec8c03ec075c6cf09b24c0ccdb1d41e5b97742280d0f6447"
    },
    "original_all": {
      "name": "_annotations.coco.json",
      "size": 2531061,
      "sha256": "1b756f0cab2be19fb338e35aea80c3a1e40eca6c5343f4daee9bf3d675bd466f"
    }
  },
  "counts": {
    "original_all": {
      "images": 359,
      "instances": 5344,
      "classes_present": 69,
      "positive": 359,
      "negative": 0,
      "class_counts": {
        "dining table": 112,
        "plate": 104,
        "food-other-merged": 149,
        "floor-other-merged": 239,
        "chair": 353,
        "cup": 131,
        "wall-other-merged": 650,
        "light": 295,
        "bed": 71,
        "mirror-stuff": 34,
        "curtain": 107,
        "ceiling-merged": 149,
        "window-blind": 71,
        "door-stuff": 64,
        "potted plant": 138,
        "tv": 35,
        "cabinet-merged": 110,
        "table-merged": 258,
        "rug-merged": 107,
        "window-other": 144,
        "pillow": 37,
        "floor-wood": 130,
        "book": 349,
        "paper-merged": 51,
        "toy": 41,
        "cardboard": 14,
        "toaster": 4,
        "bowl": 76,
        "wall-brick": 25,
        "laptop": 27,
        "person": 413,
        "handbag": 125,
        "toilet": 14,
        "towel": 19,
        "sink": 30,
        "couch": 75,
        "wall-wood": 30,
        "refrigerator": 5,
        "bottle": 58,
        "counter": 67,
        "stove-merged": 33,
        "wall-tile": 47,
        "pan": 34,
        "cell phone": 40,
        "plug": 14,
        "keyboard": 7,
        "oven": 13,
        "clock": 4,
        "wallet": 5,
        "backpack": 31,
        "mouse": 8,
        "spoon": 36,
        "fork": 24,
        "stairs": 6,
        "vase": 39,
        "trash-bin": 4,
        "blanket": 13,
        "knife": 19,
        "teddy bear": 6,
        "scissors": 6,
        "paper": 6,
        "fire-extinguisher": 4,
        "scissor": 2,
        "hair drier": 2,
        "microwave": 7,
        "toothbrush": 14,
        "flower": 6,
        "box": 1,
        "wall-stone": 2
      }
    },
    "original_hard3": {
      "images": 359,
      "instances": 530,
      "classes_present": 3,
      "positive": 137,
      "negative": 222,
      "class_counts": {
        "bed": 71,
        "cabinet-merged": 110,
        "book": 349
      }
    },
    "personal_all": {
      "images": 40,
      "instances": 213,
      "classes_present": 38,
      "positive": 40,
      "negative": 0,
      "class_counts": {
        "towel": 2,
        "handbag": 6,
        "door-stuff": 7,
        "curtain": 3,
        "wall-other-merged": 29,
        "plate": 5,
        "spoon": 4,
        "knife": 1,
        "toilet": 1,
        "wall-tile": 2,
        "floor-other-merged": 21,
        "backpack": 4,
        "pillow": 5,
        "bed": 10,
        "chair": 4,
        "cabinet-merged": 12,
        "bottle": 18,
        "cup": 1,
        "window-other": 3,
        "book": 11,
        "table-merged": 10,
        "laptop": 4,
        "paper-merged": 6,
        "dining table": 2,
        "fork": 2,
        "cell phone": 1,
        "mouse": 3,
        "keyboard": 3,
        "paper": 4,
        "box": 10,
        "light": 3,
        "shelf": 7,
        "rug-merged": 1,
        "mirror-stuff": 2,
        "trash-bin": 2,
        "toothbrush": 2,
        "plug": 1,
        "ceiling-merged": 1
      }
    },
    "personal_hard3": {
      "images": 40,
      "instances": 33,
      "classes_present": 3,
      "positive": 25,
      "negative": 15,
      "class_counts": {
        "bed": 10,
        "cabinet-merged": 12,
        "book": 11
      }
    }
  },
  "images": {
    "original": {
      "0": {
        "name": "114_jpeg.rf.bdf757579a02c0e1f1035f14d15a3890.jpg",
        "size": 197002,
        "sha256": "72c0416f4f4f47853614ad0b57871c6b06c87b09e9a2baa9ec9fcbb70029da86"
      },
      "1": {
        "name": "099_jpg.rf.adfa2556136b57bb92d9d42ac345513e.jpg",
        "size": 7245,
        "sha256": "c09a07a367ca6dddf90af0ccf101e6e380b2bebc174cc4cd7a65872259155b2e"
      },
      "2": {
        "name": "346_jpg.rf.c19951ec89774ceca284693d02206915.jpg",
        "size": 100213,
        "sha256": "613ed5ef5baacec764cc2e3990e13d9a3aa5bf9dbcacb91b0a2dc7795afb1102"
      },
      "3": {
        "name": "323_jpeg.rf.b734fe13f3161815cb32f982a4243f8a.jpg",
        "size": 156648,
        "sha256": "85e5a03181048814aa6ebcde1b22ad4f4d6a19503ea3a1126e259b0ba94dfccd"
      },
      "4": {
        "name": "343_jpg.rf.b603699dbd7dad70c38e433f68925761.jpg",
        "size": 107421,
        "sha256": "4bf4fc600efe591a3a540cb2792b22b215eaa6aad4e7d884434df67e21283613"
      },
      "5": {
        "name": "132_jpg.rf.bacefe2828cd506fbc9bcc3246a7b02d.jpg",
        "size": 50089,
        "sha256": "03c68c209bc419d07326dc5919b95a50e4cc5bdf8ba199d671c16c961ac9a6dc"
      },
      "6": {
        "name": "109_jpeg.rf.bd0d8199ee5d08f539aed34dfc34d8a3.jpg",
        "size": 136477,
        "sha256": "a69f6b6a5b58b07f75fad3c1c9f38ea8d24ad8fcb991e15fb444255e497508a6"
      },
      "7": {
        "name": "241_jpg.rf.bbb29905ac7ccc3de9725c541801a64b.jpg",
        "size": 9584,
        "sha256": "8486bb3711338323ad537d4aee437615a39e57bbdee281f2bbc45ded7a51d4f6"
      },
      "8": {
        "name": "177_jpg.rf.b0799e2acae4df88044955e1c2e59342.jpg",
        "size": 90353,
        "sha256": "f0ff413584a0004f81c2ef925a77ca5e801a49e1a1db1717d0bf42d1250edbeb"
      },
      "9": {
        "name": "129_png.rf.beb27f1e5ae6e55d058b85364db52d16.jpg",
        "size": 23231,
        "sha256": "80e7653554d191ed650253900f7ba66edd63777d7a5a6b757bcb223497cebfee"
      },
      "10": {
        "name": "304_jpg.rf.adb0d2520c6c665b007cfa3bb47b3306.jpg",
        "size": 153608,
        "sha256": "a8c7f0916978c0f4577aefbaffe89c31b12310825c2f6229e41f063183295916"
      },
      "11": {
        "name": "033_jpeg.rf.b78f7dd7211b2bec8f16c764864e215b.jpg",
        "size": 53072,
        "sha256": "1765444b83d5b9931a2472d26bf8852dbd43095c44f2e2957d8de92446cff803"
      },
      "12": {
        "name": "285_jpg.rf.b08bc5599dbdecacc35e8808e3e2c14d.jpg",
        "size": 378653,
        "sha256": "308fd5b85b00d16d894131da54022d726c97509adb4e334370cfa86976f12607"
      },
      "13": {
        "name": "249_jpg.rf.c1e9aacac39c1e8d08c0b4a0a7df61e8.jpg",
        "size": 59899,
        "sha256": "80f9bb56c0ad640a82fd8db59415cf40e0a7cb8a24f53c98d3c2865f59907cf5"
      },
      "14": {
        "name": "116_jpeg.rf.ac96698ba4b890db16a3e8fb7022707a.jpg",
        "size": 201128,
        "sha256": "cc14432936eecf8b74caf1b5376659a92283c6f157e69e6f6c67e6eaf641c720"
      },
      "15": {
        "name": "009_jpg.rf.ae66a6121188f40c6e0fc183e343786b.jpg",
        "size": 11274,
        "sha256": "1bc4e205e14ba95a2f02dc3f41da06959aa2c03c560155223fc47eb028563426"
      },
      "16": {
        "name": "050_jpg.rf.c194582c8d8c2038a59cdf16cccaddd7.jpg",
        "size": 43555,
        "sha256": "daa7a75478483453f64ccb947fa7403d7f4fa8b2e0e759b781f04609ef3108c5"
      },
      "17": {
        "name": "133_jpg.rf.c1e588c65f5ba60e4a3ff4135e611a03.jpg",
        "size": 34178,
        "sha256": "432d00684eaa46d9b3a8410289acb2e8cf2c038b8c5447681901e4033ef79196"
      },
      "18": {
        "name": "046_jpg.rf.b1c899fe8b626c570875f2dff1089514.jpg",
        "size": 33026,
        "sha256": "12ffdee7d1756b5a463bc37d4d50d7ea8748d15c9954ab0109f0250d3513db53"
      },
      "19": {
        "name": "274_jpg.rf.bd0d065935440e4fbbb70344eaa1ad1a.jpg",
        "size": 144194,
        "sha256": "dd9a6619586577575bfe2b76f95d8fa45f871c1847ed85a66e12e82197e252fe"
      },
      "20": {
        "name": "267_jpg.rf.bbf9ca259a7c3467533404ee98ad4550.jpg",
        "size": 59506,
        "sha256": "f993802d37ac658c6faf05b33ebc40bb871899786855022f99ecdac8847bcbc8"
      },
      "21": {
        "name": "036_jpg.rf.b645ae298ccea2e74d5ff07b4da8ac60.jpg",
        "size": 43594,
        "sha256": "688f1da2c8a38091625490543dbc76a5aa0b858ed324d931f4de4efc13c20585"
      },
      "22": {
        "name": "098_jpg.rf.ba5fdd58be4f0a3fbe6766da6c7bb85d.jpg",
        "size": 32069,
        "sha256": "a13b9fdf15c294f971f5a89c36fec6e18cf375d360f19ca96a5d623e8b355260"
      },
      "23": {
        "name": "007_jpg.rf.befa33d0982433cf61bc4660d53d757b.jpg",
        "size": 73197,
        "sha256": "54391fd952ccacfa6b6833f16eef0964199ffdb890ef192e5d7b77baf0bf45f2"
      },
      "24": {
        "name": "135_jpg.rf.b32c72139ef54827b320e65652f8aeea.jpg",
        "size": 74939,
        "sha256": "5b0c8e410164c4cc7619209fdc8349145ecd1b51016a1d5251ab1971ccb21233"
      },
      "25": {
        "name": "024_jpg.rf.c3e1b823b0efb20792949ee0f1cbaf4e.jpg",
        "size": 52755,
        "sha256": "d8c8316e90c70f0efb553f472f83cec1f2a3bc0dad20ccc30c099cbfbcd6d238"
      },
      "26": {
        "name": "075_jpg.rf.c27a3326d05400ec5a0d0529ffbdd356.jpg",
        "size": 25771,
        "sha256": "e8c593b3df53192a2afe7ef0a745251ffaed2b93e5d3649948c0ae01cd3eab98"
      },
      "27": {
        "name": "215_jpg.rf.c60946a6083d67619af1415a5d492347.jpg",
        "size": 127421,
        "sha256": "1ca955db10bcce822ae22f99a5f900b20708a5e5d2177e14f736e2147785a08e"
      },
      "28": {
        "name": "014_jpg.rf.c7aefb8f589d0fadf9aabd0d8bca562b.jpg",
        "size": 9572,
        "sha256": "bf586948bdff0941aab1eb6b78bd5a9b5391dd211fb31c087d9198be1a159e22"
      },
      "29": {
        "name": "315_jpeg.rf.c3703af8afa8979c2c83ee05ae1741d0.jpg",
        "size": 121121,
        "sha256": "ed99d65b644ce11d5b3abf475339f864e2ae9a9d8800e72b286bf2f5f5eb7b90"
      },
      "30": {
        "name": "254_png.rf.c5fbd27f2971b50af2cb509c140143ec.jpg",
        "size": 27905,
        "sha256": "e635f8064a29e11f9c055b063c88a7881b84b45911719c85a0cb894cea45aa34"
      },
      "31": {
        "name": "155_jpg.rf.cdee537758d83fcbe0d4c1b7620d5382.jpg",
        "size": 17219,
        "sha256": "7e8f8703a7e4ebe87a67a5b8a1d1627e2e22216e0729ba45b3ad9484b006ae82"
      },
      "32": {
        "name": "010_jpg.rf.c492c9c71bf55650caf2a6cff92aec91.jpg",
        "size": 47952,
        "sha256": "42446c886f2b3b50217c91a834b47b273dd47e50ba5d9bd49c0bf40e2a372a63"
      },
      "33": {
        "name": "242_jpg.rf.cb470213705196585ae201605c81c050.jpg",
        "size": 8856,
        "sha256": "b902bce2adb9ca0238bcb06601f3ea123aa4f7b30132a9a499425cdb0421f40b"
      },
      "34": {
        "name": "113_jpeg.rf.ccec22c8ee859e4f7591ad43f66fddd7.jpg",
        "size": 227168,
        "sha256": "b7b03998b581941418ace4675f5debabc02361191b042baaf57663846486c068"
      },
      "35": {
        "name": "189_jpg.rf.c853e42bfd4a587a75f573553ec45699.jpg",
        "size": 39573,
        "sha256": "26749c4e21191f4521f85547bb5a3869f8a783277c7b475ffe21d0bef02a7f98"
      },
      "36": {
        "name": "032_jpg.rf.c72873d2a481c9bb7861e61e342af32a.jpg",
        "size": 11247,
        "sha256": "6baf15d72c7349d9f7ecb8439fe9de2400085e495a642311fa090149b6dd73f1"
      },
      "37": {
        "name": "235_jpeg.rf.c9f245b0fee3486a05d4977a0394631b.jpg",
        "size": 117186,
        "sha256": "71b735200753c7132dfd29b6152a90507c2c6395b0dfb0e55f1b1201a563726f"
      },
      "38": {
        "name": "130_jpg.rf.cd307142f349c9c97a6f7b641222086d.jpg",
        "size": 42366,
        "sha256": "6c251c5186cf3a0242364492886b04e2ca065e06e6c2bb14f92df1092e10383c"
      },
      "39": {
        "name": "162_jpg.rf.cd72e3d7eedcd72cc85b492c890506f1.jpg",
        "size": 147403,
        "sha256": "b2bc65721676a1fd609be58b821e39d4e36816a7ac47366d397186de5ed75b8a"
      },
      "40": {
        "name": "056_jpg.rf.c6f1fbdfcaa99165039d01d9ff428e8a.jpg",
        "size": 49503,
        "sha256": "b252ba298ec74f36cf993a8fdcb435257ff477a6c345a1ea3cb4b30a222038d5"
      },
      "41": {
        "name": "236_jpeg.rf.ce43674993187cb554fcbd2c85146baf.jpg",
        "size": 175623,
        "sha256": "0781fce9bcc66165a5798b91292219b7a04f36f672356e5ff2603d8ffa31c300"
      },
      "42": {
        "name": "161_jpg.rf.cea0dd030cb257194adf3d3d67e6a068.jpg",
        "size": 53226,
        "sha256": "8fc5d596cd34ac1f95e7eab9f07384fa75c333576ed038464f10a758e4c8f0b4"
      },
      "43": {
        "name": "019_jpg.rf.d1141c53a060d64839040d2105db1a7b.jpg",
        "size": 8217,
        "sha256": "80b7fb4f60291ba0ee213b23eaf470e1d475221e22dd2752d93c56d3ddde3854"
      },
      "44": {
        "name": "015_jpg.rf.d40dfea3d5d9af38fcb883f2354a5e9b.jpg",
        "size": 42454,
        "sha256": "b835f1e4d3627fcf382053c037c76c1b43e804637b463eefa4c81fe44b9616a0"
      },
      "45": {
        "name": "217_jpg.rf.d8bb7e3521ae5abdf5cfd0f7ec35aecc.jpg",
        "size": 46642,
        "sha256": "3b7fb001d67c211bbd8737343ed6499f3e16b81031129eda1a54a62711fb3b31"
      },
      "46": {
        "name": "338_jpg.rf.d09d3963500f7a0b6d5ac93516508242.jpg",
        "size": 250459,
        "sha256": "020b79ff64ec577ca1e27f2294cedaa3252908399f764db59ec60bb1dd96e388"
      },
      "47": {
        "name": "320_jpg.rf.da92d2a204f091c9fcf09bc706e024e7.jpg",
        "size": 4995,
        "sha256": "b9323b58ff60b4bdc273fc9f68f2a561d1aa61fa07005f2ed29f65a2971e5357"
      },
      "48": {
        "name": "078_jpg.rf.d5d32b5a850f48453b414924ae0fc0a8.jpg",
        "size": 50188,
        "sha256": "8257b96c8f804d67bb1225a12a8b1c409c7dd1abd7717878edd8aed1a7518828"
      },
      "49": {
        "name": "101_jpeg.rf.d3d4636a7d24ca86481e46bb38aa6f6a.jpg",
        "size": 109194,
        "sha256": "1e2b234442ad767d4d94ecbecb20dd7375b16645a5040b210fe1cde5be8d1b2a"
      },
      "50": {
        "name": "319_jpg.rf.dcb106004b4ea0558a0e1495c7517651.jpg",
        "size": 8453,
        "sha256": "0190971393aa8c49b5cc7afaf78fd050c759415e27a712461ae5c061f6d4edc4"
      },
      "51": {
        "name": "016_jpg.rf.d9375fc2ffb980b9622d1f685b25d8a8.jpg",
        "size": 11319,
        "sha256": "f8b9c413eaf5df6d1b760d07945a7bb5de30f0f2d7b7f6dd4fbac0f366e6ebfd"
      },
      "52": {
        "name": "110_jpeg.rf.dbbf10f53998b140711cc2545cd4b4b0.jpg",
        "size": 164884,
        "sha256": "3ab9f24400dae007e821bba29e2b404828a5aabd4049c9a9ba74586c33fdc67a"
      },
      "53": {
        "name": "147_jpg.rf.dc6a95956d847d17dec4e0859ddc5e34.jpg",
        "size": 9816,
        "sha256": "bbbc621ec4935a8b6d3c85a5891595ce2a26b4d9466ed14a2389116ad6dac0cc"
      },
      "54": {
        "name": "121_jpg.rf.dce690207d3c517b5e479199530d7735.jpg",
        "size": 14961,
        "sha256": "7b05181b53a5305ab2b7aa40aeeba19b0ac26d63107f4e37bbbb4ae14d626f9f"
      },
      "55": {
        "name": "286_jpg.rf.dbfb858a690d2a510d89ab8a00c91b58.jpg",
        "size": 14377,
        "sha256": "004d79d81c108f0f9ab28f9c419b38bf10b969d9cdc4ffe62be571a3db407e75"
      },
      "56": {
        "name": "070_png.rf.daf36c965ed0f6fb63d6460f86f375b5.jpg",
        "size": 141655,
        "sha256": "b5b0b258b9e5ec9ccbf494d580492763b93267506c9201a530c4249e6d69c57c"
      },
      "57": {
        "name": "359_jpeg.rf.dd47b1d6727d1f638156dd7dc12c0ade.jpg",
        "size": 30433,
        "sha256": "cc506db0565c1442e91fadcf0cf1a270470b2c3f593d2ff85a254f420284e1be"
      },
      "58": {
        "name": "066_png.rf.dd2ab1e83344b4ce1e6a7817b06df728.jpg",
        "size": 26865,
        "sha256": "d6f8825b64179ad2cf341cf610af6d617c9112986d1d42da88c91e46b1ff2666"
      },
      "59": {
        "name": "255_jpg.rf.e01c870b67b2dc17a50798080b4595c4.jpg",
        "size": 76492,
        "sha256": "29676a7e88bf813fc3c8e0d9a97d93e19676fbdc00c01c9d84a3984b7ccab369"
      },
      "60": {
        "name": "148_jpg.rf.dd9bff45b238bdb4fcfc68cf2a221a54.jpg",
        "size": 73999,
        "sha256": "90f554cd966e4d32a5b2b98909b92062396b9b14daf3a398efca2b64fe5e5f85"
      },
      "61": {
        "name": "280_jpg.rf.dd06f1242398617c5fcc5a576dddf6f5.jpg",
        "size": 95723,
        "sha256": "43b9b2cf21fbd821f1dfc817f0d575d94706c71227f266fd247e26cac98f4cd4"
      },
      "62": {
        "name": "220_jpg.rf.e199956bb7b64d702dc77f274f0f6251.jpg",
        "size": 9925,
        "sha256": "4032d4fe969f1b102b8ac944185fece9cf8a44c7cf98918d604825555096e730"
      },
      "63": {
        "name": "295_jpg.rf.e033224411af9e27c6c375c245a7a6e8.jpg",
        "size": 68383,
        "sha256": "e95afa113fbe553a7bcc3e477850736a58c124dcf3fbdc93acd7126a293b251c"
      },
      "64": {
        "name": "030_jpg.rf.e2182c6e7f8845a46d6ed62b007adacc.jpg",
        "size": 56101,
        "sha256": "72681fcfa62466d1b7d164f37decbcb637b39598eec41b3e16423d1f807ac75b"
      },
      "65": {
        "name": "237_jpeg.rf.e23d2e6e1c0385b1fbd70cfcc93f5457.jpg",
        "size": 140705,
        "sha256": "e797b9ca5210ed649d2b5081a540fe3fc68ab1f2fcfaabd00d3212195138b220"
      },
      "66": {
        "name": "049_jpg.rf.dd831484dd3626cb619186dfd6d5ba57.jpg",
        "size": 41191,
        "sha256": "923068979e8ab081fc9931d5188fd7d5877b1baa30dd10d895b55b07b91430f2"
      },
      "67": {
        "name": "060_jpg.rf.e4bba298f0ec333629e235f4a160b6fc.jpg",
        "size": 33203,
        "sha256": "b9a348e60edbe1550b808c93f170cbba7a83ff9df50d7d81ab3131f4ce879e43"
      },
      "68": {
        "name": "287_jpg.rf.e08b8bc57dc2082d912881bdccae51f4.jpg",
        "size": 37900,
        "sha256": "0efc7b60908639ac24882610fb99393674ddef4eb1b0af11336aa300139b215e"
      },
      "69": {
        "name": "074_jpg.rf.e3e60c921ecf97aaee1d7270cd51e72f.jpg",
        "size": 14775,
        "sha256": "37a32f3f3f16022e923170f7020aea66f14525826fa1a411c5eb003ba28623fe"
      },
      "70": {
        "name": "091_jpg.rf.e316184e643a000e5d907124608d5174.jpg",
        "size": 86354,
        "sha256": "7d756aa7ebd1d21fc56438cafc4f4a05efdfc31aefa7b7c25c31cc899cad28d4"
      },
      "71": {
        "name": "298_jpg.rf.e4ef75f2f2f0b4cbf6edcdc1402bfc3d.jpg",
        "size": 125310,
        "sha256": "f6be5711eb8658c046810fbd3a467ca747f9d404acfa2a714f2f846fed57e8b5"
      },
      "72": {
        "name": "201_jpg.rf.e2b37cac09e6801b773ac573321bbc5a.jpg",
        "size": 12177,
        "sha256": "53d8797b55f0b2266185463f70e88acb251b536ba539f829f7094d73ff2fa819"
      },
      "73": {
        "name": "084_jpg.rf.e60a89caf09ebf06dc8e3158ecad3541.jpg",
        "size": 222803,
        "sha256": "10357c312dc209d13f3a573bb1e6e0cf623a817348e80f8002b68ae202987fbf"
      },
      "74": {
        "name": "261_jpg.rf.e8b9bc47de58d5ba6d84700b21751a88.jpg",
        "size": 269441,
        "sha256": "500b3553794df68f4689976773a12c41870dc664da3e893baa11dfa6a92727b7"
      },
      "75": {
        "name": "037_jpeg.rf.e0764e2ad3ca3e586d310c212a6364bf.jpg",
        "size": 145725,
        "sha256": "b64776e490ac9293542e5106d3680b43342962dddbbe05a596ecc155eb66f181"
      },
      "76": {
        "name": "187_jpg.rf.e8d4b210e8a0cd2e1ae58ac4359abb48.jpg",
        "size": 18655,
        "sha256": "8a1e36314f34a7d2a0296e873c7653ba56784dbce7a0f6a529737d51f72ddfc8"
      },
      "77": {
        "name": "353_jpg.rf.eb1f2b4fecabe30702d834fdc44a5dea.jpg",
        "size": 49759,
        "sha256": "849b9d23fa30dc0206a46e949119db2a4c4f268a5553dbe13a49240122ec95d4"
      },
      "78": {
        "name": "093_jpg.rf.e650bc0b942686077c7fe6e623fc3644.jpg",
        "size": 75617,
        "sha256": "eea6053c65175779f4a7eff350ad418a3d23d20d7dd81a59d66405522050d4d0"
      },
      "79": {
        "name": "245_jpg.rf.ec4b64266b89ddeba17e9dd64cf2e98b.jpg",
        "size": 14804,
        "sha256": "d92bdbb9984c2db35592743ad2fa351db55c78c7bb3ee852f207cad873420a5a"
      },
      "80": {
        "name": "021_jpg.rf.edbdae081ff87bed89914180db466953.jpg",
        "size": 43637,
        "sha256": "355e2cf6a22dc061968e61948b157003b17254ccf4b670af1f2dea483b580c8a"
      },
      "81": {
        "name": "149_jpg.rf.f052def26613fc6180228df3376155c0.jpg",
        "size": 9244,
        "sha256": "6847fbf08c3b23ab431890c0845676a81f3cecb00643bc2b7ecefe2ed0f934ab"
      },
      "82": {
        "name": "073_jpeg.rf.f1e5c7c7a1f2a20b32c16c8972ef7bcc.jpg",
        "size": 73806,
        "sha256": "4f853f788d45a9d27a2fe92a1ffd1bf10c12230bda672412e9c2861f54838d6f"
      },
      "83": {
        "name": "355_jpg.rf.edc99ef01671605b004e27976c6fa9b1.jpg",
        "size": 274050,
        "sha256": "f9594b07d8588a8b5113de44e7748d553c1e8c303b344fd62a8fbc0066eca8e9"
      },
      "84": {
        "name": "013_jpg.rf.ef0c857e10bade755d3824e1f6e19641.jpg",
        "size": 129185,
        "sha256": "61ab8e8dc702c973835496191692da748f42c1beac7105f54ae3573a1298632c"
      },
      "85": {
        "name": "005_jpg.rf.f2248d837b7b97b5870fc8796a0eefd8.jpg",
        "size": 255851,
        "sha256": "989023c7a612e6961c724fbb14354e065dda8c71189005d0b1b22f26fb1f212b"
      },
      "86": {
        "name": "266_jpg.rf.f7055269dccce6720c4333762e87eb47.jpg",
        "size": 19375,
        "sha256": "5cd8a803cdbc75f691a08174ef93c51c6fa41db3a502cca1db3b1441b439fc05"
      },
      "87": {
        "name": "202_jpg.rf.f69fe19a0f3a5b1ae7604c00dfd9e908.jpg",
        "size": 70304,
        "sha256": "17320f8861d143e4c35747a2d60d3258066fceae685d76a495725d7547896312"
      },
      "88": {
        "name": "351_jpg.rf.fb8300375f74c897d373404512ca5cb5.jpg",
        "size": 109406,
        "sha256": "f056c6b0e00427ee7f9f9ba91998c91782bdfaedbda047a1e098bcc32e657d7a"
      },
      "89": {
        "name": "271_jpg.rf.fb9061e4d17b0f2a393f7f23fc9ec284.jpg",
        "size": 216620,
        "sha256": "0c5af418bc126c1d43f65f52b21cfd58688978e82617f6b24196906199b63d19"
      },
      "90": {
        "name": "329_jpg.rf.fc104d050c0e0354a3bfe02f9eaafa75.jpg",
        "size": 26661,
        "sha256": "c8dd40f6e0f7bdcf42c47a0c3752b06fd85fb61303f8b8e68722b341b38fc19f"
      },
      "91": {
        "name": "169_jpg.rf.f668372cb3cd33d62c2fb014171e249d.jpg",
        "size": 132691,
        "sha256": "8afc3f11587eb0e7f91c57d6db5e683adaa85057287403fb25ea4c88ccdc452b"
      },
      "92": {
        "name": "054_jpg.rf.f953eecd1db5524b9fb242309e091cc6.jpg",
        "size": 58360,
        "sha256": "2318e949e03a5ac5445ff62fb40d36b999906da6a28c7c2fde15dde323ac0e82"
      },
      "93": {
        "name": "105_jpeg.rf.f72f37f07628caa5e0c9b6f12a1debc7.jpg",
        "size": 169902,
        "sha256": "fdbd9342ab4def2a59ae89c09283e427efb9a364edbed9cef668e068d119ee23"
      },
      "94": {
        "name": "258_jpg.rf.fdcb79215b4bbd9de1d56850a8ee7dcd.jpg",
        "size": 15385,
        "sha256": "63950637ab153235cefe82026d59885a0139727186cc79fc959cea495a449095"
      },
      "95": {
        "name": "262_jpg.rf.fd63032e6fa45cd2dfbae150c1ea8223.jpg",
        "size": 9330,
        "sha256": "583ecb842d88b6dd3003eeb5ce0434d3e8864aceab48b04284426b896f102620"
      },
      "96": {
        "name": "039_jpeg.rf.fc6a977f6e82baa29fe63e5a5dc39792.jpg",
        "size": 55488,
        "sha256": "f0a1360c37d4c73be5a4340fd2dfb28310aa2f1653291e0d59cae1df2e99a50b"
      },
      "97": {
        "name": "095_jpg.rf.fa97a231ab47029bdfd5bccffd4b7a23.jpg",
        "size": 37544,
        "sha256": "7399b2e4b1df05120c1d3bb942cd64c1d768ac745586dc0774ffa197ad5912ff"
      },
      "98": {
        "name": "195_jpg.rf.fda41288d097345e671db55b498657bb.jpg",
        "size": 10059,
        "sha256": "16994732acdf95a3c64a9387ef10c6fa34b6feaa9c68ed108a50f8eeac2cd5df"
      },
      "99": {
        "name": "083_jpg.rf.fd33d51c29bc939450cd20f57994aa38.jpg",
        "size": 348387,
        "sha256": "3908b134ec41fec9cf3706f1a770d7f9922eb1c955da343ca2c1378cce48422c"
      },
      "100": {
        "name": "247_jpg.rf.fcd488e353a8094a177501250177ed79.jpg",
        "size": 85155,
        "sha256": "7ae292ccdc939d0a63a4b75b9999df079cd551357a7823661c1f2f81d2592400"
      },
      "101": {
        "name": "334_png.rf.fd93edb0af1c0ce81afe2f4094f49b3e.jpg",
        "size": 42765,
        "sha256": "0368dbcefbda70228de02bd991b283728e4a05941ca38d0e301b5725ff97b3e9"
      },
      "102": {
        "name": "325_jpeg.rf.fb98501c22f5516af181c0904d61b88a.jpg",
        "size": 359049,
        "sha256": "d497734f7ff0a04c608b29c205f07aad72dbda006c3d4820846dbe4126c5665a"
      },
      "103": {
        "name": "145_jpg.rf.fdb63401e4c51339b5ffea88f5c3d8be.jpg",
        "size": 34067,
        "sha256": "0cc60835eaad33c4741ba1bf2ce820b4ced62399c8b012c382dab9204bf10da0"
      },
      "104": {
        "name": "182_jpg.rf.fd1d2d2eea2758f0b69987a5b128f764.jpg",
        "size": 195244,
        "sha256": "97b1e2a4a9fff3eef6b8590ca20c8b34e29a467cbd966e697f66d6f927c88a35"
      },
      "105": {
        "name": "302_jpg.rf.fe800a7c96bd4d1ff06a17a984aabf5b.jpg",
        "size": 40604,
        "sha256": "0ce384a15ad134fcf31717d86fca87212a786bb2c879549732c760af793c63ea"
      },
      "106": {
        "name": "154_jpg.rf.ff077b4c9c52472285547da19d838ca0.jpg",
        "size": 318366,
        "sha256": "2c9c2c6ee1d0f047899428f14e2bead2aa93e587e014766a47b1ffe4e4c59868"
      },
      "107": {
        "name": "180_jpg.rf.fe174207556b20d8a86b36ccef763678.jpg",
        "size": 62770,
        "sha256": "b6eadb865e396a5ff360fa24d07dba52afd708a1cc0f27e890f32706dc2672d6"
      },
      "108": {
        "name": "089_jpg.rf.fdf08f23f0c3521cd67e61417ff13f5c.jpg",
        "size": 54223,
        "sha256": "a86828ffcff4334c260e3b2c717096dff6b844f6b9730653203a8c94e9b1a2c4"
      },
      "109": {
        "name": "112_jpeg.rf.0c0a67001b05080632bf4bf0f0dbbf81.jpg",
        "size": 131593,
        "sha256": "cf8d338ac486c0036eb344b1ae304b03c0cb4d3179b5a1cdb837d054c3af8ad2"
      },
      "110": {
        "name": "333_jpg.rf.00c5ac50ad78c6929cccb09a7dc2edb1.jpg",
        "size": 49889,
        "sha256": "5558cf4a3b3cd2269aa47c2bf502825e7dad3efc548ab7b14dfd4a027120f534"
      },
      "111": {
        "name": "175_jpg.rf.0886b01968684669c9d522296d2b85bd.jpg",
        "size": 138168,
        "sha256": "16e35e396184881ad53deb6de69a8e69ed9db7a5b6abb7dc74afb7b49d2bfe25"
      },
      "112": {
        "name": "053_jpg.rf.002f0adcbefe28f38a23d14e639c1865.jpg",
        "size": 9041,
        "sha256": "3efda0b73d514e0d271cc969af3cfcc4b6703f125569ab9099618c0a43a3baf7"
      },
      "113": {
        "name": "184_jpg.rf.108f51fd93ae37363c82bfcf45d059c8.jpg",
        "size": 81083,
        "sha256": "ceb1a2cc692b8fee3f729ca43fe1b2d9c5ba1a1b6ca1af56391a0ae893b95bbb"
      },
      "114": {
        "name": "017_jpg.rf.0ce7944aa744db2423976c455e33c1df.jpg",
        "size": 161955,
        "sha256": "a944e01075a1db17b4fc32516a11d46f1da0481bebfe93a9ff0fa0b03e573dc4"
      },
      "115": {
        "name": "061_jpeg.rf.0e5340fc48d497f57c51641f16fd84b2.jpg",
        "size": 71860,
        "sha256": "0abb45ed65d31f7332b0689862ce6debf70b819f88a8df84a1997830855b247a"
      },
      "116": {
        "name": "072_jpg.rf.1114e11f20d38da52d2c674f9d2c38e7.jpg",
        "size": 53732,
        "sha256": "cef60b641cef4798bcb412d7fabfe994d1958d800a5102def60d458ed0b15b27"
      },
      "117": {
        "name": "224_jpeg.rf.0d904c7eb5b7cccb8413491de8bbf66a.jpg",
        "size": 97171,
        "sha256": "71c311fb6f4e0d66025c2976aab3bd8547a2b015adbc6f2f3eb09614989be700"
      },
      "118": {
        "name": "200_jpg.rf.0606cb8986d31460073f0acd48270e71.jpg",
        "size": 253958,
        "sha256": "06c05483afa32c124e4178a082e579e93036b716256f0981b2046a2f8c88517a"
      },
      "119": {
        "name": "222_jpeg.rf.0800a3f9dc1143ec640609151e684836.jpg",
        "size": 81507,
        "sha256": "6ac89196d6f4fd8ac218d5316e09fd2ede310039db0141fbccb0220f92be10e0"
      },
      "120": {
        "name": "041_jpg.rf.0c720b20f77cce77d1da2096e5a140a9.jpg",
        "size": 39833,
        "sha256": "4030b82b01f210fdab1f3b7415507a2b5ef579a4146fba3f544a611cba25da9b"
      },
      "121": {
        "name": "226_jpeg.rf.008c4aa861af0e369489c674652c8a82.jpg",
        "size": 181308,
        "sha256": "a5d23183495794ba2a529fef2dce34ed7b904741d7db3282ea6d7f12d2a7e2d5"
      },
      "122": {
        "name": "234_jpeg.rf.01befa7ab5a054b7f68df959c5ef6c6c.jpg",
        "size": 131013,
        "sha256": "85a4bbf57dc8489d5ab69c806ff73d6c90cd43cc2b5afe317d37f52e6761c38c"
      },
      "123": {
        "name": "171_jpg.rf.0089e87634106adb298a6e86f58888c3.jpg",
        "size": 340352,
        "sha256": "32ef1d1548c0824ee593a74fa165071409492f6cbdc1b8e1f88c644785a95883"
      },
      "124": {
        "name": "119_jpeg.rf.04571dcb8e2ef90f8571cdd49aabb338.jpg",
        "size": 305179,
        "sha256": "c77e864546c84357dba8fb3175c9327203e214363b0eb5e19e211cbc26b2d324"
      },
      "125": {
        "name": "183_jpg.rf.04c9d71f3548924298db64b61c8a887b.jpg",
        "size": 56518,
        "sha256": "e8b6dafdc31f56e69b244b99410e7bee6c9692169d203e1f1a00fd69840cee2f"
      },
      "126": {
        "name": "102_jpeg.rf.0c82a99aca53adef8bd43a72823aabc0.jpg",
        "size": 194941,
        "sha256": "5531aa553d727090f8bf534532ad173293a1329260bad25534eceae324ffc446"
      },
      "127": {
        "name": "038_jpg.rf.087629d581008637626b77fdacdc183e.jpg",
        "size": 40774,
        "sha256": "bac3ed28eb500e3c4fe48a762b3f9a9876aacc0616768204c59d7dfc86db9943"
      },
      "128": {
        "name": "026_jpg.rf.0f204eddc09f8ae4d1921bb88a7b4d68.jpg",
        "size": 19272,
        "sha256": "fd2352cb9abac2112f010b23aefba99434c20502ff7ca17a1cdc7b60c789d736"
      },
      "129": {
        "name": "204_jpg.rf.0aa8f6d144e9e4446771b1dbe63e4135.jpg",
        "size": 113815,
        "sha256": "b2f29ffd214ada35e3e46b0571306ea18fa0b54c5348d3bd98d3137f6d6a4f18"
      },
      "130": {
        "name": "118_jpeg.rf.0bed9fd17b08c53db76ea5db6af630d7.jpg",
        "size": 296955,
        "sha256": "3fca29c1759f1517f21a9da33b26e87ac41feff2f9e9f7eaf924fe93c3e560c9"
      },
      "131": {
        "name": "136_jpg.rf.0a0ba247c3b8a4c1b2a3366090a72426.jpg",
        "size": 55964,
        "sha256": "1d96826a11fd12f1e2d2e6429f5b9f27c645675608cbccd0e3ab764c41ee0b6d"
      },
      "132": {
        "name": "212_jpg.rf.04cdd24cf66c88d075aa11e01a5bd5cb.jpg",
        "size": 70242,
        "sha256": "ac4802bc63f8c58cfeeb6504a73aa5be8d18d0c38ad8947867581aa439fb6c28"
      },
      "133": {
        "name": "042_png.rf.1296c2fe70a01c6e2e9561a27cfdab24.jpg",
        "size": 35504,
        "sha256": "d2ab066e62545693454befea811e8b2fd2129508f20be7709ef8f4b9502c22dc"
      },
      "134": {
        "name": "289_jpg.rf.0697938f2a3ced6d8b5f480177268f1f.jpg",
        "size": 19744,
        "sha256": "14e3b5e289bac2ed9225c678ff60537b6bdadd4697b3cf31602d416bc4b585e4"
      },
      "135": {
        "name": "349_jpg.rf.11afe76c8a03c137eceaf5ab79762969.jpg",
        "size": 20978,
        "sha256": "2859954c5bdd5346165a71419bc4229bb46fea0f3000d6d212af1f243b3a47e9"
      },
      "136": {
        "name": "188_jpg.rf.12d2c03ef400a957d2f6354afca1dd32.jpg",
        "size": 33408,
        "sha256": "1cdb9565689620d8a7be49f3d7b4eeff9ea58bfc11eeb8dd0c3219d05784c9cb"
      },
      "137": {
        "name": "290_jpg.rf.123735fc7fb3b6da80e7681b6c59e950.jpg",
        "size": 30200,
        "sha256": "0b023b31edf3bc4aef4a319c4ed965c844dd1b0053db284e6a02acf18ca4a7f5"
      },
      "138": {
        "name": "357_jpg.rf.1531058e56ebffc508a3d737c311aa28.jpg",
        "size": 134132,
        "sha256": "76499a0dd1a9a0a998545e1f25577f485d203590b1cee1cf4d4aa8e06386dcd6"
      },
      "139": {
        "name": "031_jpg.rf.14c5c5c6eec9226489054c89c2143a46.jpg",
        "size": 207631,
        "sha256": "a9ef90c791958ea35bdee6f72f77caab8a83e60217a0bcf254e6f037d024b3f3"
      },
      "140": {
        "name": "068_jpg.rf.1678a5487a4bec92c1e716031a018677.jpg",
        "size": 24078,
        "sha256": "3e308bf6631244e07a3625cc8b19a58dcf7597f021b45554a3d864617303a02c"
      },
      "141": {
        "name": "140_jpg.rf.182db45a9ea47fba75132d0079b8b0d1.jpg",
        "size": 50762,
        "sha256": "a7ad48596630693d7aa772597a7f8385cbc4358519bce56ff6cab2bb8a44f667"
      },
      "142": {
        "name": "276_jpg.rf.1406dc1ebccb1bd40510d00627aa5023.jpg",
        "size": 360013,
        "sha256": "075719f795f6ab9491157109887bf3f878ab11c6241871f0039e3b6830901ae6"
      },
      "143": {
        "name": "253_jpg.rf.15bbb787726870a5f91518e2cb6b9e5a.jpg",
        "size": 23733,
        "sha256": "c93b40d55339771cc01f7dc9948d96db2b8b789e2900329d1caa97415e6961a6"
      },
      "144": {
        "name": "218_jpg.rf.1878856fea3358ea251568883faff020.jpg",
        "size": 63285,
        "sha256": "17b588a5607525e3062a2b6b6837cef2f03ac521935b20868449d1733ae7fd33"
      },
      "145": {
        "name": "324_jpg.rf.140a9c6f5be62eab889b92e99eea8ab4.jpg",
        "size": 195004,
        "sha256": "b97dbe64a2ded6196e5905286f684acb78c6f83dcf2a95901891e25ee3526e45"
      },
      "146": {
        "name": "094_jpg.rf.1b5322854f959b2498d8ec21e2cd564b.jpg",
        "size": 173882,
        "sha256": "bef08684707817d0873bdc9d54cb5e645ca9465f5b47752fb44502bbea8c45cb"
      },
      "147": {
        "name": "159_jpg.rf.1c13e0df080ee2cbc6d6a7fc80f28838.jpg",
        "size": 41197,
        "sha256": "f854e6e4c8a05b870841d6ef45826e9f9c4af2f5b470f581334383652017b593"
      },
      "148": {
        "name": "337_jpg.rf.17a47814697cc4034131b3d3a1997e7e.jpg",
        "size": 15583,
        "sha256": "834a98517d515b4cd141feb016f5aa67bb02b9e4907d84f11d10ca155f46a07e"
      },
      "149": {
        "name": "232_jpeg.rf.1b36434da66343c786b138fab98e8f86.jpg",
        "size": 151117,
        "sha256": "1391969a19984d67d742ece22baa6da823c2299475b73cd049b70fe6d885dcdd"
      },
      "150": {
        "name": "076_jpg.rf.1fdf23624feab243966ba786862943a2.jpg",
        "size": 9732,
        "sha256": "a09b24840491469f04114c57b104868a627d99b6f004c3dd4ab2838249d3f62e"
      },
      "151": {
        "name": "126_jpg.rf.1d5e62fb3ec534f0d94ec0735a48d4ae.jpg",
        "size": 53129,
        "sha256": "8f869c8fe9c2cee465eeef8c005017286eddfe6e72760a0cbd362439941f0547"
      },
      "152": {
        "name": "297_jpg.rf.1c88423b76091988be960a3cea0fb159.jpg",
        "size": 93701,
        "sha256": "45a92878dc72532a058f2b79461252dcb9ad27a8b4989e585be35aa514df85e8"
      },
      "153": {
        "name": "356_jpg.rf.1dce4cdb92547c2d0a2ea5c9f0da88cf.jpg",
        "size": 64396,
        "sha256": "7faf6f4dde4aae98ae81347267bd747f6988d6c290b6277a94686fa62b6ab862"
      },
      "154": {
        "name": "186_jpg.rf.206d384d7702d45b1e143dd803c50931.jpg",
        "size": 241607,
        "sha256": "0f3ec0ebd487bb1179a977addf98ecbaa0694ee54cef36a8bcc75a0099ba3f0c"
      },
      "155": {
        "name": "335_jpeg.rf.2018a604fe9af2e499e4df6898308fb1.jpg",
        "size": 45035,
        "sha256": "5208703d2b9624b49420527b7e32625fc9cb971ba94a1731501026f9bcc2d7d3"
      },
      "156": {
        "name": "080_jpg.rf.201742167ff15b0e3520e176e8cc8af2.jpg",
        "size": 50509,
        "sha256": "9572bf71e8e6d71f18a9f81157a7e2940d114fcfa247de7e0522e0bd3f1c0e22"
      },
      "157": {
        "name": "win1_jpeg.rf.21c2226ecebabc8cc838eb39eea45c67.jpg",
        "size": 84439,
        "sha256": "eec6f0d93feefe768cfbce85413eba96ec22c4f2ee8ef60d8c123841f8e4c768"
      },
      "158": {
        "name": "225_jpeg.rf.21fc837cd6f6409b34039c2e08fbd2c1.jpg",
        "size": 170155,
        "sha256": "edcc8337b92d72c5b3ae7661bd3e93450fb42295b0d19097503ab8732a04fe67"
      },
      "159": {
        "name": "006_jpg.rf.247f9ee9fec35c4f647c6c47002472ca.jpg",
        "size": 15917,
        "sha256": "cd848a4d2d238eb2637eb86ad122019e650f7a5e8b205fb3a4c144bf3e487bdf"
      },
      "160": {
        "name": "265_jpg.rf.22abdd8ad23afc39387b16490a7120ae.jpg",
        "size": 11940,
        "sha256": "493a30b8c42b188d3fe4b002cc4e0c0d64a2803ec680f2c0ad3635c4792bf3dd"
      },
      "161": {
        "name": "300_jpg.rf.2322c26d26d5e866b52f8d38a9c33601.jpg",
        "size": 72190,
        "sha256": "d153a3a51b85ab2ae79dabcd2a9679792d1a1ff24e780baf01977267afaa956d"
      },
      "162": {
        "name": "097_jpg.rf.2217b5f12ad3c409fd4d0e753511d331.jpg",
        "size": 576387,
        "sha256": "15ff6d211c98102ba08dd4e49abb21a50da969f1931a5681ba52bafb0b53053a"
      },
      "163": {
        "name": "048_jpg.rf.2117889e3d7dd3c5da295a078d1b8a53.jpg",
        "size": 365438,
        "sha256": "d8c99304af0caa9a58295ae0b50014ce48dbd93d92ee0f1d154cc45a25b6091c"
      },
      "164": {
        "name": "251_jpg.rf.265324224b3460469551c6eef1e7f38d.jpg",
        "size": 12067,
        "sha256": "b53dd941096d93ab02369021dd589d88334f3aa76b611fb01bad799de3af3c8d"
      },
      "165": {
        "name": "176_jpg.rf.25248602e75e68885efa9bdd6c651362.jpg",
        "size": 9307,
        "sha256": "11842632a4184f39e56d32fde6597202eebea76fe973a5d266783ccdc235eded"
      },
      "166": {
        "name": "264_jpg.rf.2390f408297752ab626276646ff98d44.jpg",
        "size": 100447,
        "sha256": "abd15b4cbac985dbd529194f6a4fb497cc6aa57206d97dc79661811bee2181e6"
      },
      "167": {
        "name": "214_jpg.rf.22ea1a2ff8e97e861473dfcaec9206fa.jpg",
        "size": 52111,
        "sha256": "967c3140a6178bfd6c91134a2d60588daeaddc78f05273743e720aafd12b3486"
      },
      "168": {
        "name": "062_jpeg.rf.28c913d0a9e2509741a74fbc45222585.jpg",
        "size": 106660,
        "sha256": "457195632b2daf27c474ae2ca9510b61b3ae999b37a645cca1ea7fb5455dd139"
      },
      "169": {
        "name": "281_jpg.rf.29f358557c410d03d12d8a4293eedbb5.jpg",
        "size": 128297,
        "sha256": "d9b84a6626daff8db165b1f4f5861a6032340fba2e0e4c1b52e2599c32024929"
      },
      "170": {
        "name": "294_jpg.rf.28b731048cbd021b9b443644fc2e5913.jpg",
        "size": 67350,
        "sha256": "e3763fff53be4d50b126fb64d9f66121b7decec9881b77190cfb4030e578cf2d"
      },
      "171": {
        "name": "221_jpeg.rf.25e8cd0bbfbcd0c81b279fb46c65b24c.jpg",
        "size": 256497,
        "sha256": "61e34a56f9358fc6f0fb918e115f86748477ed7df5be32ef127cbaefb73d71b5"
      },
      "172": {
        "name": "328_jpg.rf.2ae0aa92a226e7b78ef249d98f02534d.jpg",
        "size": 89732,
        "sha256": "adf6ad427f3080f1495bbc4d849c9d805219a0850cb832f4079d80d1cb80bbf6"
      },
      "173": {
        "name": "059_jpg.rf.2bc0110e3d4b209590c7ff671b05b7f9.jpg",
        "size": 17618,
        "sha256": "04d8b0532fe2a1350ff9c7da1776bdce348c13da556e5aacb01421df64519f18"
      },
      "174": {
        "name": "305_jpg.rf.2b166f39bcaf8a3b4824caeb3777b2c0.jpg",
        "size": 11967,
        "sha256": "af14c9c4df9a14b8f3ad76940a94e016118cd9f8d0ea324940cfbbb95e09451c"
      },
      "175": {
        "name": "243_jpg.rf.2c0057b195a9c2a713518f160425b9a9.jpg",
        "size": 366437,
        "sha256": "c76a654bf6b9af573f3d2a18d2aff70330c121b15dff3c9950bdb803431a0787"
      },
      "176": {
        "name": "352_jpeg.rf.2b963e8cf2657e0f8a4783190c4b9b02.jpg",
        "size": 53744,
        "sha256": "dbc3fbace7bd72f06bc1cbb9b264b69032ff162a91e38eac1bb8f053e61798a2"
      },
      "177": {
        "name": "146_jpg.rf.2d4321a814a3dc7a2606c6b85acea780.jpg",
        "size": 46970,
        "sha256": "4bdda3f3d739519d75f10febc6671f17cb4066e36070bde3441882cb5441ae25"
      },
      "178": {
        "name": "216_jpg.rf.2cdc8ebac6a124c3e142be2428050fe8.jpg",
        "size": 105052,
        "sha256": "580bb5f745f18603592f18a117bf249a048e9533767f4c413c83dba6ce18c59f"
      },
      "179": {
        "name": "120_jpeg.rf.2983e6613cbc97d400068ef957cffb39.jpg",
        "size": 377175,
        "sha256": "13ce0a194676e1f13ebfeb69aee2f4c2e8464c434efd3a5bf866948796ff3f00"
      },
      "180": {
        "name": "339_jpg.rf.2e206a67a83a0e1cad74079059d49a02.jpg",
        "size": 6837,
        "sha256": "e31b07155f9388cfad247aa0f3d17bb27541dbbcbf00c42f7be73e1caf42a488"
      },
      "181": {
        "name": "156_jpg.rf.2de85139d189523c073a63e5581e1542.jpg",
        "size": 101009,
        "sha256": "a28b92b7ed5cc9478df992f70adb34e67342089f963ceb506b94c83fbf96e049"
      },
      "182": {
        "name": "043_jpg.rf.2d112f99076a3782938b5fe80f2165b3.jpg",
        "size": 54388,
        "sha256": "9df2042c32a5f6b483f723838a9f562df7f5121526a961d2ba7821372d38aa9c"
      },
      "183": {
        "name": "292_jpg.rf.2c527de809d1e7ae925c315227e20e63.jpg",
        "size": 50351,
        "sha256": "4074812c93b54bbd1ebd1d69a0ff591e5f0c40a0663ee38fb0eecc3e6559946b"
      },
      "184": {
        "name": "213_jpg.rf.30abd04e6cb4f9793dc3dc8f857ff590.jpg",
        "size": 22421,
        "sha256": "f970fd22afab0bfc3cbc28c6abc6357f45cb18f41c2a4336d4fdbb4c03e3ee88"
      },
      "185": {
        "name": "051_jpg.rf.2e650923c5253e315e66bea75ff750a5.jpg",
        "size": 39881,
        "sha256": "8cb9fca4a2f00afe31c66f1149ab7953ce88ddaf0bcddbaa4e58086c20910221"
      },
      "186": {
        "name": "211_jpg.rf.332a77b6ff2e00e3e6c39793e8624f9a.jpg",
        "size": 45569,
        "sha256": "92503c51163b76d4df0915b7482d397c8bd3e99ec93ddf4df488b4445adedd40"
      },
      "187": {
        "name": "205_jpg.rf.3126e4e0b73aba931b449e9a857bc66f.jpg",
        "size": 38005,
        "sha256": "8d81463e6d0da73b8786672191d3e86046c8434e3f4fd19b9be30f18767cc693"
      },
      "188": {
        "name": "170_jpg.rf.2ecdf60c2713649168f2b93b8f0b61b1.jpg",
        "size": 10746,
        "sha256": "9dc007967b12185f7330f40c9547749c3beed14a0b227565e7253fd0c3160009"
      },
      "189": {
        "name": "193_jpg.rf.3159f1d136dfef0e23a2071a061ad2a0.jpg",
        "size": 241538,
        "sha256": "8e825f6a11b9ec8663e4c763a923ec0ff999ba921d925e5e626f615ca59b48ee"
      },
      "190": {
        "name": "209_jpg.rf.389af9d2859bd97cd146137ed627a063.jpg",
        "size": 26972,
        "sha256": "b9cb23702e92edf526cf3f5eb902ab5f17c39db18f2c08e139aaad1170638000"
      },
      "191": {
        "name": "150_jpg.rf.37658c9d9d3ee6a619f0fea2f213102d.jpg",
        "size": 38848,
        "sha256": "93529711a5667c328d9664b991d577fd832d4975f460086501c7bb7f7f4bfe60"
      },
      "192": {
        "name": "322_jpg.rf.3e947ec1530cef0ea6a6ad80f092b22b.jpg",
        "size": 15541,
        "sha256": "29664105b0d7dd0c37e9a8ea1df5422ff9af7046c444289caeb92890e892c05e"
      },
      "193": {
        "name": "125_jpg.rf.39b0875a5e3a99aad39635d345510275.jpg",
        "size": 43094,
        "sha256": "71d1421d6f27510cbdf9b0127365c98d1ccb4348774bdff024d4df9c79df2f9f"
      },
      "194": {
        "name": "309_jpg.rf.3c6c5761dfc870aabbe56808f97e2f3b.jpg",
        "size": 9406,
        "sha256": "403c823db42825cfa1bf973b69dcd28a525fc97decf898a28720b11f9afb1a04"
      },
      "195": {
        "name": "229_jpeg.rf.3b8f3680c78988cedbc357e396a829a1.jpg",
        "size": 116083,
        "sha256": "75ca2dd856717b9a47e69c85d17a47428e22caf6cc64cacfa25fcb52aa9ccc2e"
      },
      "196": {
        "name": "210_jpg.rf.35d3906433f43eaa7e93421ad1afe45d.jpg",
        "size": 596443,
        "sha256": "3c73fab2729f321feb0f24cc1c368516416c73d266e4f259dfbcd151f92184b8"
      },
      "197": {
        "name": "115_jpeg.rf.393a911bb5ef38a6656afaf4c70ebe25.jpg",
        "size": 100893,
        "sha256": "62c0f6d99fcd8315063571040f3723bad49d05e1a88a55456d043adb08ea222a"
      },
      "198": {
        "name": "004_jpg.rf.403e439be69fbeb646bf7010aaf52ed0.jpg",
        "size": 96402,
        "sha256": "8d70d94ffbeadb9c4b25d422b37748aa4d05fff4fbc4c7216fd34e3f40a76b49"
      },
      "199": {
        "name": "311_jpg.rf.40219a7bf564a0ffcb28208931e4ca46.jpg",
        "size": 34516,
        "sha256": "44c0993eeedd988ad2aadbf11e388fd1898cc6bbc49bbcfeddcb8e802b9d4533"
      },
      "200": {
        "name": "057_jpg.rf.40c74d4093c78918315644bdfc6f01a0.jpg",
        "size": 46459,
        "sha256": "9618dbacc3b132dc161587d45913e72618ff52a88d3dcb75a9478e94e7a752e4"
      },
      "201": {
        "name": "108_jpeg.rf.3eadd885b7ee1a1c6afc7dd96f0f3d7d.jpg",
        "size": 164356,
        "sha256": "4eb828e6788ea6cc6742baba494938acb8e2eb750684051fd31a902fabc74ad3"
      },
      "202": {
        "name": "230_jpeg.rf.3f5aa3924348907c6f8f5fee45fd8441.jpg",
        "size": 188435,
        "sha256": "5da5097beede52ecdee212236f75ceec299db1a3f4a0731e96d8a63124c49c36"
      },
      "203": {
        "name": "023_jpg.rf.43f5356db3d3cb1eae97dff86afa5143.jpg",
        "size": 94725,
        "sha256": "f68c1ccc351eb2a59819b92842d8ed6996a989c8d3d974ba7053324137b31b61"
      },
      "204": {
        "name": "141_jpg.rf.440a23e3727150e988c5920f17f83275.jpg",
        "size": 36695,
        "sha256": "cb6f1e08a1e3e9f8e467c05cd608b59966042944c4099191c96fdfd55a0270bd"
      },
      "205": {
        "name": "336_jpg.rf.41a95f7e650e89223b1d192e9afb2023.jpg",
        "size": 39280,
        "sha256": "adeb2a56ece28f52d369cb24a29d2cc7a0246df80fb17cfc622594ddb19171dc"
      },
      "206": {
        "name": "079_jpg.rf.4329298260d986a3dab90b274248503e.jpg",
        "size": 115412,
        "sha256": "a3c46dc3f3b0dc4e4d3970fa3d8b47f49bcf1d82a00613c9c84b09c33c7a3dc3"
      },
      "207": {
        "name": "025_jpg.rf.434a3f96b3a4ed4d11810fc3b6f9c8a2.jpg",
        "size": 58172,
        "sha256": "69ad8feccbf14fb2cb88932188466b1510e8f3598d3ab23e6b94b471a6308791"
      },
      "208": {
        "name": "207_jpg.rf.44d0efc2a14f5160284727ed240b19e9.jpg",
        "size": 50141,
        "sha256": "dfd8b256cdcb4f068fbdaa96b6b6b2b6058b411ade6920d803791427ff5cf34f"
      },
      "209": {
        "name": "277_jpg.rf.46bafb496a201bb8405dc3aded3167ac.jpg",
        "size": 205578,
        "sha256": "d1e0d79794196eb52c020b63033adce6784cc8bf81f7584a6f3f5d9c14e86e2d"
      },
      "210": {
        "name": "151_jpg.rf.43a243a416f6cf6b2606472a4ca44a8d.jpg",
        "size": 48422,
        "sha256": "15605363184ab38ccb92aad9f9d11ff8cc8c078d98a97d710f368d8122a38a1d"
      },
      "211": {
        "name": "326_jpg.rf.46112df88ebfd485f060728181016ab2.jpg",
        "size": 79941,
        "sha256": "7cd3f1fe680af7a58364f96ecef956099966962bac59dba0f287659a7008205e"
      },
      "212": {
        "name": "190_jpg.rf.46c6fab84bf1aebd8218b1c64f247180.jpg",
        "size": 465946,
        "sha256": "a1ec01439eec31c9365167decd6053dd45b2ad1179864a2b97d127e9b61d9609"
      },
      "213": {
        "name": "152_jpg.rf.486c8b21f74890a136630e360b581faf.jpg",
        "size": 154825,
        "sha256": "84d6d3925a0540087e5121ed326ceaafe8de3c86c4b7da8720b288c3bd8bcd53"
      },
      "214": {
        "name": "330_jpg.rf.496d505e1a6230d5b3e3a687d9ef28c1.jpg",
        "size": 24824,
        "sha256": "19904c37569f5f9b0d165dbc8d4be154aaca99fb3b146dcf5a07c00a3b1afa79"
      },
      "215": {
        "name": "122_jpg.rf.48e4a86f96bafe38daa403d5b54c0200.jpg",
        "size": 106981,
        "sha256": "7191ecd5f7104226275d89d2594a1a8734d49bed2a8ddd1b648b0f51ec4e79fe"
      },
      "216": {
        "name": "347_jpg.rf.46f23088a17da2239cb4fe9d3184a2e0.jpg",
        "size": 21059,
        "sha256": "4ac8a0122e65ff60e4fb728d79d417a2def2793a935c0d21187c894a7d0a6e1a"
      },
      "217": {
        "name": "163_jpg.rf.4b43ecde18a611f4177d22b1daa43349.jpg",
        "size": 59541,
        "sha256": "a0a4f28a1622d527ccc5540b13339fe1987907941428cf038804fbf884bfb236"
      },
      "218": {
        "name": "278_jpg.rf.49d7819f13c6a174d6be0bc0597ad264.jpg",
        "size": 236134,
        "sha256": "16064d26b1a68db375b771035f23ee21204a2b0a58ab65506de8e7770b87fa27"
      },
      "219": {
        "name": "003_jpg.rf.4bbaea88cef2fe3ec5e2f7d0af935f6a.jpg",
        "size": 84712,
        "sha256": "f37d4df618344686f13303bfd8e01a9099eb979857304d8a04327730c54bea00"
      },
      "220": {
        "name": "142_jpg.rf.485f78eebb7e925eca4b9283de9c005d.jpg",
        "size": 96087,
        "sha256": "3ed2b1926d0e8ca0ac2a1e2e917a88d347cc4d448c82cffe8a63a2838b524de6"
      },
      "221": {
        "name": "248_jpg.rf.4ef5a908f8e38dcc27d9331bed98127e.jpg",
        "size": 49605,
        "sha256": "12d1347dd584b43aa59a2bc54d219d3cf4e48b19498865b10ba87a022e6f9884"
      },
      "222": {
        "name": "143_jpg.rf.4f7c4faefb6615708879c3913200b2ff.jpg",
        "size": 71124,
        "sha256": "c2db6ab288beec8433ba0a845f7cce195ef0fef15ec20656e6289897dee635df"
      },
      "223": {
        "name": "020_jpg.rf.509b886757c51ba91a68a909648f4cd7.jpg",
        "size": 12498,
        "sha256": "60ae8d4bcbb41a5f9b232a565c2834e26851f586b8c013ebf931a8a613a62d14"
      },
      "224": {
        "name": "134_jpg.rf.4c16979fb1e25c9698dbf54cdeae3632.jpg",
        "size": 57195,
        "sha256": "18bab5c3fcb5afb50cb9f7249b1c94d3399319b4517d0c8a53a16245b2c9d0ef"
      },
      "225": {
        "name": "257_jpg.rf.513a779cc7d78c42f628a626bd1688e0.jpg",
        "size": 81122,
        "sha256": "a5d8cbd0abdf5d93ba7430ff4b450ff148cc47e18df1101d47fbf4be71f0f4c4"
      },
      "226": {
        "name": "035_jpg.rf.515759ff38c09899254b0d8e610aec71.jpg",
        "size": 315030,
        "sha256": "730b0867933c1edc74724a173a3c2feffcce5474f2749a010dd605acf55b3d46"
      },
      "227": {
        "name": "259_jpg.rf.5174f8505f432841f9710d4f6c088de2.jpg",
        "size": 16241,
        "sha256": "1d2177223f23d0b0f84cb6d2441f92308c2f256b171cb5aff854ee895173ebe6"
      },
      "228": {
        "name": "282_jpg.rf.49951a5ae791c5856e370f8c86d18e71.jpg",
        "size": 165690,
        "sha256": "738091ff765ebd164e7c316ac6253b33a059725e2dd48dbe85b09c60240799ec"
      },
      "229": {
        "name": "246_jpg.rf.52385a3d9a635df32c8e67d47c9a3492.jpg",
        "size": 54299,
        "sha256": "4abe03fee1f09e0e1aaf6474c0f646ab2eef10c16a75ce6e62bc35e4b2b73758"
      },
      "230": {
        "name": "040_jpeg.rf.52bbc9a97c0ec81fe060fb60a5ca3a23.jpg",
        "size": 51069,
        "sha256": "44e742144edf95d9b3e3d87247586670606de09fbf0ac9ba6b4f98b872675ccb"
      },
      "231": {
        "name": "231_jpeg.rf.528c38fb7cdefd0ece4e04d0c23be53f.jpg",
        "size": 182571,
        "sha256": "16aab2abb32bd3b7a5ac0b507d7d7fb2dcb54ea6e11fef16e8bc298b2ca2cb22"
      },
      "232": {
        "name": "313_png.rf.5385bf78a871538ee1e921fee5ca332e.jpg",
        "size": 43432,
        "sha256": "a28d8c246c0b57eb8125dfc8c8e1efac96e14325b49473717882d93013c11c1d"
      },
      "233": {
        "name": "348_jpg.rf.555cde5895ce594006a4a1bafcff88c3.jpg",
        "size": 24968,
        "sha256": "1c9fc9346651bc2995d2e3413c3f98fec2c985e67723e5ac67794949c2c226c9"
      },
      "234": {
        "name": "360_jpg.rf.55d6a091eb444f2d10038ee5894f449d.jpg",
        "size": 70828,
        "sha256": "9a656f6b82a61c720279a6d0af3f2666e96316539916523337e5355de78f4b35"
      },
      "235": {
        "name": "001_jpg.rf.4e68c60b740dfb573ae743bc71df83e2.jpg",
        "size": 108757,
        "sha256": "d43edf07b2a40b5860f4dfbd083999cf0bcb7e2a2a552192a8cf5b0de676d68f"
      },
      "236": {
        "name": "272_jpg.rf.541b8165e8b54faaf05ddea49ea51f41.jpg",
        "size": 9493,
        "sha256": "d4b9d27630335317cd248bd68ad0683749533598ef6121573dfbb93099cc45b8"
      },
      "237": {
        "name": "340_jpg.rf.57f584f45f5211695195cc232ff57f39.jpg",
        "size": 6685,
        "sha256": "b8f58f0c4d64944da9cf353c054f46ef4b21093df32fcf53db7d6a9a5490fb58"
      },
      "238": {
        "name": "219_jpg.rf.536c15eddf0d4d4dbfe8cec8a9c26a0f.jpg",
        "size": 42336,
        "sha256": "d3637363ff50e7961a6fe830e2e434cd24f567846f0f4a4352d965afd9fbc8c8"
      },
      "239": {
        "name": "052_jpg.rf.5a38013f8120b55880ee9b853810dcbb.jpg",
        "size": 7049,
        "sha256": "36b6ab44bc86aa8fcdc61f4578c6a3b71e5b265cfdeb70eb22926fcf4729a39f"
      },
      "240": {
        "name": "197_jpg.rf.5658a79fe73ce8934fa0680d4dfb5fe9.jpg",
        "size": 65294,
        "sha256": "2988087f9c03ec323e3b52344da7f5416494af405dd297f997756a347538d2d9"
      },
      "241": {
        "name": "027_jpg.rf.5a7785ba04a67b62bef57017ba5d0a82.jpg",
        "size": 53142,
        "sha256": "481c46be7e1c4a6941735393387f1fbd8837df34ed8a88ab3232f427cb83c5a3"
      },
      "242": {
        "name": "128_jpg.rf.5b1b9910cdc2c928fdeff9ffa7a7093e.jpg",
        "size": 183980,
        "sha256": "e44d261f0c604afd5ae21a7be4fef71c57f78f0bc1ff374e66825218a3bdb4fe"
      },
      "243": {
        "name": "117_jpeg.rf.5af5cba0841fc9ef6538bfbfcadc3d9f.jpg",
        "size": 147826,
        "sha256": "bfbb3d8b0f5f5cc9104f5a650f3d4a14d3f5866559d4b4115ae12002ed03396a"
      },
      "244": {
        "name": "198_jpg.rf.5cb1f4c2d3a3015d1df45bea1735ec25.jpg",
        "size": 25276,
        "sha256": "4710c11ce1740a0278ced1e231d6ffe4436ce735cf6eca3892f4c817464ec50f"
      },
      "245": {
        "name": "284_jpg.rf.5a9a02757830a826dcb823f911b44a3a.jpg",
        "size": 15242,
        "sha256": "ddb18e0de5c6fec5be75ffb97459b01d595ac81eebf5d07eb622f359d19372eb"
      },
      "246": {
        "name": "269_jpg.rf.59ca314d40ce478e6a2afab2c40ef0f2.jpg",
        "size": 9755,
        "sha256": "903b7d75df08794b7fe1c48ca1ed67b2dd9a370a42c5de1bf67136065506272d"
      },
      "247": {
        "name": "306_jpeg.rf.5dda5944b3424b5121b9ed41de557d09.jpg",
        "size": 105533,
        "sha256": "29a288134f3d4442e7b44b28b4b48ff0e71eb76e05164acba19f8a5d5505c95f"
      },
      "248": {
        "name": "291_jpg.rf.5e54f7f62224de8a505632434e6dbc36.jpg",
        "size": 112626,
        "sha256": "33b155c1eb73a2f2062d613a4c7a42e9c5e11289492fc15e07dbd28b30829c2c"
      },
      "249": {
        "name": "275_jpg.rf.5e6e9c48a29fed6d3e74688335e506a5.jpg",
        "size": 189893,
        "sha256": "dba349aa665cb42972522d821a71af9646c3142f8ce304a6b3cba3db5e77f383"
      },
      "250": {
        "name": "301_jpg.rf.5d1b53fbbc08a3b7c4fba873969d2a70.jpg",
        "size": 41283,
        "sha256": "d3cdb822165a8bfbb8102e7cb24e61f4510563ff4ec4f20c8d5dab394c00a4ed"
      },
      "251": {
        "name": "087_jpg.rf.5d7b4211c508d9e511bfd36818b59d09.jpg",
        "size": 73792,
        "sha256": "1fb42da30cfd50012a4283bb160c8d80bc8eba28dc803c8cf83ca1cc263e0611"
      },
      "252": {
        "name": "018_jpg.rf.61812d9c1730ecf5526a7e61a04923d7.jpg",
        "size": 27534,
        "sha256": "451c2ec68a85f6f5442d398082f8d50720f3befdb6b3e10e66341f0bd74f1997"
      },
      "253": {
        "name": "203_jpg.rf.5b6f75ffed99b20675a63e38245d8056.jpg",
        "size": 37295,
        "sha256": "ce402258ee2c7d1c8b6f35ce781f8918f074fef61e87b9957cd4a3f853045e83"
      },
      "254": {
        "name": "354_jpg.rf.5f15f326ac41cdceb686862369e1a910.jpg",
        "size": 13456,
        "sha256": "d8fd220f892326a8bd87bf25995ff7380d33d4921b23defe97fe9b7027c590b9"
      },
      "255": {
        "name": "012_jpg.rf.5b50de83e972d320dc959cdac9eae8fe.jpg",
        "size": 94802,
        "sha256": "dc081e83bc629f0f193ac26adcb769cac9e44e7dc54810a9a0fab6a9118277b8"
      },
      "256": {
        "name": "342_jpg.rf.5c5960c7886d61932e5fe1d2dc99e416.jpg",
        "size": 27233,
        "sha256": "2aa431c35db278bb6fe18e9c1b9b6f7d7475c4e43601bcbce6c0c7445074054e"
      },
      "257": {
        "name": "192_jpeg.rf.5f6a5391b12e4a8b877de29edea63958.jpg",
        "size": 79337,
        "sha256": "7151ecaf256a4960af255a2179996bcff969018c16d482a3f816702875f83f97"
      },
      "258": {
        "name": "008_jpg.rf.5e7223201a1d33c45b18fc3feed7cc5d.jpg",
        "size": 327050,
        "sha256": "ce78051a5789327a7c4d529410fbdb2ea215564be8eff882d1004341ef587eaf"
      },
      "259": {
        "name": "088_jpg.rf.5fbde9338d6f149f72edd404fc8a5678.jpg",
        "size": 64087,
        "sha256": "0d8a27ea30f338a984c3eec12b89014a726412cdde12237c99d5ae807f741cc9"
      },
      "260": {
        "name": "296_jpg.rf.5fe00164d30aaa4131bfb883afea4e56.jpg",
        "size": 142078,
        "sha256": "21973bc4044a48333a7d8ab9786af1dd80ef189b0c54a520de587307cf462b71"
      },
      "261": {
        "name": "160_jpg.rf.5f84749a4934c0237365262b89f702c8.jpg",
        "size": 129921,
        "sha256": "e84e70d707c4483d61d49099aee55c76dc341d3d8655b48a8a3040b00fc46a60"
      },
      "262": {
        "name": "283_jpg.rf.6388b3ac28b5c18f51143a30fd45f8d8.jpg",
        "size": 69535,
        "sha256": "99c9c412cc7fbddd3238801196629f1b68cf2e82677ac5f6dca15519f2459ca3"
      },
      "263": {
        "name": "168_jpg.rf.6296297ad6d10c869d9f8e9793857804.jpg",
        "size": 56626,
        "sha256": "6793c5a087b6f0c86e2194013522d77374485eae45588c1ec61a8b3a67bdcc15"
      },
      "264": {
        "name": "331_jpg.rf.65cd1b071063dc127ffc474e82ed393e.jpg",
        "size": 18485,
        "sha256": "4d2e6470ef7990601685a0511aa7e1ab44a271ae9a532b9356ce60b44cdd2014"
      },
      "265": {
        "name": "244_jpg.rf.62c6e07c5a835bad87869ffc403cbbf0.jpg",
        "size": 6378,
        "sha256": "27b41c5aab2a3f302b1b0bb26b7a1e0e478a3295a99dc1435a6991deadebaaeb"
      },
      "266": {
        "name": "196_jpg.rf.65e32223e150f68dd08b63bf6ab905ee.jpg",
        "size": 7355,
        "sha256": "507eed8e3a8c4d29db89a680aab159670585df2ea6a1a5234e2a2974c8b6a07f"
      },
      "267": {
        "name": "316_jpg.rf.66913148c24e3a8477d24fc3ddc89070.jpg",
        "size": 49317,
        "sha256": "f30e151bc8891a54164f554f52e7a66c0f90b8f3d10488f2d0cd127539e73a08"
      },
      "268": {
        "name": "131_jpeg.rf.67970bd7801f801eeb6aeec6ea0e1921.jpg",
        "size": 78405,
        "sha256": "493765c4a9693a6a782f53b551d96e21bb88c9f79e29b8675827fb83019f06b1"
      },
      "269": {
        "name": "086_jpg.rf.661451f60f8efea2a3d1e05042ea5836.jpg",
        "size": 50996,
        "sha256": "62f8eba805f860b617f60de2f8bb932d8dcc4075e87d002b905c67788b6be2d9"
      },
      "270": {
        "name": "179_jpeg.rf.694808959f3d45495fffad6e8546e71a.jpg",
        "size": 343035,
        "sha256": "8ba0ae6133bc2e8bca0f486be181d99c8bca79799bc1a567b14f95fae602055b"
      },
      "271": {
        "name": "158_jpg.rf.6614d108aa9c89e2674a5aef1e379be0.jpg",
        "size": 65524,
        "sha256": "e7bde863270dafcf749f6e27dfd39828cfeb5ad11119e03e0266d8407d752a30"
      },
      "272": {
        "name": "092_jpg.rf.68633eaece031fb1fa9e66365b29dfa8.jpg",
        "size": 29359,
        "sha256": "d673fd4ba1e30518bd38480b31ed245296d323fa77e21b5bc08b0e39cc991cd2"
      },
      "273": {
        "name": "350_png.rf.67b4e18af4532f8ebce9c120ff8ce742.jpg",
        "size": 51980,
        "sha256": "db96afc3deb00d35cc126dfb673292eb5fe4fc894bdcbf9aeceb83b003b2ed2c"
      },
      "274": {
        "name": "223_jpeg.rf.6ac6c269c81cace2ae742aaff836911e.jpg",
        "size": 98656,
        "sha256": "3c2d0cfc358e4df7784cbba8c07e46bef3954fd3cd60085bbc4652490975517c"
      },
      "275": {
        "name": "165_jpg.rf.6ecd4f4763b49eb4863720c766689c3f.jpg",
        "size": 200210,
        "sha256": "e5752a163a35f6e72b8ee54a54c049d71f59e3794b52bd430c59a5379c4684cc"
      },
      "276": {
        "name": "045_jpg.rf.71a3a929fa52bed195bff06a409b9dad.jpg",
        "size": 51322,
        "sha256": "733755fd41c195276f881f4af6112dd59eb3e7a0ccac7f62704dfb48e1dfaee6"
      },
      "277": {
        "name": "137_jpg.rf.7040268b9f59dd57b860dd1c1be717d9.jpg",
        "size": 64755,
        "sha256": "590d2c366eeb427b9b76ebdc1c99de14806b24d54a9949018c66b14bcd8552c1"
      },
      "278": {
        "name": "167_jpg.rf.6fe50fc05f7c4623db5ed2ad77da52ca.jpg",
        "size": 24620,
        "sha256": "aac0e7a5ac2a0df9e82ac4ded76fe411a14d0f3d0da3fc7cb5298468ec6083e5"
      },
      "279": {
        "name": "208_jpg.rf.703d4b935b122bd8aa43fec3ea3620e1.jpg",
        "size": 9650,
        "sha256": "27929afdf25da456cbe5469f385d79cb4a6e9fd7c0559f50170bebaf4f6ea13e"
      },
      "280": {
        "name": "011_jpg.rf.6c91962ddfb5f99378f6af5c7aac038b.jpg",
        "size": 12539,
        "sha256": "866bdca2f5f6884c6604a3595007d87adb05d49191eb220c6febde6db0149b14"
      },
      "281": {
        "name": "288_jpg.rf.7951d880ad8c9ae0178cf0af79ee3bec.jpg",
        "size": 16643,
        "sha256": "e2210fbc4189a07f97d620e94bf8dd23ab49c9c06f3d190c1744fc499b2abee1"
      },
      "282": {
        "name": "303_jpg.rf.715e4167b3ef2ca983dd18a9eb90baea.jpg",
        "size": 83416,
        "sha256": "a074af5735f30fb927bec842ade49573901004d5473cc8a0bc135ab47bb3a32c"
      },
      "283": {
        "name": "314_jpg.rf.7477f21fccb8b863de416b7e27bff14f.jpg",
        "size": 77809,
        "sha256": "ff75d9c6c6d7f2e306b92dff9586b682b023e2e7a2e183c089e94321169356a8"
      },
      "284": {
        "name": "310_jpg.rf.6db552cfe7e16e93187e0ff14664fa46.jpg",
        "size": 41195,
        "sha256": "c49eade5f09155d291b1428779ccc140065c9865265777112cef3a2393bad5b3"
      },
      "285": {
        "name": "090_jpg.rf.774fe956460d7f0180c47f1c286fed7b.jpg",
        "size": 60706,
        "sha256": "e01c0b299ab6d86e8cb821a21194568ad591a39dbb4bb06f2ff54b4ad1b807ac"
      },
      "286": {
        "name": "127_jpg.rf.79b578bb2df77b8139f79998f2fac042.jpg",
        "size": 59840,
        "sha256": "131cd5fd1b5641852ff61b73968ed1e308f1fa0a1c98cd86e58272d0e76696c5"
      },
      "287": {
        "name": "103_jpeg.rf.7a6ce7938c6be91f46b29062144e5e32.jpg",
        "size": 187656,
        "sha256": "01e8736a0d2c0a05f0162fdc6945383d002fba7c5f850bfe1d17ab0855e90fc3"
      },
      "288": {
        "name": "321_jpg.rf.7cfb3e3b83506d4000b779706734f9dc.jpg",
        "size": 134823,
        "sha256": "30254f809dc52e8fb462daad5216fe478632955d2968e1b3bc7e81c02ecb9951"
      },
      "289": {
        "name": "104_jpeg.rf.7b5a1c4cac5823496b5deff4987edd62.jpg",
        "size": 135989,
        "sha256": "e8ccc4439957693c7a90ce15a01202deb2f6f79a29670e06d135d2d05aaecec7"
      },
      "290": {
        "name": "240_jpeg.rf.7cf2d56169333e866335c9b654918191.jpg",
        "size": 128187,
        "sha256": "ce5091478b3faac7733c293691ff43ae151fd07554d8aa3c91a4ac7a472516ed"
      },
      "291": {
        "name": "157_jpg.rf.7aa9fc4c3df03efad9f352d2544ab160.jpg",
        "size": 7053,
        "sha256": "b5a9f8773ed6ab83cc7409f6db69348538c45ba4cb205f2b0190a095b69aad4a"
      },
      "292": {
        "name": "238_jpeg.rf.7ea91f0efc067736f5d259c26bbe6ac4.jpg",
        "size": 159820,
        "sha256": "b9e8e254d838f2c0280ccab1c77c4ebd6f2abbbf51c0f08a6b13dcec8556e278"
      },
      "293": {
        "name": "100_jpg.rf.79ff2b0c548d1ebfffe1de1419c7e2d9.jpg",
        "size": 39487,
        "sha256": "4753df595a5ce7685c67bdb76c367970325416029baaf35cfc7cf7af039114e0"
      },
      "294": {
        "name": "172_jpg.rf.7c4f790fa9f7daf247a554e3a8ee2b19.jpg",
        "size": 9282,
        "sha256": "9ed28be6b6223cd517566aed9fd57a4bfd758d9705b097c5e2a67557b35ef11a"
      },
      "295": {
        "name": "233_jpeg.rf.7f67983d3aa5d7ceb9ed8453d236ff7c.jpg",
        "size": 198336,
        "sha256": "4411b581ac5c23779cad7f2b4ce1f1e5235184c9c8db699f18e67e2414e47ecc"
      },
      "296": {
        "name": "077_jpg.rf.7edcae38e53292f031a38649cc68f572.jpg",
        "size": 182264,
        "sha256": "680296be16ec42d14e687849b200cccfe918619d55dedde7d9167ab2714ceba1"
      },
      "297": {
        "name": "318_jpg.rf.7f568df7a8865a5ff1d348f6722f752e.jpg",
        "size": 8927,
        "sha256": "5633005eb0bc0aabfca00bc1588a2ef6b2f67d56326aad433d3742802c10d03e"
      },
      "298": {
        "name": "058_jpg.rf.7f6102989d84b25dc3d3d2ce59fe353e.jpg",
        "size": 13298,
        "sha256": "d651055a134810923cc60de8c375379d954841f2f97d4ef83077432819d2eb4c"
      },
      "299": {
        "name": "166_jpg.rf.80a2c6d1bec91c03f0ece142ed33acc8.jpg",
        "size": 9449,
        "sha256": "f8b36f5b553ecacce06269012ee44cdeb1c15d92f68716cb23b1daa85fc236da"
      },
      "300": {
        "name": "174_jpeg.rf.808b85da2617d5c01085d0b084787b20.jpg",
        "size": 93278,
        "sha256": "10a038e31c2a8669f17028e8bb41da6127c53f01e3c78810d22015d4ac4b31c8"
      },
      "301": {
        "name": "028_jpg.rf.7f9f775b4ad289541b9863312e1445af.jpg",
        "size": 87133,
        "sha256": "af9c7c6e702474260bedb59dd986e5a8faa72040ce09f48ede9d401fe9bb9a47"
      },
      "302": {
        "name": "191_jpg.rf.821647dfc1b7a827ff9c36afaa792858.jpg",
        "size": 89688,
        "sha256": "7398b7644446839f895d245ee5a172d2789aabea23342ac9a1fa01bf0c64bd80"
      },
      "303": {
        "name": "067_jpg.rf.801eea7616802e3c228071a814ebaebe.jpg",
        "size": 40849,
        "sha256": "5adcddee71840a12d1a07c48e8fc172e46dcd116fbdca0c3c4f129016c67c71b"
      },
      "304": {
        "name": "299_jpg.rf.86f914bed603c3502b55f1b1e0d00188.jpg",
        "size": 9377,
        "sha256": "e140723495d9a8eec8e981030a9667dd4cf58826fea1bab888ddcd5de58e23d4"
      },
      "305": {
        "name": "317_jpg.rf.83e987197fac0c357934ec7d0db1a652.jpg",
        "size": 103129,
        "sha256": "f099cffbcceceee0b03ad7c9fd0a85d8d5e07d284b36d725fc5aee2b0a7bca4a"
      },
      "306": {
        "name": "181_jpg.rf.8172f7d531eace7043492e0850f07a59.jpg",
        "size": 7734,
        "sha256": "4a21c2c8e44ed763e159e558e558ab8c3f4fb72fc6de7c530136894dac648f91"
      },
      "307": {
        "name": "065_jpg.rf.853f146f381e056b2578307479d34845.jpg",
        "size": 4738,
        "sha256": "c0d286ae0e960b0d0c16039e7e356afa791b83186afb18a0d0d096a8a550f939"
      },
      "308": {
        "name": "153_jpg.rf.890be922a1089c5fa8c2a5e828cc8052.jpg",
        "size": 221056,
        "sha256": "f9c483f13e834e0f7aa46a3adcbc5a95c3981652a37540663cee33bf3904aa65"
      },
      "309": {
        "name": "034_jpg.rf.8a6cc0965983b2dedf8a630351eb0beb.jpg",
        "size": 24079,
        "sha256": "60c13e79cc5e85fec8ee53ed86b24743edb8041885ed21c8fb9bd5e75cc388fc"
      },
      "310": {
        "name": "178_jpg.rf.8045fec7ac97c5c521195e58039ab4b4.jpg",
        "size": 15429,
        "sha256": "af077c5d5e63d8bb48cb92f28c0ed7b49b3b727f6b7e187aedb99f1eb8d98d1e"
      },
      "311": {
        "name": "047_jpeg.rf.8b6e60b8e548f137d7be384087ccf2aa.jpg",
        "size": 52825,
        "sha256": "4a3fe3c39eb86e974603b50c2e21da66fcae6ba1bf2348b43688c378419d3cab"
      },
      "312": {
        "name": "312_jpg.rf.82d4cfc306de6baf27deba7f89adffd0.jpg",
        "size": 40886,
        "sha256": "8ca0f896227dea00bb7b8b8042a4873d97c0c3697b2f8d766675f0d53debc5ba"
      },
      "313": {
        "name": "250_jpg.rf.8dab78895fb6c174ea7ea8910b3de0b8.jpg",
        "size": 25808,
        "sha256": "0bc9d7e155b4d7c7de2a33e870ac3c8df5bea0bd24d8ef22ea2bb9fed9b6a12a"
      },
      "314": {
        "name": "273_jpg.rf.8c9f9e27c7f22bb51dafe9e61ef3e033.jpg",
        "size": 198847,
        "sha256": "6beb1a4639b282413c45cf283f9c70081289315a655dd123659fcab0612c9037"
      },
      "315": {
        "name": "063_jpg.rf.8c066441cd3b303c3c74799353305ac4.jpg",
        "size": 54227,
        "sha256": "ff6ff04b5af702cbcee708abd81f78d2da81efcd400e368c3fd037a374ba03f9"
      },
      "316": {
        "name": "279_jpg.rf.864340f9423635a47007bb2b39b10c48.jpg",
        "size": 1589633,
        "sha256": "1855e19c0f7d3ca3e9aa77f91e3e7be88cf6f4421bb1790b2084a6f676251810"
      },
      "317": {
        "name": "270_jpg.rf.8b904f90fb4dd7f7208218a3b236692b.jpg",
        "size": 8475,
        "sha256": "0cb3db540961d5e9b5920f344f3bd767f741d30d057ed019005a2fd21c9dd947"
      },
      "318": {
        "name": "206_jpg.rf.8e4cf9e27323d69e54ea87c81711f61f.jpg",
        "size": 10002,
        "sha256": "ddda04adaa76ff982e09370b47ea00ad14416dde55afb17abe1c186844d16b46"
      },
      "319": {
        "name": "260_jpg.rf.92840c9cc3b4bcf671a64cc3e97c374f.jpg",
        "size": 14755,
        "sha256": "dac6d5d5c5a3dbe996cb061037ef58e6b1931cd0a608eca5bdc941d9e353611c"
      },
      "320": {
        "name": "085_jpg.rf.8962de79606241f149d002cc23079f36.jpg",
        "size": 45635,
        "sha256": "7a0130aaa0195a3fdecdda612f574dcabdcfcb0fcfc4c15c7d734252b9e081f5"
      },
      "321": {
        "name": "002_jpg.rf.941b44d4633817fd833de04127e2f3a3.jpg",
        "size": 72623,
        "sha256": "656954ca8d7f0348997c64e4b3bf2bdcf7523e3b7e16647d463fb4e42ff5356e"
      },
      "322": {
        "name": "263_jpg.rf.914959626b0d0cc2cd1668abdff52728.jpg",
        "size": 165030,
        "sha256": "a97e34985362fe98b26f4a644b072a279e3662a495d2bc0c4bf3d7b1c3fedb88"
      },
      "323": {
        "name": "111_jpeg.rf.8d6a8af4e1b93fc0cc0403157fae3947.jpg",
        "size": 269561,
        "sha256": "153d494fffc4dc22c77f7723c2ab65a6be6337dceb757636e60b1e8f6e5aa23e"
      },
      "324": {
        "name": "096_jpg.rf.8ffe9c48cf187343ebb846a4d983b6ca.jpg",
        "size": 34076,
        "sha256": "38e7409b520a32699e09d38d930a1feb9c89ce2cda074f5ad1d0ce8189c80d61"
      },
      "325": {
        "name": "341_jpg.rf.927800c832274a8ce8a00e1f8d3a371d.jpg",
        "size": 6978,
        "sha256": "29ce67b6ac7326a791c5835e2e5fdf17e7a7dd0b201e2b2653aa6bd6131bec5b"
      },
      "326": {
        "name": "173_jpg.rf.8e2e8a811bd4a9b2ac10a2f2f4ca9d54.jpg",
        "size": 509618,
        "sha256": "1168998330d30bf72f90bfd6b69dc23dc6c54392ee4d6c84ccd3d392c44cad60"
      },
      "327": {
        "name": "044_jpeg.rf.93228b041d9dea7fc4bb5703f293d5d6.jpg",
        "size": 35500,
        "sha256": "696b76f1de1d690b4a354e7e312057a30020b14a3498334decee70280e31d9ff"
      },
      "328": {
        "name": "199_jpg.rf.94b10017048d2eb0c71e21f25402ea79.jpg",
        "size": 57439,
        "sha256": "279a0d4dd80c16310e42616685e818ec4f738cedb7145911b6951f49120b5b3f"
      },
      "329": {
        "name": "185_jpg.rf.94c45a11c1e344b02a930226a545a161.jpg",
        "size": 17620,
        "sha256": "bac906ede2861a5b894196bb83d5381ad84a474e090ddcafc1fa42d95937077d"
      },
      "330": {
        "name": "327_jpg.rf.94446c90c6a7b84c462b14d29be20f7a.jpg",
        "size": 95306,
        "sha256": "c1b1153b723f10f3cb7d583a7d6c99a1450b92e79d66464371405dd75fae30b8"
      },
      "331": {
        "name": "293_jpg.rf.95b96295149575dbd8716d9b4a1e480d.jpg",
        "size": 11787,
        "sha256": "35adac463bf2f03e4275495f28db74e4593c06a0f0c38bed8f9de4ad7427aa7d"
      },
      "332": {
        "name": "106_jpeg.rf.93eeddd014f0394c93292466b9705227.jpg",
        "size": 149480,
        "sha256": "255bc264391faa8dc4f8e597f35c9efb8a84456fd27af3bf0a1562c7aff93bbd"
      },
      "333": {
        "name": "358_jpeg.rf.95256c05cb5675c037e0debc4277f9c7.jpg",
        "size": 107963,
        "sha256": "abe37320251b6f7f1feca0b5ea1f3cc954d61126e017203740bcedc27f52d4e4"
      },
      "334": {
        "name": "332_jpg.rf.961843a1aaa304a4b3592f4aa1c3693d.jpg",
        "size": 29410,
        "sha256": "cc298a5b1e321d06edd4a36dcbecc5c3b552a908af167a2f6b6328390a41a317"
      },
      "335": {
        "name": "308_jpg.rf.9638cabcf33b1d3c3e049887c35ac39c.jpg",
        "size": 153273,
        "sha256": "a2416832441be23d6b5c79ae1ecf8fa80c90992f52cf14f337fc9b61529d4301"
      },
      "336": {
        "name": "124_jpg.rf.98a3036faf33568c53220c6510c76bae.jpg",
        "size": 203858,
        "sha256": "6abb6c546db05ddd1cb4d7f44f7c58230a032fcefb1ea44e692d7f7c57c6b99f"
      },
      "337": {
        "name": "239_jpeg.rf.9a7e24666b9e7c49e5acc65ee08cb7aa.jpg",
        "size": 233363,
        "sha256": "79f3048fd7917f1d529b8ce1df76b459c76bad2929040cc164885522851b7558"
      },
      "338": {
        "name": "139_jpg.rf.980d8cec61599153b35ceb4a92cb8885.jpg",
        "size": 175966,
        "sha256": "f7ddbd47e7031c4622f3596bc40b668d20d9bb2029ac53a37316d63464b70b83"
      },
      "339": {
        "name": "029_jpg.rf.977dc4ca58cabd4516ca3922e538697b.jpg",
        "size": 231321,
        "sha256": "89c1fe3432b045b3a3d7bb761d1611359b66b4c363bfbf37dfca62d318c2f5e0"
      },
      "340": {
        "name": "164_jpg.rf.9e1b520c14df0d6f9d07b9e735833cbb.jpg",
        "size": 55828,
        "sha256": "5766f406490c92763a4ce92d6a40db8d829974447ce7191c938119b96bf5df74"
      },
      "341": {
        "name": "082_jpg.rf.a4bbd401760c7f0fc64b57daa996fc7b.jpg",
        "size": 134931,
        "sha256": "c9763acf9ca0acea734954e29284373d72d3364cd8bd9ce2739b3a54896397ef"
      },
      "342": {
        "name": "138_jpg.rf.9b1862aa3f64ff40d9decf49fbb8b38b.jpg",
        "size": 68665,
        "sha256": "54f562da404053f1e27b26f2387d7e1a4354c8bec8845d2e0a8647ac78c05828"
      },
      "343": {
        "name": "069_jpg.rf.a18b30620c36ab71816c6884820e094c.jpg",
        "size": 103448,
        "sha256": "b79df95fb97edaa99f64e23d97aea19175edfe6972ce6960737991131a1a9d75"
      },
      "344": {
        "name": "055_jpg.rf.9b7769083bb6208aafa80ddc381493e1.jpg",
        "size": 40585,
        "sha256": "768ad578c54a8c313785d68569e407d4cd32457be9ae1d22ccc8cd05a8ba43b4"
      },
      "345": {
        "name": "064_jpg.rf.a107042b5b32970d545c3e4d5162be21.jpg",
        "size": 9837,
        "sha256": "0264f46f4cb5babfd7f52fd4fa85ec29bf252e8eb7cf6f05e6c375e842985293"
      },
      "346": {
        "name": "071_jpg.rf.9bad67b9fc33037718f6a8b4ce09569f.jpg",
        "size": 58441,
        "sha256": "0488e3aee378e2058bcf25983e6ab83ec258940de74cd85070c71ac0ac4a9eea"
      },
      "347": {
        "name": "123_jpg.rf.a66e70f1b666940732c031a131d86ef6.jpg",
        "size": 74426,
        "sha256": "fa12e5b15d520c49d5fa564fdbb3fa99bfeb941f0503beb4c68acc3ed289f0ed"
      },
      "348": {
        "name": "227_jpeg.rf.a654dd3ada8fbf4517fcde606e725e69.jpg",
        "size": 99860,
        "sha256": "6545d3f0277174f535a0dcc3375cddc12e7873d60b66b32cb85b5f7c0b903b25"
      },
      "349": {
        "name": "081_jpg.rf.9e1585cdd2c1b8560845f20fc8661055.jpg",
        "size": 55921,
        "sha256": "2cfce687fa28e5971d05b4c8c2814b750104c3c164adf1aa299cedc196656df1"
      },
      "350": {
        "name": "228_jpeg.rf.a69c6ab1c0970f59dab940abf72ec7fa.jpg",
        "size": 130346,
        "sha256": "e8daa8bace95807affd2dd7696c065e186894577e19ef822ab7332fef2a4bbfa"
      },
      "351": {
        "name": "107_jpeg.rf.ac6fb33b78015d8b5025c3b0fdac9a38.jpg",
        "size": 204839,
        "sha256": "ba6bd4f49d8998fcc168712b4fbf8b8175b280d2c308d138040a7d6d65c1b8cd"
      },
      "352": {
        "name": "256_jpg.rf.a4d247b6e388b9d89f98442f9c7d70c7.jpg",
        "size": 173972,
        "sha256": "94a825462c9a1972ea728f0ac550d57ad4d294f5d309015e6b6efc55588bd4f6"
      },
      "353": {
        "name": "344_jpg.rf.9b41aa8481beea2de50a47dafb05b141.jpg",
        "size": 72416,
        "sha256": "e7533c227ed669da9537cf19766fea2e2a17c8111922b4b12f7dcd38200efbf4"
      },
      "354": {
        "name": "252_jpg.rf.a878862620baa7e8809b941897f22c6f.jpg",
        "size": 32949,
        "sha256": "c84e80f920c76785e16bc558e63ffee04bd2d353890a8a451f37cdc987184ed8"
      },
      "355": {
        "name": "022_jpg.rf.9f1c7ae931ad2359714b49af9fd05cba.jpg",
        "size": 38558,
        "sha256": "0c733792fc9c859e677883ecf9e9a9244ad903fa4c6550bcce78384be96df2a9"
      },
      "356": {
        "name": "144_jpg.rf.a339c41cdedb43535ce3033f59d7a770.jpg",
        "size": 67486,
        "sha256": "7f684802182e82ebabc15113032a2f8b0a0e4a76db67b660023b08bea20659b0"
      },
      "357": {
        "name": "268_jpg.rf.a6a3f4fdc7bbe12fe0e923eda2406a59.jpg",
        "size": 142536,
        "sha256": "73f762800fc1a38412d3f20ab894c18f9d6568e53bdbda7946e3500fc3418c05"
      },
      "358": {
        "name": "194_jpg.rf.a50b4e46e0f927eab7cf94d63dece2ac.jpg",
        "size": 67836,
        "sha256": "71e77a6bba415c4208155ba616285e96fcabb8752e316a73d028e36d34c2c902"
      }
    },
    "personal": {
      "1": {
        "name": "test/P001_jpg.rf.a27fff9eabe86eecf317923a7ef41e47.jpg",
        "size": 1044640,
        "sha256": "132ce9feb1ce8960e02d279b43d9fbb1dee8aebf5f909bf3e690a6717e0f85af"
      },
      "2": {
        "name": "test/P002_jpg.rf.9f7ef107060bde893c085661b4d3db11.jpg",
        "size": 858767,
        "sha256": "9c71eb39cee4e14081a53cc9ae457b356f66a1614bf9a5aefd287a6aa17a4444"
      },
      "3": {
        "name": "test/P003_jpg.rf.23423c03fcb67cb407b3c8e8904e8062.jpg",
        "size": 757146,
        "sha256": "80b7821d6b6d2e318100289d85f86cb986504c5932caf3ac90c2a9ed5d1a9c55"
      },
      "4": {
        "name": "test/P004_jpg.rf.cf53746555abaab3a66a982b71ecbd27.jpg",
        "size": 621091,
        "sha256": "b5e06980608e6b738a5abfa057f23eca941dab1d4982b9cc5be04317e9ff062a"
      },
      "5": {
        "name": "test/P005_jpg.rf.6608068fa6e1abdc77bcec4c94b131bd.jpg",
        "size": 959411,
        "sha256": "72e27969d4e4953fe0d4c01431c5ac639b57f4b6ee01a07899a6e9fafa9824f1"
      },
      "6": {
        "name": "test/P006_jpg.rf.856ca8c9788b12eff0f7555380d2ce90.jpg",
        "size": 769482,
        "sha256": "8a4b17f56182a1b9e4c47c77e8aa2bc4f1ea23211d341062d1f59769dea9835f"
      },
      "7": {
        "name": "test/P007_jpg.rf.e54add7614912be2a743e2aaa6385489.jpg",
        "size": 732571,
        "sha256": "b67ff0fddd4a492e62c1840a4bff04c91895c9b8f4e8e01f805a7bbf72bf8798"
      },
      "8": {
        "name": "test/P008_jpg.rf.eb835e9dcfdf30f8befe6d11cde146fc.jpg",
        "size": 1146119,
        "sha256": "5bc7cc47a94f11c6affd09dec75b1db63591db780a50d7e99ec1ee37145847f8"
      },
      "9": {
        "name": "test/P009_jpg.rf.b8add004862d420964f62ff0e8b4e15e.jpg",
        "size": 708344,
        "sha256": "7531060ab5d2896f589440b3105af5de7b63229df575ded299b398f704fb5190"
      },
      "10": {
        "name": "test/P010_jpg.rf.200d0ac7266cfdc9722f58b63e0358f7.jpg",
        "size": 734280,
        "sha256": "b3623b9544cf7644c8d4d090e79c0298ecc5972b5d5bc4ff2a134bc0fa145ae2"
      },
      "11": {
        "name": "test/P011_jpg.rf.baa9c9d356b6ab48aa8d577a1b92c041.jpg",
        "size": 949336,
        "sha256": "eafa8990d1c64b3b876c982c0d96c6b2562b6bee3f837f11f0867d2511e67c16"
      },
      "12": {
        "name": "test/P012_jpg.rf.35a7d8c804936f45b40cf5799594a711.jpg",
        "size": 676192,
        "sha256": "f6d9c47f9e73552c58eff699c646553205efd8a9cf4d1032d6642ff07ea363aa"
      },
      "13": {
        "name": "test/P013_jpg.rf.ba9b8945714d5c83606accc3707ae835.jpg",
        "size": 606762,
        "sha256": "5ac385db2a504c7fcfbe6852bf0233c0489f1079665f6b62c06c04e64bfb3676"
      },
      "14": {
        "name": "test/P014_jpg.rf.244ae94dced6c753f86cae19836db1c8.jpg",
        "size": 596410,
        "sha256": "ea240ee1d710d54a7258fd7c329879eeefa28e563aca725b44c1799c97723398"
      },
      "15": {
        "name": "test/P015_jpg.rf.d7b5ef7f14cd0daa5e708b882204c3a2.jpg",
        "size": 682018,
        "sha256": "5448eb27fd55e78e497fc95d171e88ad2b743437b99a5a19d7def520296355fa"
      },
      "16": {
        "name": "test/P016_jpg.rf.ac406bbf554acea74d4c846965c9258c.jpg",
        "size": 763800,
        "sha256": "00072183abdc1dd5bef05e5a06c3147976f8fc906a0306b3c4f440a9b3bed3a8"
      },
      "17": {
        "name": "test/P017_jpg.rf.95038e5350578fe15367ff5f6481892a.jpg",
        "size": 612406,
        "sha256": "388273f3a732e589ae9ade4a3eaa5f3191840eed88076f4e135e230eb9dce00a"
      },
      "18": {
        "name": "test/P018_jpg.rf.170198f77d2e2fe4d6fc831ab7b57e81.jpg",
        "size": 760070,
        "sha256": "a1c1feea66e03a0b17e9f9dcfcd29e23aa2cb03bc0979b0680fd84a28d28cf01"
      },
      "19": {
        "name": "test/P019_jpg.rf.4d385f3708fc3ba9429d375449900100.jpg",
        "size": 723018,
        "sha256": "37880a0358c3b816fbada3ff64f592078035906f82cab34c12b941c5a5b60b8d"
      },
      "20": {
        "name": "test/P020_jpg.rf.f63e968e2db38fd7d5d4dce8e931950e.jpg",
        "size": 608887,
        "sha256": "ee03356fd1a7a67b69276525ee55236495af1b628d12c477a9dd4201bc042fcd"
      },
      "21": {
        "name": "test/P021_jpg.rf.102c811d200e66d723638749f817098f.jpg",
        "size": 666314,
        "sha256": "b0b564d1977a0aa96cb92c4370a25f16ade36fe59df1af23ce8d62a977b75db8"
      },
      "22": {
        "name": "test/P022_jpg.rf.b03ad5f4a46f144d1db20b150f9d3c60.jpg",
        "size": 804199,
        "sha256": "4ea5bd2f6909969844b308792e75b0b2ce6cb97f27e2eab9a0e0500876fb30fd"
      },
      "23": {
        "name": "test/P023_jpg.rf.626fff340cd9c1d066b36cda9217e026.jpg",
        "size": 629948,
        "sha256": "9c730cf4b696166d535062c39868e49674064931598c75909c4efc1fcebbf3e4"
      },
      "24": {
        "name": "test/P024_jpg.rf.ddee7dfea29ba7eb54ff71242127e760.jpg",
        "size": 750322,
        "sha256": "7e725d4a1e07b018c1ca934903bd9f8be402f0ad0a5541e42b9830210aaeb5b2"
      },
      "25": {
        "name": "test/P025_jpg.rf.0777945b1017aea036cfeb34e089a66c.jpg",
        "size": 553112,
        "sha256": "8637711ba226f95590e24cb2617c7b8c4cbe58fffbbfc7ab1058de25da2404db"
      },
      "26": {
        "name": "test/P026_jpg.rf.8febd6cf9a95eb6201e71007fb6b8258.jpg",
        "size": 542393,
        "sha256": "dad3c0b25d8c06a595e9afc995d8a0461513d9646ad1cdf8f327e223361899a8"
      },
      "27": {
        "name": "test/P027_jpg.rf.48e56afc96832880827f9829c4c4e548.jpg",
        "size": 691735,
        "sha256": "8a4c00f4618006537dbc1b0287158e444c2ccfd7c73098a494837691761f78cc"
      },
      "28": {
        "name": "test/P028_jpg.rf.4ddb811fb104f6f426aaaba46a90a2f8.jpg",
        "size": 1018947,
        "sha256": "e0a71d14614f095e5b20f89c34cacd4158ee45feeea345c067a82fea6cefb0eb"
      },
      "29": {
        "name": "test/P029_jpg.rf.a728e270c1e3705de4b3a9176dcb478f.jpg",
        "size": 550315,
        "sha256": "afabe31edb1a26251dde511d189b54991d498ccbded8e25d30e4bea8d9191bf1"
      },
      "30": {
        "name": "test/P030_jpg.rf.f005e145dc5b33779a80883b2997cbf6.jpg",
        "size": 916696,
        "sha256": "ed86b45fa77bbbddd633acf648db497a95023df1fe6af0e9ba922ebd193a2562"
      },
      "31": {
        "name": "test/P031_jpg.rf.9f77e192cccd07a34470439b66aa7478.jpg",
        "size": 429665,
        "sha256": "20c6b35eca9328b16dc9a33f609f449c92590fb07220c786aeb780286564f5cc"
      },
      "32": {
        "name": "test/P032_jpg.rf.7785c57d80a7a632977592c962b6d6aa.jpg",
        "size": 1183098,
        "sha256": "aad94cbc07c386b549d2bf6696692a1d84732a552a9e3f1ac6a10ac810f94e1a"
      },
      "33": {
        "name": "test/P033_jpg.rf.070b42fc916a28dbcf1faf844c51db4b.jpg",
        "size": 1018468,
        "sha256": "57569ac90d277cb9369b37d53c81486c5bcc4faff44df1714e05b97b15d27781"
      },
      "34": {
        "name": "test/P034_jpg.rf.ab6bf1191cc004a2440085595d58fc15.jpg",
        "size": 699547,
        "sha256": "2568c4f19502c8b022e795abf3cbb5b01a6028449a7c772f989720bd7330acb1"
      },
      "35": {
        "name": "test/P035_jpg.rf.86807732bb53116d99396c1b8621382e.jpg",
        "size": 596469,
        "sha256": "2efcd4c546a6fa562b9e279a3acea85e132da996c0e5744cf581d2800467b4a1"
      },
      "36": {
        "name": "test/P036_jpg.rf.e69ed7c56d88d4178042566886fc64f3.jpg",
        "size": 484595,
        "sha256": "b56718fca4719b78902e2056175a2c5ce5137f07bedde98da857ec4dc8f3c530"
      },
      "37": {
        "name": "test/P037_jpg.rf.ebcef20efe77f07fd0fa3bfe127c1c13.jpg",
        "size": 666226,
        "sha256": "6151f2feadce41006407a29e2c3adfeb13fad172abe04c733ccd6cea344b1741"
      },
      "38": {
        "name": "test/P038_jpg.rf.80dda60873b08d586cfc4dd0a37b3f9d.jpg",
        "size": 793071,
        "sha256": "e7e08f433521bc62bd26d0df4cdcdf8ad13c547e1e85067d6dc5ed5f0825f8e8"
      },
      "39": {
        "name": "test/P039_jpg.rf.64046e4a55d14bb5d57dda682bed6490.jpg",
        "size": 778726,
        "sha256": "4474e96f2b9be12d7604924db3245351b5f62974d52f149bf9fda28e7561c658"
      },
      "40": {
        "name": "test/P040_jpg.rf.56a4d8128f99bfcf1139cc1b46ba15b9.jpg",
        "size": 574693,
        "sha256": "f4920a137e9551a7cc29ba0285ac2224dedc59b744d3e774cdc96d8408536c7b"
      }
    }
  }
}

### 03 Kaggle input discovery and integrity

In [5]:
PROTOCOL = {
    "confidence": 0.001, "max_predictions": 100, "mask_threshold": 0.5,
    "yolo": {"imgsz": 640, "iou": 0.7, "rect": True, "retina_masks": True,
             "agnostic_nms": False, "batch": 1, "half": False},
    "maskrcnn": {"min_size": 640, "max_size": 960, "box_nms_thresh": 0.5,
                 "backbone_norm": "FrozenBatchNorm2d", "norm_eps": 0.0, "batch": 1},
    "coco": {"iou": np.linspace(0.50, 0.95, 10).tolist(),
             "recall": np.linspace(0, 1, 101).tolist(), "maxDets": [1, 10, 100],
             "area": [[0, 1e10], [0, 32**2], [32**2, 96**2], [96**2, 1e10]],
             "area_labels": ["all", "small", "medium", "large"]},
    "precision": "float32; autocast dan TF32 nonaktif",
    "smoke_subset": "ID positif terendah + ID negatif terendah jika ada; sisanya ID terendah",
    "seed": 42,
}


def require(condition, detail):
    if not condition:
        raise ValueError(detail)


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def canonical(data):
    return json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False).encode("utf-8")


def fingerprint(data):
    return hashlib.sha256(canonical(data)).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_bytes())


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_bytes(canonical(data) + b"\n")
    temp.replace(path)


def safe_member(name):
    path = PurePosixPath(name)
    require(not path.is_absolute() and ".." not in path.parts and "\\" not in name
            and all(":" not in part for part in path.parts), f"Path arsip tidak aman: {name}")
    return path


class InputIndex:
    """Cari file biasa atau anggota ZIP terverifikasi, ekstrak hanya file yang diminta."""

    def __init__(self, roots, cache):
        self.cache = cache
        self.by_name = {}
        self.archives = []
        self.indexed_archives = False
        self.errors = []
        files = set()
        for root in map(Path, roots):
            require(root.exists(), f"Input root tidak ditemukan: {root}")
            files.update(root.rglob("*") if root.is_dir() else [root])
        for path in sorted(files):
            if path.is_file():
                self.by_name.setdefault(path.name, []).append((path, None, path.stat().st_size))
                if path.suffix.lower() == ".zip":
                    self.archives.append(path)

    def index_archives(self):
        if self.indexed_archives:
            return
        self.indexed_archives = True
        for archive in self.archives:
            try:
                with zipfile.ZipFile(archive) as z:
                    counts = Counter(member.filename for member in z.infolist())
                    for member in z.infolist():
                        if member.is_dir() or counts[member.filename] != 1:
                            continue
                        path = safe_member(member.filename)
                        self.by_name.setdefault(path.name, []).append((archive, member.filename, member.file_size))
            except (OSError, zipfile.BadZipFile, ValueError) as exc:
                self.errors.append(f"{archive}: {exc}")

    def find(self, names, digest, size=None, override=None):
        if override:
            path = Path(override)
            require(path.is_file() and sha256(path) == digest, f"Override tidak cocok checksum: {path}")
            return path, str(path)
        probes = []
        seen = set()
        for archived in (False, True):
            if archived:
                self.index_archives()
            for name in names:
                for path, member, actual_size in self.by_name.get(name, []):
                    if (member is not None) != archived or (path, member) in seen:
                        continue
                    seen.add((path, member))
                    if size is not None and size != actual_size:
                        continue
                    if member is None:
                        if sha256(path) == digest:
                            return path, str(path)
                        probes.append(str(path))
                        continue
                    cached = self.cache / digest / name
                    if cached.is_file() and sha256(cached) == digest:
                        return cached, f"cache SHA-256 {digest}"
                    cached.parent.mkdir(parents=True, exist_ok=True)
                    temp = cached.with_suffix(cached.suffix + ".tmp")
                    with zipfile.ZipFile(path) as z, z.open(member) as src, temp.open("wb") as dest:
                        shutil.copyfileobj(src, dest)
                    if sha256(temp) == digest:
                        temp.replace(cached)
                        return cached, f"{path}!{member}"
                    temp.unlink()
                    probes.append(f"{path}!{member}")
        raise FileNotFoundError(f"Tidak ada input {names} dengan SHA-256 {digest}. "
                                f"File berbeda: {probes[:5]}. Masalah ZIP: {self.errors[:3]}")

### 04 Ground truth validation and category mapping

In [6]:
def coco_index(dataset):
    with redirect_stdout(io.StringIO()):
        coco = COCO()
        coco.dataset = dataset
        coco.createIndex()
    return coco


def rows_by_id(records):
    table = {r["id"]: r for r in records}
    require(len(table) == len(records) and all(type(i) is int for i in table), "ID COCO duplikat atau bukan integer.")
    return table


def category_names(categories):
    return {i: r["name"] for i, r in rows_by_id(categories).items()}


def project_hard3(dataset, mapping, categories):
    return dict(dataset, categories=categories, annotations=[
        dict(a, category_id=mapping[a["category_id"]]) for a in dataset["annotations"] if a["category_id"] in mapping
    ])


def rle_for_annotation(annotation, shape):
    height, width = shape
    segments = annotation["segmentation"]
    if isinstance(segments, list):
        require(segments and all(len(p) >= 6 and len(p) % 2 == 0 and np.isfinite(p).all() for p in segments),
                f"Poligon tidak valid: {annotation['id']}")
        return coco_mask.merge(coco_mask.frPyObjects(segments, height, width))
    require(segments["size"] == [height, width], "Ukuran RLE ground truth salah.")
    if isinstance(segments["counts"], list):
        return coco_mask.frPyObjects(segments, height, width)
    return dict(segments, counts=segments["counts"].encode("ascii"))


def validate_gt(dataset, names, expected):
    images = rows_by_id(dataset["images"])
    ann = rows_by_id(dataset["annotations"])
    require(category_names(dataset["categories"]) == names, "Taksonomi COCO berbeda dari mapping training.")
    require(len(images) == expected["images"] and len(ann) == expected["instances"], "Ukuran dataset berubah.")
    require(len({row["file_name"] for row in images.values()}) == len(images), "Nama gambar duplikat.")
    counts, represented = Counter(), set()
    positive_images = set()
    for record in ann.values():
        image_id, cat_id = record["image_id"], record["category_id"]
        require(image_id in images and cat_id in names and record["iscrowd"] in (0, 1), "Referensi anotasi salah.")
        image = images[image_id]
        shape = (image["height"], image["width"])
        x, y, w, h = record["bbox"]
        require(np.isfinite([x, y, w, h, record["area"]]).all() and min(w, h, record["area"]) > 0,
                f"Bounding box atau area tidak valid: {record['id']}")
        require(coco_mask.area(rle_for_annotation(record, shape)) > 0, f"Mask kosong: {record['id']}")
        counts[names[cat_id]] += 1
        positive_images.add(image_id)
        if not record["iscrowd"]:
            represented.add(cat_id)
    for key, actual in (("classes_present", len(counts)), ("positive", len(positive_images)),
                        ("negative", len(images) - len(positive_images)), ("class_counts", dict(counts))):
        require(key not in expected or actual == expected[key], f"{key} berbeda dari audit. Aktual: {actual}")
    return {"images": len(images), "instances": len(ann), "positive_images": len(positive_images),
            "negative_images": len(images) - len(positive_images), "class_counts": dict(counts),
            "represented_categories": sorted(represented)}


def prepare_datasets(index, lock):
    sources = {}
    for key, item in lock["references"].items():
        path, source = index.find([item["name"]], item["sha256"], item["size"])
        sources[key] = (read_json(path), source)
    freeze, mapping = sources["freeze"][0], sources["mapping"][0]
    require(freeze["ground_truth_frozen"] and freeze["status"] == "FROZEN", "Freeze personal tidak sah.")
    require(freeze["provenance"]["review_artifacts"]["sha256"] == lock["references"]["personal_provenance"]["sha256"],
            "Provenance review berbeda dari freeze record.")
    rows = mapping["classes"]
    all_names = {r["all_class_coco"]: r["name"] for r in rows}
    hard_names = {r["hard3_coco"]: r["name"] for r in rows if r["hard3_coco"] is not None}
    hard_map = {r["all_class_coco"]: r["hard3_coco"] for r in rows if r["hard3_coco"] is not None}
    require(all_names == dict(enumerate(lock["class_order"]["all"], 1)) and
            hard_names == dict(enumerate(lock["class_order"]["hard3"], 1)), "Urutan nama kelas tidak cocok.")
    require(all(r["all_class_yolo"] + 1 == r["all_class_coco"] == r["all_class_maskrcnn"] for r in rows),
            "Mapping All-Class YOLO/Mask R-CNN tidak cocok.")
    require(all(r["hard3_yolo"] + 1 == r["hard3_coco"] == r["hard3_maskrcnn"]
                for r in rows if r["hard3_coco"] is not None), "Mapping Hard-3 YOLO/Mask R-CNN tidak cocok.")
    require(mapping["maskrcnn_background_id"] == 0, "Mapping background Mask R-CNN berbeda.")

    for key, approval_key in (("personal_all", "all_class_master"), ("personal_hard3", "hard3")):
        require(lock["references"][key]["sha256"] == freeze["evaluation_inputs"][approval_key]["sha256"],
                f"Hash {key} berbeda dari freeze record.")
    original = sources["original_all"][0]
    require(original["categories"][0]["id"] == 0 and not any(a["category_id"] == 0 for a in original["annotations"]),
            "Kategori induk COCO asli berubah.")
    original = dict(original, categories=[r for r in original["categories"] if r["id"] != 0])
    personal, personal_hard = sources["personal_all"][0], sources["personal_hard3"][0]
    original_hard = project_hard3(original, hard_map, personal_hard["categories"])
    same_personal = project_hard3(personal, hard_map, personal_hard["categories"])
    require(same_personal["images"] == personal_hard["images"] and
            sorted(same_personal["annotations"], key=lambda a: a["id"]) ==
            sorted(personal_hard["annotations"], key=lambda a: a["id"]),
            "Versi personal Hard-3 berbeda dari proyeksi ground truth master.")
    datasets = {}
    for domain, all_gt, hard_gt in (("original", original, original_hard), ("personal", personal, personal_hard)):
        paths, image_map = {}, []
        image_lock = lock["images"][domain]
        for image in sorted(all_gt["images"], key=lambda row: row["id"]):
            image_id, file_name = image["id"], image["file_name"]
            basename = safe_member(file_name).name
            record = image_lock[str(image_id)]
            require(record["name"] == file_name, f"File name COCO berbeda: {file_name}")
            photo_id = None
            if domain == "personal":
                photo_id = f"P{image_id:03d}"
                require(basename.startswith(photo_id + "_"), f"Identitas foto tidak cocok: {basename}")
            path, source = index.find([basename], record["sha256"], record["size"])
            with Image.open(path) as decoded:
                require(decoded.size == (image["width"], image["height"]) and decoded.getexif().get(274, 1) == 1,
                        f"Orientasi atau dimensi berbeda: {file_name}")
                decoded.load()
            paths[image_id] = path
            image_map.append(dict(image, photo_id=photo_id, sha256=record["sha256"], input_source=source))
        for scope, gt, names in (("all", all_gt, all_names), ("hard3", hard_gt, hard_names)):
            key = f"{domain}_{scope}"
            source_key = "original_all" if domain == "original" else key
            datasets[key] = {
                "gt": gt, "paths": paths, "image_map": image_map,
                "summary": validate_gt(gt, names, lock["counts"][key]),
                "effective_gt_sha256": fingerprint(gt),
                "source_gt_sha256": lock["references"][source_key]["sha256"],
                "source_gt": sources[source_key][1],
                "image_set_sha256": fingerprint(image_lock),
                "categories": names,
            }
    return datasets, hard_map, sources

### 05 Checkpoint verification and model reconstruction

In [7]:
def runtime_environment(device):
    return {"python": platform.python_version(), "platform": platform.platform(),
            "torch": torch.__version__, "torchvision": torchvision.__version__,
            "torch_cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
            "gpu": torch.cuda.get_device_name(device) if device != "cpu" else None, "device": device,
            "ultralytics": ultralytics.__version__, "pycocotools": version("pycocotools"),
            "numpy": np.__version__, "opencv": cv2.__version__, "pillow": version("Pillow"),
            "pandas": pd.__version__, "matplotlib": version("matplotlib")}


def configure_runtime(device):
    torch.manual_seed(PROTOCOL["seed"])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    if device != "cpu":
        torch.cuda.manual_seed_all(PROTOCOL["seed"])
    torchvision.extension._assert_has_ops()
    torchvision.ops.nms(torch.tensor([[0., 0., 1., 1.]], device=device),
                        torch.ones(1, device=device), 0.5)


def freeze_backbone_norm(layer, eps):
    for name, child in layer.named_children():
        if isinstance(child, torch.nn.BatchNorm2d):
            setattr(layer, name, FrozenBatchNorm2d(child.num_features, eps=eps))
        else:
            freeze_backbone_norm(child, eps)


def load_model(spec, path, device, class_order):
    expected = class_order[spec["scope"]]
    if spec["family"] != "maskrcnn":
        model = YOLO(str(path), task="segment")
        head = model.model.model[-1]
        actual = [model.names[i] for i in range(len(model.names))]
        require(model.task == "segment" and head.__class__.__name__ == "Segment" and
                head.nc == len(expected) and actual == expected, "Head segmentasi atau urutan kelas YOLO berbeda.")
        yaml_name = Path(str(model.model.yaml.get("yaml_file", ""))).name
        expected_yaml = {"yolo11n": ("yolo11n-seg.yaml", "C3k2"), "yolov8n": ("yolov8n-seg.yaml", "C2f")}
        require((yaml_name, model.model.yaml["backbone"][2][2]) == expected_yaml[spec["family"]] and
                model.overrides.get("imgsz") == PROTOCOL["yolo"]["imgsz"], "Arsitektur atau resolusi training YOLO berbeda.")
        model.model.float().eval()
        model.to(device)
        return model, {"classes": actual, "head": type(head).__name__, "nc": head.nc,
                       "model_yaml": yaml_name, "training_imgsz": model.overrides["imgsz"]}

    config = PROTOCOL["maskrcnn"]
    # Muat state beku secara langsung, tanpa download bobot pralatih.
    model = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None, num_classes=len(expected) + 1,
                                min_size=config["min_size"], max_size=config["max_size"],
                                box_detections_per_img=PROTOCOL["max_predictions"],
                                box_score_thresh=PROTOCOL["confidence"], box_nms_thresh=config["box_nms_thresh"])
    freeze_backbone_norm(model.backbone.body, config["norm_eps"])
    norms = [m for m in model.backbone.body.modules() if isinstance(m, FrozenBatchNorm2d)]
    require(len(norms) == 53 and all(m.eps == 0 for m in norms), "Backbone FrozenBatchNorm berbeda dari ResNet-50 COCO V1.")
    # TorchVersion merupakan satu metadata yang diizinkan pada checkpoint historis.
    with torch.serialization.safe_globals([torch.torch_version.TorchVersion]):
        ckpt = torch.load(path, map_location="cpu", weights_only=True)
    require("model_state_dict" in ckpt, "Checkpoint Mask R-CNN tidak memiliki model_state_dict.")
    if spec["scope"] == "hard3":
        require(ckpt["classes"] == expected, "Nama kelas checkpoint Mask R-CNN Hard-3 berbeda.")
    state = ckpt["model_state_dict"]
    require(not any(k.endswith("num_batches_tracked") for k in state), "Checkpoint bukan backbone FrozenBatchNorm.")
    model.load_state_dict(state, strict=True)
    heads = (model.roi_heads.box_predictor.cls_score.out_features,
             model.roi_heads.mask_predictor.mask_fcn_logits.out_channels)
    require(heads == (len(expected) + 1,) * 2, "Ukuran box/mask predictor salah.")
    evidence = {"classes": expected, "background_id": 0, "heads_with_background": list(heads),
                "frozen_bn_layers": len(norms), "eps": config["norm_eps"],
                "class_order_evidence": "checkpoint.classes" if spec["scope"] == "hard3" else
                "COCO category_id dari training + taxonomy lock + SHA-256 checkpoint",
                "checkpoint_epoch": int(ckpt["epoch"])}
    del state, ckpt
    return model.float().eval().to(device), evidence


def release_model(device):
    gc.collect()
    if device != "cpu":
        torch.cuda.empty_cache()


def resolve_models(index, lock, overrides):
    require(set(overrides) <= set(lock["models"]), "Key checkpoint override tidak dikenal.")
    models = {}
    for key, spec in lock["models"].items():
        names = [spec["alias"], Path(spec["local"]).name]
        path, source = index.find(names, spec["sha256"], spec["size"], overrides.get(key))
        models[key] = (path, source)
    return models


def check_models(models, lock, device):
    summary = {}
    for key, (path, source) in models.items():
        spec = lock["models"][key]
        require(sha256(path) == spec["sha256"], f"Bobot berubah selama preflight: {key}.")
        model, details = load_model(spec, path, device, lock["class_order"])
        summary[key] = {"source": source, "sha256": spec["sha256"], "loading": details}
        del model
        release_model(device)
        print(f"Checkpoint valid: {key}", flush=True)
    return summary

### 06 Unified inference and prediction encoding

In [8]:
def sync_device(device):
    if device != "cpu":
        torch.cuda.synchronize(device)


def encode_mask(binary):
    encoded = coco_mask.encode(np.asfortranarray(binary, dtype=np.uint8))
    encoded["counts"] = encoded["counts"].decode("ascii")
    return encoded


def append_detections(boxes, labels, scores, masks, image_id, shape, valid_ids, bbox, segm):
    require(len(boxes) == len(labels) == len(scores) and len(boxes) <= PROTOCOL["max_predictions"],
            "Jumlah deteksi tidak konsisten.")
    if len(boxes):
        require(masks is not None and tuple(masks.shape) == (len(boxes), *shape), "Mask bukan resolusi asli.")
    height, width = shape
    for i, (box, label, score) in enumerate(zip(boxes, labels, scores)):
        require(np.isfinite([*box, score, label]).all() and label == int(label) and int(label) in valid_ids
                and 0 <= score <= 1, f"Deteksi nonfinite, skor atau kelas tidak valid pada {image_id}.")
        if score < PROTOCOL["confidence"]:
            continue
        x1, y1, x2, y2 = np.clip(box, 0, [width, height, width, height]).tolist()
        if x2 <= x1 or y2 <= y1:
            continue
        row = {"image_id": image_id, "category_id": int(label), "score": float(score),
               "bbox": [x1, y1, x2 - x1, y2 - y1]}
        bbox.append(row)
        one = masks[i]
        if torch.is_tensor(one):
            require(bool(torch.isfinite(one).all().item()), f"Mask nonfinite pada {image_id}.")
            binary = (one >= PROTOCOL["mask_threshold"]).to("cpu", dtype=torch.uint8).numpy()
        else:
            binary = one >= PROTOCOL["mask_threshold"]
        if binary.any():
            segm.append(dict(row, segmentation=encode_mask(binary)))


def infer_run01(model, spec, dataset, image_ids, device):
    paths = {i: dataset["paths"][i] for i in image_ids}
    metadata = {r["id"]: r for r in dataset["gt"]["images"]}
    valid_ids = set(dataset["categories"])
    bbox, segm, seen = [], [], set()
    stages = Counter()
    sync_device(device)
    started = time.perf_counter()
    with torch.inference_mode():
        if spec["family"] == "maskrcnn":
            for image_id, path in paths.items():
                point = time.perf_counter()
                with Image.open(path) as photo:
                    tensor = to_tensor(photo.convert("RGB")).to(device)
                sync_device(device)
                stages["input_decode_tensor_transfer_seconds"] += time.perf_counter() - point
                point = time.perf_counter()
                result = model([tensor])[0]
                sync_device(device)
                stages["torchvision_call_including_transforms_seconds"] += time.perf_counter() - point
                point = time.perf_counter()
                height, width = metadata[image_id]["height"], metadata[image_id]["width"]
                require(tuple(tensor.shape[-2:]) == (height, width), "Dimensi input Mask R-CNN berbeda.")
                append_detections(result["boxes"].cpu().numpy(), result["labels"].cpu().numpy(),
                                  result["scores"].cpu().numpy(), result["masks"][:, 0], image_id, (height, width),
                                  valid_ids, bbox, segm)
                seen.add(image_id)
                stages["prediction_serialization_preparation_seconds"] += time.perf_counter() - point
                del result, tensor
        else:
            by_path = {str(path.resolve()): i for i, path in paths.items()}
            outputs = model.predict(source=[str(path.resolve()) for path in paths.values()], stream=True,
                                    device=device, conf=PROTOCOL["confidence"], max_det=PROTOCOL["max_predictions"],
                                    save=False, verbose=False, **PROTOCOL["yolo"])
            for output in outputs:
                point = time.perf_counter()
                image_id = by_path[str(Path(output.path).resolve())]
                require(image_id not in seen, f"Prediksi YOLO duplikat pada gambar {image_id}.")
                seen.add(image_id)
                shape = (metadata[image_id]["height"], metadata[image_id]["width"])
                require(tuple(output.orig_shape) == shape, f"Dimensi YOLO berbeda pada {image_id}.")
                for stage, ms in output.speed.items():
                    if ms is not None:
                        stages[f"ultralytics_reported_{stage}_seconds"] += ms / 1000
                if output.boxes is not None and len(output.boxes):
                    append_detections(output.boxes.xyxy.cpu().numpy(), output.boxes.cls.cpu().numpy() + 1,
                                      output.boxes.conf.cpu().numpy(), output.masks.data if output.masks else None,
                                      image_id, shape, valid_ids, bbox, segm)
                stages["prediction_serialization_preparation_seconds"] += time.perf_counter() - point
    sync_device(device)
    require(seen == set(image_ids), "Sebagian gambar tidak mendapat giliran inferensi.")
    return {"bbox": bbox, "segm": segm}, {
        **stages, "inference_pipeline_seconds": time.perf_counter() - started,
        "images_processed": len(seen), "predictions_bbox": len(bbox), "predictions_segm": len(segm),
        "method": "wall clock dengan sinkronisasi CUDA; angka tahap YOLO merupakan timing Ultralytics",
    }

from ultralytics.models.yolo.segment.predict import SegmentationPredictor
from torchvision.models.detection.roi_heads import paste_masks_in_image


def memory_snapshot(device):
    if device == "cpu":
        return {}
    return {"gpu_allocated_bytes": torch.cuda.memory_allocated(device),
            "gpu_reserved_bytes": torch.cuda.memory_reserved(device),
            "gpu_peak_allocated_bytes": torch.cuda.max_memory_allocated(device),
            "gpu_peak_reserved_bytes": torch.cuda.max_memory_reserved(device)}


class RecoverySegPredictor(SegmentationPredictor):
    def preprocess(self, imgs):
        require(len(imgs) == 1, "Ultralytics menerima batch lebih besar dari satu.")
        self.recovery_diagnostic.update(stage="yolo_preprocess", actual_batch_size=len(imgs))
        prepared = super().preprocess(imgs)
        require(tuple(prepared.shape) == (1, 3, 640, 640) and prepared.dtype == torch.float32,
                "Tensor model tidak memakai kanvas square FP32 Run01.")
        self.recovery_input_shape = list(prepared.shape)
        self.recovery_diagnostic["stage"] = "yolo_forward_and_native_masks"
        return prepared


def yolo_mask_crops(masks, shape, count):
    if count == 0:
        return
    require(masks is not None and tuple(masks.shape) == (count, *shape),
            "Shape mask YOLO tidak cocok dengan foto.")
    for (x1, y1, x2, y2), crop in masks.crops():
        require(0 <= x1 <= x2 <= shape[1] and 0 <= y1 <= y2 <= shape[0],
                "Koordinat crop mask di luar gambar.")
        require(tuple(crop.shape) == (y2 - y1, x2 - x1), "Shape crop mask tidak valid.")
        if torch.is_tensor(crop):
            require(bool(torch.isfinite(crop).all().item()), "Mask YOLO nonfinite.")
            binary = (crop >= PROTOCOL["mask_threshold"]).to("cpu", dtype=torch.uint8).numpy()
        else:
            require(np.isfinite(crop).all(), "Mask YOLO nonfinite.")
            binary = crop >= PROTOCOL["mask_threshold"]
        canvas = np.zeros(shape, dtype=np.uint8, order="F")
        canvas[y1:y2, x1:x2] = binary
        yield canvas


def rcnn_mask_canvases(roi_masks, resized_boxes, shape):
    require(len(roi_masks) == len(resized_boxes), "Jumlah mask ROI dan box berbeda.")
    for i in range(len(resized_boxes)):
        # Fungsi, box yang sudah di-resize, padding dan interpolasi sama dengan postprocess Torchvision.
        pasted = paste_masks_in_image(roi_masks[i:i+1], resized_boxes[i:i+1], shape)[0, 0]
        require(tuple(pasted.shape) == shape and bool(torch.isfinite(pasted).all().item()),
                "Mask Mask R-CNN nonfinite atau bukan resolusi asli.")
        canvas = (pasted >= PROTOCOL["mask_threshold"]).to("cpu", dtype=torch.uint8).numpy()
        del pasted
        yield canvas


def rcnn_forward_without_dense_masks(model, tensor):
    transform = model.transform
    postprocess = transform.postprocess
    had_override = "postprocess" in vars(transform)
    captured = []

    def boxes_only(results, processed_sizes, original_sizes):
        captured.extend(result.pop("masks") for result in results)
        return postprocess(results, processed_sizes, original_sizes)

    try:
        transform.postprocess = boxes_only
        results = model([tensor])
        require(len(results) == len(captured) == 1, "Output Mask R-CNN bukan singleton.")
        result = results[0]
        result["roi_masks"] = captured[0]
        return result
    finally:
        if had_override:
            transform.postprocess = postprocess
        else:
            del transform.postprocess


def append_recovery_detections(boxes, labels, scores, canvases, image_id, shape, valid_ids, bbox, segm):
    require(len(boxes) == len(labels) == len(scores) and len(boxes) <= PROTOCOL["max_predictions"],
            "Jumlah deteksi berbeda atau melebihi batas.")
    height, width = shape
    for box, label, score, canvas in zip(boxes, labels, scores, canvases, strict=True):
        require(np.isfinite([*box, score, label]).all() and label == int(label) and int(label) in valid_ids
                and 0 <= score <= 1, f"Kelas, box atau skor tidak valid pada {image_id}.")
        if score < PROTOCOL["confidence"]:
            continue
        x1, y1, x2, y2 = np.clip(box, 0, [width, height, width, height]).tolist()
        if x2 <= x1 or y2 <= y1:
            continue
        row = {"image_id": image_id, "category_id": int(label), "score": float(score),
               "bbox": [x1, y1, x2 - x1, y2 - y1]}
        bbox.append(row)
        if canvas.any():
            segm.append(dict(row, segmentation=encode_mask(canvas)))


def infer_recovery(model, spec, dataset, image_ids, device, diagnostic):
    by_id = {row["id"]: row for row in dataset["gt"]["images"]}
    bbox, segm, timings = [], [], Counter()
    categories = set(dataset["categories"])
    seen = set()
    sync_device(device)
    start = time.perf_counter()
    if spec["family"] != "maskrcnn":
        model.predictor = None  # Jangan mewarisi predictor referensi dari stress test.
    with torch.inference_mode():
        for image_id in image_ids:
            path = dataset["paths"][image_id]
            meta = by_id[image_id]
            shape = (meta["height"], meta["width"])
            diagnostic.update(image_id=image_id, filename=meta["file_name"], original_shape=list(shape),
                              stage="load_or_forward", actual_batch_size=None, prediction_count=None,
                              **memory_snapshot(device))
            point = time.perf_counter()
            if spec["family"] == "maskrcnn":
                with Image.open(path) as photo:
                    tensor = to_tensor(photo.convert("RGB")).to(device)
                require(tuple(tensor.shape[-2:]) == shape, "Dimensi asli Mask R-CNN berbeda.")
                diagnostic["actual_batch_size"] = 1
                sync_device(device)
                timings["input_decode_tensor_transfer_seconds"] += time.perf_counter() - point
                point = time.perf_counter()
                result = rcnn_forward_without_dense_masks(model, tensor)
                del tensor
                sync_device(device)
                timings["torchvision_call_including_transforms_seconds"] += time.perf_counter() - point
                point = time.perf_counter()
                diagnostic.update(stage="serial_roi_paste_and_rle", prediction_count=len(result["boxes"]),
                                  **memory_snapshot(device))
                append_recovery_detections(result["boxes"].cpu().numpy(), result["labels"].cpu().numpy(),
                                           result["scores"].cpu().numpy(),
                                           rcnn_mask_canvases(result["roi_masks"], result["boxes"], shape),
                                           image_id, shape, categories, bbox, segm)
                del result
            else:
                # Full Run01 memiliki bentuk gambar campuran sehingga rect tidak mengaktifkan auto padding.
                options = dict(PROTOCOL["yolo"], rect=False)
                outputs = model.predict(source=[str(path.resolve())], stream=True, predictor=RecoverySegPredictor,
                                        device=device, conf=PROTOCOL["confidence"],
                                        max_det=PROTOCOL["max_predictions"], save=False, verbose=False, **options)
                require(isinstance(model.predictor, RecoverySegPredictor), "Predictor singleton tidak aktif.")
                model.predictor.recovery_diagnostic = diagnostic
                try:
                    output = next(outputs)
                    diagnostic.update(stage="crop_and_rle", actual_batch_size=model.predictor.recovery_input_shape[0],
                                      prediction_count=len(output.boxes) if output.boxes is not None else 0,
                                      **memory_snapshot(device))
                    require(Path(output.path).resolve() == path.resolve() and tuple(output.orig_shape) == shape,
                            "Identitas atau resolusi prediksi YOLO berbeda.")
                    for stage, ms in output.speed.items():
                        if ms is not None:
                            timings[f"ultralytics_reported_{stage}_seconds"] += ms / 1000
                    point = time.perf_counter()
                    if output.boxes is not None and len(output.boxes):
                        append_recovery_detections(output.boxes.xyxy.cpu().numpy(),
                                                   output.boxes.cls.cpu().numpy() + 1,
                                                   output.boxes.conf.cpu().numpy(),
                                                   yolo_mask_crops(output.masks, shape, len(output.boxes)),
                                                   image_id, shape, categories, bbox, segm)
                    del output
                    require(next(outputs, None) is None, "Sumber singleton menghasilkan lebih dari satu gambar.")
                finally:
                    outputs.close()
                    model.predictor.results = None
                    model.predictor.batch = None
                    model.predictor.dataset = None
            timings["prediction_serialization_preparation_seconds"] += time.perf_counter() - point
            require(image_id not in seen, "ID gambar duplikat saat inferensi.")
            seen.add(image_id)
    sync_device(device)
    require(seen == set(image_ids), "Roster inferensi tidak lengkap.")
    return {"bbox": bbox, "segm": segm}, {
        **timings, "inference_pipeline_seconds": time.perf_counter() - start,
        "images_processed": len(seen), "predictions_bbox": len(bbox), "predictions_segm": len(segm),
        "method": "wall clock CUDA sync; singleton square / serial mask encoding",
        "implementation": "recovery_memory_efficient",
    }

### 07 COCOeval and shared-category evaluation

In [9]:
AP_FIELDS = ("ap", "ap50", "ap75", "ap_small", "ap_medium", "ap_large",
             "ar1", "ar10", "ar100", "ar_small", "ar_medium", "ar_large")


def defined_mean(values):
    valid = np.asarray(values)[np.asarray(values) >= 0]
    return float(valid.mean()) if valid.size else None


def run_coco(ground_truth, records, kind, image_ids, category_ids):
    gt = coco_index(dict(ground_truth, info=ground_truth.get("info", {})))
    # COCO.loadRes memilih bbox terlebih dahulu. Untuk segm, area harus dihitung dari RLE.
    rows = [{k: deepcopy(v) for k, v in row.items() if kind != "segm" or k != "bbox"} for row in records]
    with redirect_stdout(io.StringIO()):
        if rows:
            detected = gt.loadRes(rows)
        else:
            detected = coco_index(dict(gt.dataset, annotations=[]))
        evaluator = COCOeval(gt, detected, kind)
        cfg = PROTOCOL["coco"]
        evaluator.params.imgIds = sorted(image_ids)
        evaluator.params.catIds = sorted(category_ids)
        evaluator.params.maxDets = cfg["maxDets"]
        evaluator.params.iouThrs = np.asarray(cfg["iou"])
        evaluator.params.recThrs = np.asarray(cfg["recall"])
        evaluator.params.areaRng = cfg["area"]
        evaluator.params.areaRngLbl = cfg["area_labels"]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return evaluator


def metric_scope(dataset, predictions, image_ids, category_ids, scope):
    gt = dataset["gt"]
    image_ids, category_ids = set(image_ids), set(category_ids)
    counts = Counter(a["category_id"] for a in gt["annotations"] if a["image_id"] in image_ids)
    usable = Counter(a["category_id"] for a in gt["annotations"] if a["image_id"] in image_ids and not a["iscrowd"])
    selections = {kind: [r for r in rows if r["category_id"] in category_ids and r["image_id"] in image_ids]
                  for kind, rows in predictions.items()}
    aggregate = {"evaluation_scope": scope, "images": len(image_ids), "gt_instances": sum(counts[c] for c in category_ids),
                 "categories_requested": len(category_ids), "categories_with_non_crowd_gt": sum(usable[c] > 0 for c in category_ids),
                 "predictions_bbox": len(selections["bbox"]), "predictions_segm": len(selections["segm"])}
    per_class = {key: {"evaluation_scope": scope, "category_id": key, "class": dataset["categories"][key],
                       "gt_instances": counts[key], "gt_non_crowd_instances": usable[key]} for key in sorted(category_ids)}
    if not category_ids:
        aggregate.update({f"{kind}_{metric}": None for kind in ("box", "mask") for metric in AP_FIELDS})
        return aggregate, []
    for kind, prefix in (("segm", "mask"), ("bbox", "box")):
        ev = run_coco(gt, selections[kind], kind, image_ids, category_ids)
        require(len(ev.stats) == len(AP_FIELDS), "COCOeval stats schema berbeda.")
        aggregate.update({f"{prefix}_{m}": float(v) if v >= 0 else None for m, v in zip(AP_FIELDS, ev.stats)})
        precision, recall = ev.eval["precision"], ev.eval["recall"]
        for idx, cat in enumerate(ev.params.catIds):
            values = precision[:, :, idx, 0, -1]
            per_class[cat].update({
                f"{prefix}_ap": defined_mean(values), f"{prefix}_ap50": defined_mean(values[0]),
                f"{prefix}_ap75": defined_mean(values[5]), f"{prefix}_ar100": defined_mean(recall[:, idx, 0, -1]),
            })
    return aggregate, list(per_class.values())


def remap_predictions(predictions, mapping):
    return {kind: [dict(row, category_id=mapping[row["category_id"]])
                   for row in rows if row["category_id"] in mapping] for kind, rows in predictions.items()}


def smoke_ids(dataset):
    ids = sorted(dataset["paths"])
    positive = {a["image_id"] for a in dataset["gt"]["annotations"] if not a["iscrowd"]}
    picked = sorted(positive)[:1] + [i for i in ids if i not in positive][:1]
    return (picked + [i for i in ids if i not in picked])[:2]


def paired_inputs(datasets, spec, domain):
    scope = spec["scope"]
    current = datasets[f"{domain}_{scope}"]
    shared = set(datasets[f"original_{scope}"]["summary"]["represented_categories"]) & set(
        datasets[f"personal_{scope}"]["summary"]["represented_categories"])
    return current, shared


def score_pair(datasets, spec, domain, predictions, ids, hard_map, smoke):
    dataset, shared = paired_inputs(datasets, spec, domain)
    scopes = [(dataset, predictions, dataset["categories"], "primary")]
    if not smoke:
        scopes.append((dataset, predictions, shared, "shared_represented"))
        hard_dataset = datasets[f"{domain}_hard3"]
        hard_predictions = remap_predictions(predictions, hard_map) if spec["scope"] == "all" else predictions
        scopes.append((hard_dataset, hard_predictions, hard_dataset["categories"], "shared_hard3"))
    aggregate, per_class, measured = [], [], {}
    for item, results, cats, scope in scopes:
        identity = (id(item), id(results), tuple(sorted(cats)))
        if identity not in measured:
            measured[identity] = metric_scope(item, results, ids, cats, scope)
        metrics, classes = measured[identity]
        aggregate.append(dict(metrics, evaluation_scope=scope))
        per_class.extend(dict(row, evaluation_scope=scope) for row in classes)
    return {"label": "NON-FINAL" if smoke else "FINAL_TEST", "aggregate": aggregate, "per_class": per_class}

### 08 Artifacts and resume

In [10]:
COMBINATION_FILES = {"pair_config.json", "predictions_bbox.json.gz", "predictions_segm.json.gz",
                     "metrics.json", "timing.json"}
STATUS_FILE = "evaluation_status_manifest.json"


def json_gzip(path, rows):
    temp = path.with_suffix(".tmp")
    with temp.open("wb") as stream, gzip.GzipFile(fileobj=stream, mode="wb", filename="", mtime=0) as archive:
        with io.TextIOWrapper(archive, encoding="utf-8") as text:
            json.dump(rows, text, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
    temp.replace(path)


def read_gzip(path):
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        return json.load(stream)


def check_prediction_files(folder, dataset, ids):
    by_image = {r["id"]: r for r in dataset["gt"]["images"]}
    ids = set(ids)
    predictions = {kind: read_gzip(folder / f"predictions_{kind}.json.gz") for kind in ("bbox", "segm")}
    for kind, records in predictions.items():
        counts = Counter()
        for record in records:
            image_id, cat, score = record["image_id"], record["category_id"], record["score"]
            require(image_id in ids and cat in dataset["categories"] and PROTOCOL["confidence"] <= score <= 1,
                    "Identitas atau skor prediksi tersimpan salah.")
            image = by_image[image_id]
            h, w = image["height"], image["width"]
            x, y, bw, bh = record["bbox"]
            require(np.isfinite([x, y, bw, bh]).all() and min(x, y) >= 0 and min(bw, bh) > 0
                    and x + bw <= w + 1e-6 and y + bh <= h + 1e-6, "Koordinat bbox tersimpan tidak valid.")
            if kind == "segm":
                rle = record["segmentation"]
                require(rle["size"] == [h, w] and isinstance(rle["counts"], str), "COCO RLE tersimpan tidak valid.")
                require(coco_mask.area(dict(rle, counts=rle["counts"].encode("ascii"))) > 0, "RLE kosong.")
            counts[image_id] += 1
        require(max(counts.values(), default=0) <= PROTOCOL["max_predictions"], "Jumlah prediksi melebihi batas.")
    return predictions


def stage_directory(run, stage, pair):
    return run / ("smoke" if stage == "smoke" else "results") / "combinations" / pair


def evaluation_pairs(lock):
    return {f"{key}__{domain}": (key, domain) for key in lock["models"] for domain in ("original", "personal")}


def pair_config(key, domain, stage, datasets, lock, environment):
    spec = lock["models"][key]
    dataset, shared = paired_inputs(datasets, spec, domain)
    hard = datasets[f"{domain}_hard3"]
    ids = smoke_ids(dataset) if stage == "smoke" else sorted(dataset["paths"])
    return {"schema": "cpmk3-test-v1", "pipeline_sha256": RECOVERY_PIPELINE_SHA256, "stage": stage, "model_key": key,
            "family": spec["family"], "model_scope": spec["scope"], "dataset_domain": domain,
            "checkpoint_sha256": spec["sha256"], "ground_truth_sha256": dataset["effective_gt_sha256"],
            "source_ground_truth_sha256": dataset["source_gt_sha256"],
            "shared_hard3_gt_sha256": hard["effective_gt_sha256"], "shared_represented": sorted(shared),
            "image_set_sha256": dataset["image_set_sha256"], "image_ids": ids,
            "protocol": PROTOCOL, "environment": environment}


class ResumeSource:
    """Sumber hasil lama dari folder mount atau ZIP keluaran notebook."""

    def __init__(self, path):
        self.path = Path(path)
        self.archive = self.path.is_file()
        self.candidates = []
        require(self.path.exists(), f"Input resume tidak ditemukan: {path}")
        if self.archive:
            with zipfile.ZipFile(self.path) as z:
                for name in z.namelist():
                    if PurePosixPath(name).name == STATUS_FILE and z.namelist().count(name) == 1:
                        record = json.loads(z.read(name))
                        self.candidates.append((safe_member(name).parent, record))
        else:
            for path in sorted(self.path.rglob(STATUS_FILE)):
                self.candidates.append((path.parent, read_json(path)))
        require(self.candidates, f"Manifest resume tidak ditemukan: {path}")

    def copy(self, root, relative, dest, expected):
        if self.archive:
            name = (root / relative).as_posix()
            with zipfile.ZipFile(self.path) as z:
                require(z.namelist().count(name) == 1, f"Anggota ZIP resume duplikat atau hilang: {name}")
                with z.open(name) as source, dest.open("wb") as stream:
                    shutil.copyfileobj(source, stream)
        else:
            shutil.copyfile(root / relative, dest)
        require(sha256(dest) == expected, f"Checksum artefak resume gagal: {relative}")


def recover_pair(sources, run, stage, pair, config, dataset):
    destination = stage_directory(run, stage, pair)
    for source in sources:
        for root, manifest in source.candidates:
            old = manifest.get("pairs", {}).get(stage, {}).get(pair, {})
            if old.get("status") != "complete" or old.get("fingerprint") != fingerprint(config):
                continue
            tmp = destination.with_name(destination.name + ".resume-" + uuid.uuid4().hex)
            tmp.mkdir(parents=True)
            try:
                hashes = old["output_sha256"]
                require(set(hashes) == COMBINATION_FILES, "Daftar artefak resume berbeda.")
                for name, expected in hashes.items():
                    relative = PurePosixPath(destination.relative_to(run).as_posix()) / name
                    source.copy(root, relative, tmp / name, expected)
                require(read_json(tmp / "pair_config.json") == config, "Konfigurasi file hasil lama berbeda.")
                label = "NON-FINAL" if stage == "smoke" else "FINAL_TEST"
                require(read_json(tmp / "metrics.json")["label"] == label, "Label metrik resume berbeda.")
                check_prediction_files(tmp, dataset, config["image_ids"])
                tmp.replace(destination)
                return dict(old, reused_from=str(source.path))
            except (OSError, ValueError, RuntimeError, KeyError, zipfile.BadZipFile) as exc:
                print(f"Hasil lama tidak dipakai untuk {stage}/{pair}: {exc}", flush=True)
                shutil.rmtree(tmp)
    return None


def run_pair(key, domain, stage, models, datasets, lock, hard_map, device, folder, config, diagnostic):
    dataset, _ = paired_inputs(datasets, lock["models"][key], domain)
    checkpoint, _ = models[key]
    ids = config["image_ids"]
    folder.mkdir(parents=True, exist_ok=False)
    write_json(folder / "pair_config.json", config)
    total_start = time.perf_counter()
    require(sha256(checkpoint) == config["checkpoint_sha256"], f"Checkpoint berubah sebelum inferensi: {key}")
    diagnostic["stage"] = "load_model"
    torch.cuda.reset_peak_memory_stats(device)
    loading = time.perf_counter()
    model, _ = load_model(lock["models"][key], checkpoint, device, lock["class_order"])
    sync_device(device)
    load_seconds = time.perf_counter() - loading
    try:
        predictions, timing = infer_recovery(model, lock["models"][key], dataset, ids, device, diagnostic)
    finally:
        del model
        release_model(device)
    diagnostic["stage"] = "serialize_predictions"
    serialize = time.perf_counter()
    for kind, rows in predictions.items():
        json_gzip(folder / f"predictions_{kind}.json.gz", rows)
    del predictions
    verified_predictions = check_prediction_files(folder, dataset, ids)
    after_io = time.perf_counter()
    diagnostic["stage"] = "coco_evaluation"
    metrics = score_pair(datasets, lock["models"][key], domain, verified_predictions, ids, hard_map, stage == "smoke")
    after_metrics = time.perf_counter()
    require(metrics["aggregate"][0]["images"] == len(ids), "Hasil metrik berbeda dari subset aktual.")
    timing.update(model_load_seconds=load_seconds, prediction_io_validation_seconds=after_io - serialize,
                  coco_evaluation_seconds=after_metrics - after_io,
                  total_pair_seconds=after_metrics - total_start)
    write_json(folder / "metrics.json", metrics)
    write_json(folder / "timing.json", timing)
    require({p.name for p in folder.iterdir()} == COMBINATION_FILES, "Output kombinasi tidak lengkap.")
    return {name: sha256(folder / name) for name in sorted(COMBINATION_FILES)}


def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def save_status(run, manifest):
    manifest["updated_utc"] = now()
    write_json(run / STATUS_FILE, manifest)


def execute_stage(run, manifest, stage, pairs, wanted, sources, models, datasets, lock, hard_map, device, environment):
    statuses = manifest["pairs"][stage]
    for pair, (key, domain) in pairs.items():
        if pair not in RECOVERY_REQUIRED or statuses.get(pair, {}).get("status") == "complete":
            continue
        spec = lock["models"][key]
        dataset, _ = paired_inputs(datasets, spec, domain)
        config = pair_config(key, domain, stage, datasets, lock, environment)
        resumed = recover_pair(sources, run, stage, pair, config, dataset)
        if resumed:
            resumed["implementation"] = "recovery_memory_efficient"
            statuses[pair] = resumed
            save_status(run, manifest)
            print(f"{stage} / {pair}: complete (resume)", flush=True)
            continue
        if pair not in wanted:
            statuses[pair] = dict(status="skipped", fingerprint=fingerprint(config))
            save_status(run, manifest)
            continue
        record = {"status": "running", "fingerprint": fingerprint(config), "started_utc": now(),
                  "model_key": key, "dataset_domain": domain,
                  "checkpoint_sha256": config["checkpoint_sha256"],
                  "ground_truth_sha256": config["ground_truth_sha256"],
                  "source_ground_truth_sha256": config["source_ground_truth_sha256"],
                  "image_set_sha256": config["image_set_sha256"], "protocol_sha256": fingerprint(PROTOCOL)}
        diagnostic = {"model_dataset_key": pair, "stage": "prepare_pair"}
        statuses[pair] = record
        save_status(run, manifest)
        print(f"{stage} / {pair}: running", flush=True)
        try:
            hashes = run_pair(key, domain, stage, models, datasets, lock, hard_map, device,
                              stage_directory(run, stage, pair), config, diagnostic)
            diagnostic["stage"] = "complete"
            record.update(status="complete", output_sha256=hashes, implementation="recovery_memory_efficient")
        except Exception as exc:
            trace = traceback.format_exc()
            error = run / "errors" / f"{stage}__{pair}.txt"
            error.parent.mkdir(parents=True, exist_ok=True)
            error.write_text(trace, encoding="utf-8")
            reference = error.relative_to(run).as_posix()
            diagnostic.update(exception_type=type(exc).__name__, exception_message=str(exc), traceback=reference)
            record.update(status="failed", error=str(exc), traceback=reference)
            traceback.clear_frames(exc.__traceback__)
        recovery_diagnostic(run, stage, pair, diagnostic, device)
        release_model(device)
        record["finished_utc"] = now()
        save_status(run, manifest)
        print(f"{stage} / {pair}: {record['status']}", flush=True)


def stage_complete(manifest, stage, pairs):
    records = manifest["pairs"][stage]
    return set(records) == set(pairs) and all(r["status"] == "complete" for r in records.values())

ENV_COMPATIBILITY_KEYS = {
    "python", "torch", "torchvision", "ultralytics", "torch_cuda", "cudnn",
    "gpu", "device", "numpy", "opencv", "pillow", "pycocotools",
}


def read_resume_member(source, root, filename):
    relative = safe_member(filename)
    if source.archive:
        with zipfile.ZipFile(source.path) as archive:
            member = (root / relative).as_posix()
            require(archive.namelist().count(member) == 1, f"Anggota resume duplikat atau hilang: {member}")
            return archive.read(member)
    return (root / relative).read_bytes()


def find_run01(sources, datasets, lock, environment):
    mismatches = []
    for source in sources:
        for root, legacy in source.candidates:
            manifest_bytes = read_resume_member(source, root, STATUS_FILE)
            if hashlib.sha256(manifest_bytes).hexdigest() != RUN01_MANIFEST_SHA256:
                continue
            try:
                require(legacy == json.loads(manifest_bytes), "Manifest Run01 berubah saat dibaca.")
                require(legacy["mode"] == "FULL_EVALUATION" and legacy["run_status"] == "PARTIAL",
                        "Status manifest Run01 tidak sesuai.")
                output_bytes = read_resume_member(source, root, "artifact_hashes.json")
                require(hashlib.sha256(output_bytes).hexdigest() == RUN01_INVENTORY_SHA256,
                        "Checksum inventaris Run01 berbeda.")
                inventory = json.loads(output_bytes)

                def audited_json(filename):
                    data = read_resume_member(source, root, filename)
                    require(hashlib.sha256(data).hexdigest() == inventory[filename],
                            f"Metadata Run01 berubah: {filename}")
                    return json.loads(data)

                old_env = audited_json("environment.json")
                drift = {key: [old_env.get(key), environment.get(key)] for key in old_env.keys() | environment.keys()
                         if old_env.get(key) != environment.get(key)}
                incompatible = {key: value for key, value in drift.items() if key in ENV_COMPATIBILITY_KEYS}
                require(not incompatible, f"Environment inferensi Run01 tidak kompatibel: {incompatible}")
                old_protocol = audited_json("evaluation_protocol.json")
                require(old_protocol["pipeline_sha256"] == RUN01_PIPELINE_SHA256 and
                        old_protocol["protocol"] == PROTOCOL, "Pipeline atau protokol Run01 berbeda.")
                old_datasets = audited_json("dataset_manifest.json")["datasets"]
                for key, dataset in datasets.items():
                    for field in ("effective_gt_sha256", "source_gt_sha256", "image_set_sha256", "summary"):
                        require(old_datasets[key][field] == dataset[field], f"Identitas {key}/{field} berubah.")
                old_checkpoints = audited_json("checkpoint_manifest.json")
                for key, spec in lock["models"].items():
                    require(old_checkpoints[key]["sha256"] == spec["sha256"] and
                            old_checkpoints[key]["loading"]["classes"] == lock["class_order"][spec["scope"]],
                            f"Checkpoint atau kategori {key} berbeda.")
                # Isolasi hanya kandidat dengan byte manifest persis sama.
                pinned = ResumeSource.__new__(ResumeSource)
                pinned.path, pinned.archive, pinned.candidates = source.path, source.archive, [(root, legacy)]
                return pinned, legacy, inventory, old_env, drift
            except (OSError, ValueError, KeyError, zipfile.BadZipFile) as exc:
                mismatches.append(str(exc))
    raise ValueError("Tidak ada input Run01 yang cocok dengan manifest audit. " + "; ".join(mismatches))


def import_run01(sources, run, manifest, pairs, datasets, lock, environment):
    verified = {}
    report = {"source_run_id": RUN01_ID, "original_pipeline_sha256": RUN01_PIPELINE_SHA256,
              "recovery_pipeline_sha256": RECOVERY_PIPELINE_SHA256, "manifest_sha256": RUN01_MANIFEST_SHA256,
              "passed": False, "pairs": verified}
    try:
        source, legacy, inventory, old_env, drift = find_run01(sources, datasets, lock, environment)
        report.update(source=str(source.path), non_inference_environment_differences=drift)
        for pair, audited_fingerprint in RUN01_ALLOWED.items():
            key, domain = pairs[pair]
            dataset, _ = paired_inputs(datasets, lock["models"][key], domain)
            previous = legacy["pairs"]["primary"][pair]
            old_config = dict(pair_config(key, domain, "primary", datasets, lock, old_env),
                              pipeline_sha256=RUN01_PIPELINE_SHA256)
            record = {"status": "rejected"}
            try:
                require(previous["status"] == "complete" and previous["fingerprint"] == audited_fingerprint ==
                        fingerprint(old_config), f"Fingerprint legacy berbeda: {pair}")
                for name, digest in previous["output_sha256"].items():
                    require(inventory[f"results/combinations/{pair}/{name}"] == digest,
                            f"Checksum inventaris berbeda: {pair}/{name}")
                imported = recover_pair([source], run, "primary", pair, old_config, dataset)
                require(imported is not None, f"Artefak Run01 tidak lolos verifikasi: {pair}")
                manifest["pairs"]["primary"][pair] = dict(
                    imported, status="compatibility_pending", implementation="run01_legacy",
                    lineage={"source_run_id": RUN01_ID, "manifest_sha256": RUN01_MANIFEST_SHA256,
                             "source_pair_fingerprint": audited_fingerprint,
                             "original_pipeline_sha256": RUN01_PIPELINE_SHA256,
                             "recovery_pipeline_sha256": RECOVERY_PIPELINE_SHA256,
                             "compatibility_evidence": ["pinned_run01_manifest", "exact_pair_config",
                                                        "checkpoint_ground_truth_roster_and_protocol",
                                                        "environment_and_prediction_checksums"]})
                record = {"status": "import_verified_pending_gpu", "original_pair_fingerprint": audited_fingerprint}
            except (OSError, ValueError, RuntimeError, KeyError, zipfile.BadZipFile) as exc:
                record["error"] = str(exc)
            verified[pair] = record
        report["passed"] = all(r["status"] == "import_verified_pending_gpu" for r in verified.values())
    except Exception as exc:
        report["error"] = str(exc)
        report["traceback"] = traceback.format_exc()
    write_json(run / "legacy_import_report.json", report)
    save_status(run, manifest)
    if report["passed"]:
        print("LEGACY IMPORT PASS 7/7:", ", ".join(verified), flush=True)
    else:
        print("LEGACY IMPORT BLOCKED. Periksa legacy_import_report.json.", flush=True)
    return report


def promote_run01(manifest, gate_sha256):
    for pair in RUN01_ALLOWED:
        row = manifest["pairs"]["primary"][pair]
        require(row["status"] == "compatibility_pending", f"Status impor tidak siap untuk {pair}")
        row["status"] = "complete"
        row["lineage"]["gpu_equivalence_report_sha256"] = gate_sha256


def recovery_diagnostic(run, stage, pair, runtime, device):
    runtime.update(**memory_snapshot(device))
    write_json(run / "diagnostics" / f"{stage}__{pair}.json", runtime)


def consolidate_recovered_metrics(run, manifest, datasets, pairs, hard_map):
    try:
        for pair, (key, domain) in pairs.items():
            record = manifest["pairs"]["primary"][pair]
            folder = stage_directory(run, "primary", pair)
            for filename, digest in record["output_sha256"].items():
                require(sha256(folder / filename) == digest, f"Artefak berubah sebelum konsolidasi: {pair}/{filename}")
            config = read_json(folder / "pair_config.json")
            dataset, _ = paired_inputs(datasets, LOCK["models"][key], domain)
            ids = config["image_ids"]
            predictions = check_prediction_files(folder, dataset, ids)
            scored = score_pair(datasets, LOCK["models"][key], domain, predictions, ids, hard_map, False)
            require(scored == read_json(folder / "metrics.json"),
                    f"Metrik dari prediksi terverifikasi berbeda dari metrik tersimpan: {pair}")
            write_json(run / "consolidated_metrics" / f"{pair}.json", scored)
            del predictions
    except Exception:
        manifest["consolidation"] = {"passed": False, "traceback": traceback.format_exc()}
        save_status(run, manifest)
        print("CONSOLIDATION BLOCKED. Periksa evaluation_status_manifest.json.", flush=True)
        return False
    manifest["consolidation"] = {"passed": True, "pairs_rescored": len(pairs)}
    save_status(run, manifest)
    return True

### 09 Tables and export

In [11]:
def export_tables(run, manifest, pairs, datasets, lock):
    directory = run / "tables"
    directory.mkdir(exist_ok=True)
    aggregate, per_class, timing, support = [], [], [], []
    for name, dataset in datasets.items():
        crowd_free = set(dataset["summary"]["represented_categories"])
        for category, label in sorted(dataset["categories"].items()):
            support.append({"dataset": name, "category_id": category, "class": label,
                            "gt_instances": dataset["summary"]["class_counts"].get(label, 0),
                            "has_evaluable_gt": category in crowd_free})
    pd.DataFrame(support).to_csv(directory / "ground_truth_support.csv", index=False)
    for pair, record in manifest["pairs"]["primary"].items():
        if record["status"] != "complete":
            continue
        model, domain = pairs[pair]
        folder = stage_directory(run, "primary", pair)
        for name, expected in record["output_sha256"].items():
            require(sha256(folder / name) == expected, f"Artefak hasil berubah: {folder / name}")
        reconciled = run / "consolidated_metrics" / f"{pair}.json"
        metrics = read_json(reconciled if manifest.get("consolidation", {}).get("passed") else folder / "metrics.json")
        require(metrics["label"] == "FINAL_TEST" and {r["evaluation_scope"] for r in metrics["aggregate"]}
                == {"primary", "shared_represented", "shared_hard3"}, f"Scope hasil tidak lengkap pada {pair}.")
        fields = {"model": model, "family": lock["models"][model]["family"],
                  "training_scope": lock["models"][model]["scope"], "domain": domain}
        aggregate.extend(dict(fields, **r) for r in metrics["aggregate"])
        per_class.extend(dict(fields, **r) for r in metrics["per_class"])
        row_timing = read_json(folder / "timing.json")
        row_timing["implementation"] = record["implementation"]
        row_timing["source_run_id"] = record.get("lineage", {}).get("source_run_id")
        timing.append(dict(fields, **row_timing))
    if not aggregate:
        return
    agg, per = pd.DataFrame(aggregate), pd.DataFrame(per_class)
    agg.to_csv(directory / "aggregate_metrics.csv", index=False)
    per.to_csv(directory / "per_class_metrics.csv", index=False)
    pd.DataFrame(timing).to_csv(directory / "timings.csv", index=False)
    agg.loc[agg.evaluation_scope.eq("primary")].to_csv(directory / "primary_metrics.csv", index=False)
    shared = agg.loc[agg.evaluation_scope.eq("shared_represented")]
    shared.to_csv(directory / "shared_represented_metrics.csv", index=False)

    def side_by_side(frame, group, condition, left, right):
        fields = group + ["mask_ap", "mask_ar100", "gt_instances"]
        first = frame.loc[frame[condition].eq(left), fields]
        second = frame.loc[frame[condition].eq(right), fields]
        return first.merge(second, on=group, how="outer", suffixes=(f"_{left}", f"_{right}"), validate="one_to_one")

    domain = side_by_side(agg, ["model", "evaluation_scope"], "domain", "original", "personal")
    domain["mask_ap_delta_pp_personal_minus_original"] = 100 * (domain.mask_ap_personal - domain.mask_ap_original)
    domain.to_csv(directory / "original_vs_personal.csv", index=False)
    for frame, groups, filename in ((agg, ["family", "domain"], "shared_hard3_comparison.csv"),
                                    (per, ["family", "domain", "class"], "shared_hard3_per_class.csv")):
        scope = frame.loc[frame.evaluation_scope.eq("shared_hard3")]
        comparison = side_by_side(scope, groups, "training_scope", "all", "hard3")
        comparison["mask_ap_delta_pp_hard3_minus_all"] = 100 * (comparison.mask_ap_hard3 - comparison.mask_ap_all)
        comparison.to_csv(directory / filename, index=False)
    chart_dir = run / "figures"
    chart_dir.mkdir(exist_ok=True)
    for scope in ("primary", "shared_hard3", "shared_represented"):
        values = agg.loc[agg.evaluation_scope.eq(scope)].pivot(index="model", columns="domain", values="mask_ap")
        if values.dropna(how="all").empty:
            continue
        ax = (values.astype(float) * 100).plot.bar(figsize=(10, 4), color=["#355C8F", "#D7813B"],
                                                  ylabel="Mask AP@[.50:.95] (%)", title=scope)
        ax.set_ylim(0, 100)
        ax.figure.tight_layout()
        ax.figure.savefig(chart_dir / f"{scope}.png", dpi=150)
        plt.close(ax.figure)


def export_run(run, manifest, mode, pairs):
    completed = (mode == "FULL_EVALUATION" and manifest.get("gpu_equivalence", {}).get("passed", False)
                 and manifest.get("consolidation", {}).get("passed", False)
                 and stage_complete(manifest, "primary", pairs))
    if completed:
        required = {"primary_metrics.csv", "per_class_metrics.csv", "shared_hard3_comparison.csv",
                    "shared_hard3_per_class.csv", "shared_represented_metrics.csv", "original_vs_personal.csv",
                    "ground_truth_support.csv", "timings.csv"}
        require(all((run / "tables" / name).is_file() for name in required), "Tabel hasil lengkap belum tersedia.")
        require(len(pd.read_csv(run / "tables" / "primary_metrics.csv")) == len(pairs), "Jumlah metrik primer salah.")
    state = ("COMPLETE" if completed else "PREFLIGHT" if mode == "PREFLIGHT" else
             "STRESS_NON_FINAL" if mode == "STRESS_TEST" and manifest.get("gpu_equivalence", {}).get("passed") else
             "PARTIAL")
    manifest["run_status"] = state
    save_status(run, manifest)
    write_json(run / "execution_summary.json", {
        "mode": mode, "status": state, "primary_expected": len(pairs), "primary_complete": sum(
            r["status"] == "complete" for r in manifest["pairs"]["primary"].values()),
        "smoke_complete": sum(r["status"] == "complete" for r in manifest["pairs"]["smoke"].values()),
        "statuses": {stage: dict(Counter(r["status"] for r in records.values()))
                     for stage, records in manifest["pairs"].items()},
    })
    exclude_hash = run / "artifact_hashes.json"
    files = sorted(p for p in run.rglob("*") if p.is_file() and not p.name.endswith(".tmp") and p != exclude_hash)
    write_json(exclude_hash, {p.relative_to(run).as_posix(): sha256(p) for p in files})
    suffix = {"COMPLETE": "Complete", "PARTIAL": "Partial", "STRESS_NON_FINAL": "Stress_NON_FINAL",
              "PREFLIGHT": "Preflight"}[state]
    # Satu direktori unik per run menghindari overwrite ZIP historis.
    output = run.parent / f"CPMK3_Unified_Test_Evaluation_Recovery_{suffix}.zip"
    with zipfile.ZipFile(output, "x", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
        for path in files + [exclude_hash]:
            compress = zipfile.ZIP_STORED if path.name.endswith((".gz", ".png")) else zipfile.ZIP_DEFLATED
            archive.write(path, path.relative_to(run).as_posix(), compress_type=compress)
    print(f"Status {state}. Unduh {output}", flush=True)
    return output

### 10 Run preflight

In [12]:
require(MODE in {"PREFLIGHT", "STRESS_TEST", "FULL_EVALUATION"}, "Mode tidak dikenal.")
require(torch.cuda.is_available(), "Aktifkan GPU Kaggle sebelum preflight.")
require(ultralytics.__version__ == "8.4.216" and version("pycocotools") == "2.0.10",
        "Versi Ultralytics/pycocotools berbeda. Jalankan cell instalasi pinned lalu mulai ulang notebook.")
DEVICE = "cuda:0"
configure_runtime(DEVICE)
ENVIRONMENT = runtime_environment(DEVICE)
RUN_PARENT = Path(WORK_ROOT) / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
RUN = RUN_PARENT / "run"
RUN.mkdir(parents=True, exist_ok=False)
PAIRS = evaluation_pairs(LOCK)
require(PIPELINE_SHA256 == RUN01_PIPELINE_SHA256, "Input lock bukan dari notebook Run01.")
require(len(PAIRS) == 12, "Matriks evaluasi harus memuat 12 pasangan.")
require(RESUME_INPUTS == [] and SELECT_PAIRS == [],
        "Fresh evaluation memakai RESUME_INPUTS = [] dan SELECT_PAIRS = [].")
RECOVERY_REQUIRED = frozenset(PAIRS)
RECOVERY_PIPELINE_SHA256 = "61b9f9ac63f7c354a410fa0bf2298f0c19f016eecb170b7d49ce6dbd258cd35d"
MANIFEST = {"schema": "cpmk3-test-v1", "mode": MODE, "started_utc": now(), "run_status": "preflight", "result_origin": "fresh",
            "pairs": {"smoke": {}, "primary": {}}}
save_status(RUN, MANIFEST)
write_json(RUN / "environment.json", ENVIRONMENT)
write_json(RUN / "evaluation_protocol.json", {"protocol": PROTOCOL, "pipeline_sha256": RECOVERY_PIPELINE_SHA256,
                                             "mode": MODE, "result_origin": "fresh",
                                             "planned_primary_pairs": list(PAIRS)})
try:
    INDEX = InputIndex(INPUT_ROOTS, Path(WORK_ROOT) / "input_cache")
    DATASETS, HARD_MAP, SOURCES = prepare_datasets(INDEX, LOCK)
    MODELS = resolve_models(INDEX, LOCK, CHECKPOINT_OVERRIDES)
    CHECKPOINT_REPORT = check_models(MODELS, LOCK, DEVICE)
    write_json(RUN / "dataset_manifest.json", {
        "datasets": {key: {k: ds[k] for k in ("summary", "effective_gt_sha256", "source_gt_sha256",
                                             "source_gt", "image_set_sha256", "categories")}
                     for key, ds in DATASETS.items()},
        "references": {key: dict(LOCK["references"][key], located_at=value[1]) for key, value in SOURCES.items()},
        "original_hard3_translation": HARD_MAP,
        "orientation": "pixel asli; ukuran dan EXIF tervalidasi; tanpa rotasi",
    })
    write_json(RUN / "checkpoint_manifest.json", CHECKPOINT_REPORT)
    write_json(RUN / "image_category_mapping.json", {
        "images": {domain: DATASETS[f"{domain}_all"]["image_map"] for domain in ("original", "personal")},
        "categories_all_and_hard3": SOURCES["mapping"][0],
    })
    write_json(RUN / "preflight_report.json", {"passed": True, "pairs": list(PAIRS), "device": DEVICE,
                                            "smoke_ids": {key: smoke_ids(ds) for key, ds in DATASETS.items()}})
    print("PREFLIGHT PASS - 6 checkpoint, 4 view ground truth, 12 kombinasi.", flush=True)
except Exception:
    write_json(RUN / "preflight_report.json", {"passed": False, "traceback": traceback.format_exc()})
    MANIFEST["run_status"] = "blocked_preflight"
    save_status(RUN, MANIFEST)
    raise

Checkpoint valid: yolo11n_all
Checkpoint valid: yolo11n_hard3
Checkpoint valid: yolov8n_all
Checkpoint valid: yolov8n_hard3
Checkpoint valid: maskrcnn_all
Checkpoint valid: maskrcnn_hard3
PREFLIGHT PASS - 6 checkpoint, 4 view ground truth, 12 kombinasi.


### 11 Run smoke gate

In [13]:
from ultralytics.engine.predictor import BasePredictor
from ultralytics.data.build import load_inference_source
from types import SimpleNamespace


def verify_letterbox(paths):
    reference = load_inference_source(
        [str(p) for p in paths], batch=1
    )

    require(
        len({im.shape for im in reference.im0}) > 1,
        "Fixture letterbox harus memakai shape campuran."
    )

    context = SimpleNamespace(
        imgsz=(640, 640),
        args=SimpleNamespace(rect=True),
        scale_fill=False,
        model=SimpleNamespace(format="pt", stride=32),
    )

    old = BasePredictor.pre_transform(context, reference.im0)
    context.args.rect = False

    comparisons = []

    for path, old_image in zip(paths, old, strict=True):
        one = load_inference_source([str(path)], batch=1)
        new, = BasePredictor.pre_transform(context, one.im0)

        require(
            one.bs == 1
            and new.shape == (640, 640, 3)
            and np.array_equal(old_image, new),
            f"Piksel letterbox recovery berbeda: {path.name}"
        )

        comparisons.append({
            "filename": path.name,
            "shape": list(new.shape),
            "pixel_equal": True,
        })

    return comparisons


def ranking_and_ties_equal(left, right):
    if len(left) < 2:
        return True

    order = np.argsort(-left, kind="stable")
    other_order = np.argsort(-right, kind="stable")

    return bool(
        np.array_equal(order, other_order)
        and np.array_equal(
            np.diff(left[order]) == 0,
            np.diff(right[other_order]) == 0,
        )
    )


def equivalent_predictions(previous, current, dataset):
    results = {}

    for kind in ("bbox", "segm"):
        old, new = previous[kind], current[kind]

        checks = dict(
            old_rows=len(old),
            new_rows=len(new),
            max_score_abs_delta=0.0,
            max_bbox_abs_delta=0.0,
            categories_and_order=False,
            rle_mismatches=0,
            score_ranking_unchanged=False,
            pixel_edges_unchanged=False,
            bbox_area_bins_unchanged=False,
            bbox_iou_decisions_unchanged=False,
            passed=False,
        )

        results[kind] = checks

        if len(old) != len(new) or any(
            (a["image_id"], a["category_id"])
            != (b["image_id"], b["category_id"])
            for a, b in zip(old, new)
        ):
            continue

        checks["categories_and_order"] = True

        checks["rle_mismatches"] = sum(
            a.get("segmentation") != b.get("segmentation")
            for a, b in zip(old, new)
        )

        scores_a = np.array(
            [a["score"] for a in old], dtype=float
        )

        scores_b = np.array(
            [b["score"] for b in new], dtype=float
        )

        boxes_a = np.array(
            [a["bbox"] for a in old], dtype=float
        ).reshape(-1, 4)

        boxes_b = np.array(
            [b["bbox"] for b in new], dtype=float
        ).reshape(-1, 4)

        checks["max_score_abs_delta"] = float(
            np.max(np.abs(scores_a - scores_b), initial=0)
        )

        checks["max_bbox_abs_delta"] = float(
            np.max(np.abs(boxes_a - boxes_b), initial=0)
        )

        categories = np.array(
            [a["category_id"] for a in old]
        )

        checks["score_ranking_unchanged"] = all(
            ranking_and_ties_equal(
                scores_a[categories == cat],
                scores_b[categories == cat],
            )
            for cat in np.unique(categories)
        )

        checks["pixel_edges_unchanged"] = bool(
            np.array_equal(
                np.floor(boxes_a[:, :2]),
                np.floor(boxes_b[:, :2]),
            )
            and np.array_equal(
                np.ceil(boxes_a[:, :2] + boxes_a[:, 2:]),
                np.ceil(boxes_b[:, :2] + boxes_b[:, 2:]),
            )
        )

        area_a = boxes_a[:, 2] * boxes_a[:, 3]
        area_b = boxes_b[:, 2] * boxes_b[:, 3]

        checks["bbox_area_bins_unchanged"] = all(
            np.array_equal(
                (area_a >= lo) & (area_a <= hi),
                (area_b >= lo) & (area_b <= hi),
            )
            for lo, hi in PROTOCOL["coco"]["area"]
        )

        checks["bbox_iou_decisions_unchanged"] = True

        if checks["max_bbox_abs_delta"]:
            targets = {}

            for row in dataset["gt"]["annotations"]:
                targets.setdefault(
                    (row["image_id"], row["category_id"]),
                    [],
                ).append(row)

            thresholds = np.array(PROTOCOL["coco"]["iou"])

            for i, row in enumerate(old):
                gt = targets.get(
                    (row["image_id"], row["category_id"]),
                    [],
                )

                if not gt:
                    continue

                target_boxes = [g["bbox"] for g in gt]
                crowds = [g["iscrowd"] for g in gt]

                a = coco_mask.iou(
                    [boxes_a[i].tolist()],
                    target_boxes,
                    crowds,
                )[0]

                b = coco_mask.iou(
                    [boxes_b[i].tolist()],
                    target_boxes,
                    crowds,
                )[0]

                if not (
                    np.array_equal(
                        a[:, None] >= thresholds,
                        b[:, None] >= thresholds,
                    )
                    and ranking_and_ties_equal(a, b)
                ):
                    checks["bbox_iou_decisions_unchanged"] = False
                    break

        checks["passed"] = (
            checks["rle_mismatches"] == 0
            and all(
                checks[k]
                for k in (
                    "categories_and_order",
                    "score_ranking_unchanged",
                    "pixel_edges_unchanged",
                    "bbox_area_bins_unchanged",
                    "bbox_iou_decisions_unchanged",
                )
            )
        )

    return results


def decode_verification_rle(rle):
    counts = rle["counts"]

    if isinstance(counts, str):
        counts = counts.encode("ascii")

    return coco_mask.decode({
        "size": rle["size"],
        "counts": counts,
    }).astype(bool)


def mask_difference_details(previous, current):
    old = previous["segm"]
    new = current["segm"]

    if len(old) != len(new):
        return {
            "comparable": False,
            "reason": "row_count",
        }

    if any(
        (a["image_id"], a["category_id"])
        != (b["image_id"], b["category_id"])
        for a, b in zip(old, new)
    ):
        return {
            "comparable": False,
            "reason": "category_or_order",
        }

    details = []

    for i, (a, b) in enumerate(zip(old, new)):
        if a["segmentation"] == b["segmentation"]:
            continue

        ma = decode_verification_rle(a["segmentation"])
        mb = decode_verification_rle(b["segmentation"])

        require(
            ma.shape == mb.shape,
            "Different mask dimensions in diagnostic comparison."
        )

        intersection = int(np.count_nonzero(ma & mb))
        union = int(np.count_nonzero(ma | mb))

        details.append({
            "prediction_index": i,
            "image_id": a["image_id"],
            "category_id": a["category_id"],
            "old_area": int(np.count_nonzero(ma)),
            "new_area": int(np.count_nonzero(mb)),
            "different_pixels": int(np.count_nonzero(ma ^ mb)),
            "mask_iou": intersection / union if union else 1.0,
        })

    return {
        "comparable": True,
        "matching_method": "positional",
        "mismatch_count": len(details),
        "details": details,
    }


def gpu_mask_stress(device, shape):
    from ultralytics.engine.results import Masks

    roi = torch.rand(
        (100, 1, 28, 28),
        generator=torch.Generator().manual_seed(42),
    ).to(device)

    roi[0].zero_()

    height, width = shape
    rng = np.random.default_rng(42)

    origins = rng.integers(
        0, [width - 80, height - 80], (100, 2)
    )

    boxes = np.column_stack(
        (origins, origins + 48)
    ).astype(np.float32)

    boxes[:3] = [
        [-4, 0, 2, 1],
        [0, 0, 1, 2],
        [width - 20, height - 20, width + 5, height + 2],
    ]

    gpu_boxes = torch.from_numpy(boxes).to(device)

    if device != "cpu":
        torch.cuda.reset_peak_memory_stats(device)

    reference = paste_masks_in_image(
        roi[:4], gpu_boxes[:4], shape
    )

    for i, canvas in enumerate(
        rcnn_mask_canvases(roi[:4], gpu_boxes[:4], shape)
    ):
        expected = (
            reference[i, 0] >= PROTOCOL["mask_threshold"]
        ).cpu().numpy()

        require(
            np.array_equal(canvas, expected)
            and encode_mask(canvas) == encode_mask(expected),
            f"Paste mask atau RLE Mask R-CNN berbeda pada kontrol {i}"
        )

    del reference

    count = 0

    for canvas in rcnn_mask_canvases(
        roi, gpu_boxes, shape
    ):
        rle = encode_mask(canvas)

        require(
            rle["size"] == list(shape),
            "RLE stress salah ukuran."
        )

        count += 1

    del canvas

    crop_boxes = np.column_stack((
        origins,
        origins + 48,
        np.arange(100),
        np.zeros(100),
    )).astype(np.int64)

    crop_boxes[0, :4] = [
        width - 48,
        height - 48,
        width,
        height,
    ]

    crops = [
        torch.ones(
            (1, 48, 48),
            dtype=torch.uint8,
            device=device,
        )
        for _ in range(100)
    ]

    crops[1].zero_()

    local_masks = Masks(
        (
            torch.from_numpy(crop_boxes).to(device),
            crops,
            shape,
        ),
        shape,
    )

    yolo_count = 0

    for i, canvas in enumerate(
        yolo_mask_crops(local_masks, shape, 100)
    ):
        rle = encode_mask(canvas)

        area = int(coco_mask.area(
            dict(
                rle,
                counts=rle["counts"].encode("ascii"),
            )
        ))

        require(
            rle["size"] == list(shape)
            and area == (0 if i == 1 else 48**2),
            "Stress crop YOLO gagal."
        )

        yolo_count += 1

    require(
        count == yolo_count == 100,
        "Stress test kehilangan deteksi."
    )

    sync_device(device)

    return {
        "shape": list(shape),
        "maskrcnn_serial_rles": count,
        "yolo_serial_rles": yolo_count,
        "native_torchvision_controls_equal": 4,
        **memory_snapshot(device),
    }


def verify_same_source_masks(
    model,
    spec,
    dataset,
    image_ids,
    device,
    diagnostic,
):
    original_yolo_crops = yolo_mask_crops
    original_rcnn_canvases = rcnn_mask_canvases

    checked = {
        "yolo": 0,
        "maskrcnn": 0,
    }

    def checked_yolo(masks, shape, count):
        canvases = [
            np.asarray(canvas).copy()
            for canvas in original_yolo_crops(
                masks, shape, count
            )
        ]

        require(
            len(canvases) == int(count),
            "YOLO serial mask count mismatch."
        )

        if not canvases:
            return

        native = masks.data.detach().cpu().numpy()

        if native.ndim == 4 and native.shape[1] == 1:
            native = native[:, 0]

        require(
            tuple(native.shape) == (count, *shape),
            "YOLO native mask dimensions mismatch."
        )

        for i, canvas in enumerate(canvases):
            expected = (
                native[i] >= PROTOCOL["mask_threshold"]
            )

            require(
                np.array_equal(canvas, expected)
                and encode_mask(canvas) == encode_mask(expected),
                f"YOLO same-source mask mismatch at index {i}"
            )

            checked["yolo"] += 1

        del native

        yield from canvases

    def checked_rcnn(roi_masks, resized_boxes, shape):
        canvases = [
            np.asarray(canvas).copy()
            for canvas in original_rcnn_canvases(
                roi_masks, resized_boxes, shape
            )
        ]

        require(
            len(canvases) == len(roi_masks),
            "Mask R-CNN serial mask count mismatch."
        )

        if not canvases:
            return

        native = paste_masks_in_image(
            roi_masks,
            resized_boxes,
            shape,
        )

        require(
            tuple(native.shape) == (
                len(roi_masks), 1, *shape
            ),
            "Mask R-CNN native mask dimensions mismatch."
        )

        for i, canvas in enumerate(canvases):
            expected = (
                native[i, 0] >= PROTOCOL["mask_threshold"]
            ).cpu().numpy()

            require(
                np.array_equal(canvas, expected)
                and encode_mask(canvas) == encode_mask(expected),
                f"Mask R-CNN same-source mask mismatch at index {i}"
            )

            checked["maskrcnn"] += 1

        del native

        yield from canvases

    globals()["yolo_mask_crops"] = checked_yolo
    globals()["rcnn_mask_canvases"] = checked_rcnn

    try:
        predictions, timing = infer_recovery(
            model,
            spec,
            dataset,
            image_ids,
            device,
            diagnostic,
        )

    finally:
        globals()["yolo_mask_crops"] = original_yolo_crops
        globals()["rcnn_mask_canvases"] = original_rcnn_canvases

    return predictions, timing, checked


def validate_stress_predictions(
    predictions,
    timing,
    dataset,
    image_ids,
):
    image_shapes = {
        im["id"]: [im["height"], im["width"]]
        for im in dataset["gt"]["images"]
        if im["id"] in image_ids
    }

    valid_categories = set(dataset["categories"])

    require(
        timing["images_processed"] == len(image_ids),
        "Stress inference image coverage mismatch."
    )

    for kind in ("bbox", "segm"):
        counts_by_image = Counter()

        for row in predictions[kind]:
            image_id = row["image_id"]

            require(
                image_id in image_shapes,
                "Invalid prediction image ID."
            )

            require(
                row["category_id"] in valid_categories,
                "Invalid prediction category ID."
            )

            require(
                np.isfinite(row["score"])
                and 0 <= row["score"] <= 1,
                "Invalid prediction score."
            )

            require(
                np.isfinite(row["bbox"]).all(),
                "Invalid prediction bounding box."
            )

            x, y, w, h = row["bbox"]
            height, width = image_shapes[image_id]

            require(
                x >= 0
                and y >= 0
                and w > 0
                and h > 0
                and x + w <= width + 1e-6
                and y + h <= height + 1e-6,
                "Prediction bounding box outside image."
            )

            counts_by_image[image_id] += 1

            if kind == "segm":
                rle = row["segmentation"]

                require(
                    rle["size"] == image_shapes[image_id],
                    "Invalid original-resolution RLE shape."
                )

                counts = rle["counts"]

                if isinstance(counts, str):
                    counts = counts.encode("ascii")

                area = int(coco_mask.area({
                    "size": rle["size"],
                    "counts": counts,
                }))

                require(
                    0 < area <= height * width,
                    "Invalid segmentation mask area."
                )

        require(
            all(
                count <= PROTOCOL["max_predictions"]
                for count in counts_by_image.values()
            ),
            "Prediction count exceeds protocol limit."
        )

    require(
        len(predictions["segm"]) <= len(predictions["bbox"]),
        "Segmentation predictions exceed bbox predictions."
    )


def recovery_stress(
    run,
    datasets,
    models,
    lock,
    device,
    environment,
):
    report = {
        "passed": False,
        "label": "NON-FINAL",
        "workflow": "fresh_full_evaluation",
        "environment": environment,
        "pipeline_sha256": RECOVERY_PIPELINE_SHA256,
        "models": {},
        "acceptance": (
            "Native same-source mask encoding, letterbox, "
            "prediction integrity, and GPU stress must pass. "
            "Run01 batch equivalence is diagnostic only."
        ),
    }

    fixture_diag = None
    diag = None

    try:
        require(
            not RESUME_INPUTS,
            "Fresh evaluation requires RESUME_INPUTS = []."
        )

        original = datasets["original_all"]

        small = sorted(
            (
                im for im in original["gt"]["images"]
                if min(im["height"], im["width"]) >= 224
            ),
            key=lambda im: (
                im["height"] * im["width"],
                im["id"],
            ),
        )

        control = [
            small[0],
            next(
                im for im in small
                if (im["height"], im["width"])
                != (small[0]["height"], small[0]["width"])
            ),
        ]

        control_ids = [im["id"] for im in control]

        report["control_ids_original"] = control_ids

        report["letterbox"] = verify_letterbox([
            original["paths"][i]
            for i in control_ids
        ])

        print("LETTERBOX VERIFICATION PASS", flush=True)

        personal = datasets["personal_all"]

        ranked = sorted(
            personal["gt"]["images"],
            key=lambda im: (
                -im["height"] * im["width"],
                im["id"],
            ),
        )

        stress = [
            ranked[0],
            next(
                im for im in ranked
                if (im["height"], im["width"])
                != (ranked[0]["height"], ranked[0]["width"])
            ),
        ]

        stress_ids = [im["id"] for im in stress]

        report["stress_ids_personal"] = stress_ids

        with torch.inference_mode():
            report["synthetic_100"] = gpu_mask_stress(
                device,
                (
                    stress[0]["height"],
                    stress[0]["width"],
                ),
            )

        print("SYNTHETIC MASK STRESS PASS", flush=True)

        release_model(device)

        model_keys = (
            "yolo11n_all",
            "yolov8n_all",
            "maskrcnn_all",
            "yolo11n_hard3",
            "yolov8n_hard3",
            "maskrcnn_hard3",
        )

        for key in model_keys:
            fixture_diag = None
            diag = None

            print(
                f"\nGPU verification: {key}",
                flush=True,
            )

            spec = lock["models"][key]
            path, _ = models[key]

            scope = (
                "hard3"
                if key.endswith("_hard3")
                else "all"
            )

            control_dataset = datasets[
                f"original_{scope}"
            ]

            stress_dataset = datasets[
                f"personal_{scope}"
            ]

            model, _ = load_model(
                spec,
                path,
                device,
                lock["class_order"],
            )

            model_report = {
                "passed": False,
            }

            report["models"][key] = model_report

            old = None

            try:
                if scope == "all":
                    try:
                        old, _ = infer_run01(
                            model,
                            spec,
                            control_dataset,
                            control_ids,
                            device,
                        )

                    except Exception as exc:
                        model_report[
                            "legacy_diagnostic_error"
                        ] = str(exc)

                        traceback.clear_frames(
                            exc.__traceback__
                        )

                    if spec["family"] != "maskrcnn":
                        model.predictor = None

                    release_model(device)

                fixture_diag = {
                    "model_dataset_key": f"{key}__original",
                    "stage": "same_source_mask_verification",
                }

                new, control_timing, checked = (
                    verify_same_source_masks(
                        model,
                        spec,
                        control_dataset,
                        control_ids,
                        device,
                        fixture_diag,
                    )
                )

                validate_stress_predictions(
                    new,
                    control_timing,
                    control_dataset,
                    control_ids,
                )

                model_report[
                    "same_source_masks_checked"
                ] = checked

                model_report[
                    "same_source_masks_available"
                ] = sum(checked.values()) > 0

                if scope == "all":
                    require(
                        sum(checked.values()) > 0,
                        f"No actual masks verified for {key}."
                    )

                if sum(checked.values()) > 0:
                    print(
                        f"{key}: SAME-SOURCE ENCODING PASS "
                        f"{checked}",
                        flush=True,
                    )
                else:
                    print(
                        f"{key}: NO MASKS IN CONTROL; "
                        "synthetic encoding checks passed.",
                        flush=True,
                    )

                if old is not None:
                    try:
                        equivalence = equivalent_predictions(
                            old,
                            new,
                            control_dataset,
                        )

                        model_report[
                            "comparison"
                        ] = equivalence

                        model_report[
                            "mask_difference_details"
                        ] = mask_difference_details(
                            old, new
                        )

                        legacy_equal = all(
                            row.get("passed", False)
                            for row in equivalence.values()
                        )

                        model_report[
                            "legacy_equivalent"
                        ] = legacy_equal

                        print(
                            f"{key}: legacy comparison "
                            f"{'PASS' if legacy_equal else 'DIFFERENT (diagnostic only)'}",
                            flush=True,
                        )

                    except Exception as exc:
                        model_report[
                            "legacy_diagnostic_error"
                        ] = str(exc)

                        traceback.clear_frames(
                            exc.__traceback__
                        )

                        print(
                            f"{key}: legacy diagnostic unavailable.",
                            flush=True,
                        )

                del old, new

                if spec["family"] != "maskrcnn":
                    model.predictor = None

                release_model(device)

                if device != "cpu":
                    torch.cuda.reset_peak_memory_stats(device)

                diag = {
                    "model_dataset_key": f"{key}__personal",
                    "stage": "stress",
                }

                predictions, timing = infer_recovery(
                    model,
                    spec,
                    stress_dataset,
                    stress_ids,
                    device,
                    diag,
                )

                validate_stress_predictions(
                    predictions,
                    timing,
                    stress_dataset,
                    stress_ids,
                )

                model_report["stress"] = {
                    **timing,
                    **memory_snapshot(device),
                    "passed": True,
                    "bbox_predictions": len(
                        predictions["bbox"]
                    ),
                    "mask_predictions": len(
                        predictions["segm"]
                    ),
                }

                model_report["passed"] = True

                del predictions

                print(
                    f"{key}: HIGH-RESOLUTION GPU STRESS PASS",
                    flush=True,
                )

            finally:
                if spec["family"] != "maskrcnn":
                    model.predictor = None

                del model

                release_model(device)

        require(
            len(report["models"]) == 6
            and all(
                result.get("passed", False)
                for result in report["models"].values()
            ),
            "Not all six models passed GPU verification."
        )

        report["passed"] = True

    except Exception as exc:
        report.update(
            error=str(exc),
            traceback=traceback.format_exc(),
        )

        if diag is not None:
            report["last_stress_context"] = diag

        if fixture_diag is not None:
            report["last_control_context"] = fixture_diag

        traceback.clear_frames(exc.__traceback__)

    release_model(device)

    write_json(
        run / "gpu_equivalence_report.json",
        report,
    )

    print(
        "GPU STRESS AND EQUIVALENCE PASS (NON-FINAL)"
        if report["passed"]
        else (
            "GPU STRESS / EQUIVALENCE BLOCKED. "
            "Periksa gpu_equivalence_report.json."
        ),
        flush=True,
    )

    return report


GPU_GATE = {
    "passed": False,
    "status": "not_run",
}

if MODE in {"STRESS_TEST", "FULL_EVALUATION"}:
    GPU_GATE = recovery_stress(
        RUN,
        DATASETS,
        MODELS,
        LOCK,
        DEVICE,
        ENVIRONMENT,
    )

    MANIFEST["gpu_equivalence"] = {
        "passed": GPU_GATE["passed"],
        "report_sha256": sha256(
            RUN / "gpu_equivalence_report.json"
        ),
    }

    save_status(RUN, MANIFEST)

LETTERBOX VERIFICATION PASS
SYNTHETIC MASK STRESS PASS

GPU verification: yolo11n_all
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
yolo11n_all: SAME-SOURCE ENCODING PASS {'yolo': 200, 'maskrcnn': 0}
yolo11n_all: legacy comparison DIFFERENT (diagnostic only)
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
yolo11n_all: HIGH-RESOLUTION GPU STRESS PASS

GPU verification: yolov8n_all
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'qu

### 12 Run full evaluation

In [14]:
if MODE == "FULL_EVALUATION":
    if APPROVE_FULL_RECOVERY is not True or not GPU_GATE["passed"]:
        print("FULL EVALUATION BLOCKED: perlu persetujuan eksplisit dan gate GPU PASS.", flush=True)
    else:
        try:
            DATASETS, HARD_MAP, _ = prepare_datasets(INDEX, LOCK)
            MODELS = resolve_models(INDEX, LOCK, CHECKPOINT_OVERRIDES)
            execute_stage(RUN, MANIFEST, "primary", PAIRS, set(PAIRS), [], MODELS, DATASETS,
                          LOCK, HARD_MAP, DEVICE, ENVIRONMENT)
            if stage_complete(MANIFEST, "primary", PAIRS):
                consolidate_recovered_metrics(RUN, MANIFEST, DATASETS, PAIRS, HARD_MAP)
        except Exception:
            MANIFEST["orchestration_error"] = traceback.format_exc()
            save_status(RUN, MANIFEST)
            print("FULL EVALUATION PARTIAL. Periksa evaluation_status_manifest.json.", flush=True)

primary / yolo11n_all__original: running
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be remov

### 13 Verify results and make tables

In [15]:
if read_json(RUN / "preflight_report.json")["passed"]:
    export_tables(RUN, MANIFEST, PAIRS, DATASETS, LOCK)

### 14 Export artifacts

In [16]:
OUTPUT_ZIP = export_run(RUN, MANIFEST, MODE, PAIRS)

Status COMPLETE. Unduh /kaggle/working/CPMK3_unified_test_recovery/20261126T152940Z-d1cca883/CPMK3_Unified_Test_Evaluation_Recovery_Complete.zip
